# TFBA for R. palustris GEM

Constraint-based analysis on genome-scale metabolic models (GEMs) is a popular method to study metabolism and cellular physiology. Flux Balance Analysis (FBA), in particular, has been used to predict network-level behaviors, such as specific growth rate, gene essentiality, etc. However, FBA-derived approaches often lead to flux distributions
that are contradicting with physiology and bioenergetics due to the
lack of thermodynamic constraints in their formulation. Metabolic reactions in GEMs can be futher constrained thermodynamically using TFA (Salvy...Meric et., al, 2019).

References

*   https://github.com/EPFL-LCSB/pytfa/tree/master
*   Bioinformatics, 35(1), 2019, 167–169
doi: 10.1093/bioinformatics/bty499



# Install dependencies

In [ ]:
import os
os.kill(os.getpid(), 9)


In [ ]:
!rm -rf /root/.cache/pip


In [3]:
import importlib.util
print(importlib.util.find_spec("pytfa"))


None


In [1]:
!pip3 install pytfa cobra optlang modelseedpy


Requested pytfa from https://files.pythonhosted.org/packages/bb/ac/1d6a4a72f45bfa46a95d3cc9f29c113c5f4a4ece4ac7461b56ac61c2fa2c/pytfa-0.9.4-py2.py3-none-any.whl has invalid metadata: Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier
    python-version (>="3.6") ; extra == 'equilibrator'
                   ~^
Please use pip<24.1 if you need to use this version.
Requested pytfa from https://files.pythonhosted.org/packages/a3/b4/9d3a72b34043fc5e4b75afcefd2d386ebab18ca99690cfa1c4c01612c77d/pytfa-0.9.3-py2.py3-none-any.whl has invalid metadata: Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier
    python-version (>="3.6") ; extra == 'equilibrator'
                   ~^
Please use pip<24.1 if you need to use this version.
Requested pytfa from https://files.pythonhosted.org/packages/52/75/1b2114796a7f0b51ce973798805a5ccf5e929078bc65e59dc44b4f5601ea/pytfa-0.9.2-py2.py3-none-any.whl has invalid metadata: Expected matching RIGHT_PAR

In [ ]:
# Check installation
!pip3 show cobra

Name: cobra
Version: 0.30.0
Summary: COBRApy is a package for constraint-based modeling of metabolic networks.
Home-page: https://opencobra.github.io/cobrapy
Author: The cobrapy core development team.
Author-email: cobra-pie@googlegroups.com
License: LGPL-2.0-or-later OR GPL-2.0-or-later
Location: /usr/local/lib/python3.12/dist-packages
Requires: appdirs, depinfo, diskcache, future, httpx, numpy, optlang, pandas, pydantic, python-libsbml, rich, ruamel.yaml, swiglpk
Required-by: ModelSEEDpy, pytfa


In [ ]:
!pip3 show pytfa

Name: pytfa
Version: 0.9.1
Summary: pyTFA, Thermodynamics-based Flux Analysis in Python
Home-page: https://github.com/EPFL-LCSB/pytfa/
Author: pyTFA team
Author-email: softwares.lcsb@epfl.ch
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: bokeh, cobra, networkx, optlang, pytest, scipy, tqdm
Required-by: 


## Mount drive

In [2]:
import os
from pathlib import Path
from google.colab import drive

def mount_drive():
  drive.mount('/content/drive', force_remount=True)
  drive_folder = "metabolic_modelling/phb-optimization-rpalustris/"
  os.chdir('/content/drive/MyDrive/'+ drive_folder)
  global PROJECT_ROOT
  PROJECT_ROOT = Path(os.getcwd())

mount_drive()


Mounted at /content/drive


In [ ]:
!pwd

/content/drive/MyDrive/metabolic_modelling/phb-optimization-rpalustris


# Load packages

In [12]:
# Load packages

import logging
import cobra
from cobra import Reaction, Metabolite, Model, io
from cobra.io import (
    load_model, load_json_model, save_json_model,
    load_matlab_model, save_matlab_model,
    read_sbml_model, write_sbml_model,
)

from cobra.flux_analysis import flux_variability_analysis, pfba

from cobra.util.solver import linear_reaction_coefficients

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
from itertools import product
import seaborn as sns
import json
import math
import copy
from tqdm import tqdm
import sys
import seaborn as sns
from scipy.stats import mannwhitneyu
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import pytfa, pkgutil, inspect
from pytfa.thermo import ThermoModel
from pytfa.io import load_thermoDB
from pytfa.optim.variables import DeltaG
from pytfa.analysis import variability_analysis

from pytfa.io import read_lexicon, annotate_from_lexicon, read_compartment_data, apply_compartment_data


# Set paths

In [13]:
# Import path saved in src

import sys
sys.path.append(str(PROJECT_ROOT / "src"))
from src.paths import PHBV_MODEL_DIR, PHB_TFA_DIR, PHB_MODEL_TFA_DIR, DATA_DIR, MODELS_DIR, PHB_MODEL_DIR, PHB_CHECKPOINTS_DIR, PHB_RESULTS_DIR, PHB_FIGURES_DIR, PHB_GEM_EXPERIMENTAL_DIR, PHB_GEM_AUGMENTATION_DIR, PHB_CATBOOST_DIR, PHB_PARETO_DIR


# Load thermo data




In [14]:
# Download thermo data from source page
from src.paths import DATA_DIR

!wget https://raw.githubusercontent.com/EPFL-LCSB/pytfa/master/data/thermo_data.thermodb -O {DATA_DIR}/thermo_data.thermodb
# Load thermodynamics

thermo_data_path = DATA_DIR / "thermo_data.thermodb"
thermo_data = load_thermoDB(thermo_data_path)

# Print thermo_data keys

print(f"Thermo Data loaded successfully:{thermo_data.keys()}")

--2026-03-18 02:39:57--  https://raw.githubusercontent.com/EPFL-LCSB/pytfa/master/data/thermo_data.thermodb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2184378 (2.1M) [application/octet-stream]
Saving to: ‘/content/drive/MyDrive/metabolic_modelling/phb-optimization-rpalustris/data/thermo_data.thermodb’

/content/drive/MyDr 100%[===================>]   2.08M  --.-KB/s    in 0.1s    

2026-03-18 02:39:58 (20.4 MB/s) - ‘/content/drive/MyDrive/metabolic_modelling/phb-optimization-rpalustris/data/thermo_data.thermodb’ saved [2184378/2184378]

Thermo Data loaded successfully:dict_keys(['name', 'units', 'metabolites', 'cues'])


# E.coli TFA tutorial

## Load E. coli model and data

In [ ]:
# Import ecoli model from tutorial

from pytfa.io import import_matlab_model

from pathlib import Path

tutorial_root = Path(
    "/content/drive/MyDrive/metabolic_modelling/other_materials/pytfa/"
)
print(tutorial_root)
ec_model_mat_path = tutorial_root / "models"/"small_ecoli.mat"
ec_model_json_path = tutorial_root / "models"/ "iJO1366.json"
ec_model_lexicon_path = tutorial_root / "models"/ "iJO1366" / "lexicon.csv"
ec_model_comp_path = tutorial_root / "models"/ "iJO1366" /"compartment_data.json"


/content/drive/MyDrive/metabolic_modelling/other_materials/pytfa


In [ ]:
ecoli_model= load_json_model(ec_model_json_path)

In [ ]:
ecoli_model.compartments

{'c': '', 'e': '', 'p': ''}

In [ ]:
lexicon = read_lexicon(ec_model_lexicon_path)
lexicon.columns

Index(['seed_id'], dtype='object')

In [ ]:
mytfa.solver = "glpk"
mytfa.objective = biomass_rxn

# Solver settings

def apply_solver_settings(model, solver = "glpk"):
    model.solver = "glpk"
    # model.solver.configuration.verbosity = 1
    model.solver.configuration.tolerances.feasibility = 1e-9
    if solver == 'optlang_gurobi':
        model.solver.problem.Params.NumericFocus = 3
    model.solver.configuration.presolve = True

apply_solver_settings(mytfa)

NameError: name 'mytfa' is not defined

In [ ]:
mytfa.convert()#add_displacement = True)

## Info on the cobra_model
mytfa.summary()

## Optimality
tfa_solution = mytfa_rpalustris.optimize()
tfa_value = tfa_solution.objective_value

NameError: name 'mytfa' is not defined

In [ ]:
# List metabolites with thermo data
annotated = [m.id for m in mytfa.metabolites if hasattr(m, "thermo")]
print(f"Metabolites annotated with thermodynamic data: {len(annotated)} / {len(mytfa.metabolites)}")

Metabolites annotated with thermodynamic data: 1807 / 1807


# R. palustirs TFA

## Load COBRA model

In [21]:
# Load COBRA modele produced in notebook 2

cobra_model = load_json_model(
    PHB_MODEL_DIR / "02_model_rpalustris_PHB_constrained.json"
)

## Correct formulas of key metabolites

In [16]:
# The following metabolites are

import re
from collections import Counter

# --------------------------------
# INPUT DATA
# --------------------------------
Thermodb_meta_interest = [
  "C21H27N7O17P3",
  "C21H27N7O14P2",
  "C23H35N7O17P3S",
  "C21H26N7O14P2",
  "C25H39N7O18P3S",
  "C21H26N7O17P3",
  "C25H37N7O18P3S",
  "C21H33N7O16P3S"
]

GEM_meta_interest = [
  "C21H26N7O17P3",
  "C21H27N7O14P2",
  "C23H34N7O17P3S",
  "C21H26N7O14P2",
  "C25H38N7O18P3S",
  "C21H25N7O17P3",
  "C25H36N7O18P3S",
  "C21H32N7O16P3S"
]

# GEM metabolite IDs in SAME ORDER
gem_met_ids = [
  "nadph_c",
  "nadh_c",
  "accoa_c",
  "nad_c",
  "3hbcoa__R_c",
  "nadp_c",
  "aacoa_c",
  "coa_c",
]

# --------------------------------
# FORMULA PARSING UTILITIES
# --------------------------------
def parse_formula(formula):
  """Convert formula string to Counter"""
  return Counter({
      elem: int(count) if count else 1
      for elem, count in re.findall(r'([A-Z][a-z]?)(\d*)', formula)
  })

def counter_to_formula(counter):
  """Convert Counter back to formula string (Hill order, omit 1s)"""
  parts = []

  # Hill system: C, H, then others alphabetically
  if 'C' in counter:
      parts.append(f"C{counter['C']}" if counter['C'] != 1 else "C")
  if 'H' in counter:
      parts.append(f"H{counter['H']}" if counter['H'] != 1 else "H")

  for el in sorted(counter):
      if el in ('C', 'H'):
          continue
      count = counter[el]
      parts.append(f"{el}{count}" if count != 1 else el)

  return ''.join(parts)


def compare_formulas(thermo_f, gem_f):
  """Return element-wise difference GEM − ThermoDB"""
  t = parse_formula(thermo_f)
  g = parse_formula(gem_f)
  elements = set(t) | set(g)
  return {el: g.get(el, 0) - t.get(el, 0) for el in elements}

def update_gem_formula(gem_f, thermo_f):
  """Add missing elements from ThermoDB into GEM formula"""
  g = parse_formula(gem_f)
  t = parse_formula(thermo_f)

  for el, count in t.items():
      if g.get(el, 0) < count:
          g[el] = count

  return counter_to_formula(g)

# --------------------------------
# APPLY UPDATES TO MODEL
# --------------------------------
for met_id, thermo_f, gem_f in zip(gem_met_ids,
                                  Thermodb_meta_interest,
                                  GEM_meta_interest):

    diff = compare_formulas(thermo_f, gem_f)

    # Safety check (CRITICAL for TFA)
    for el in diff:
        if el in ("C", "N", "P") and diff[el] != 0:
            raise ValueError(
                f"Unsafe formula mismatch for {met_id}: element {el}"
            )

    new_formula = update_gem_formula(gem_f, thermo_f)

    if new_formula != gem_f:
        met = cobra_model.metabolites.get_by_id(met_id) ## update cobra model

        print(f"Updating {met_id}")
        print(f"  old: {gem_f}")
        print(f"  new: {new_formula}")
        print(f"  ThermoDB:       {thermo_f}")

        met.formula = new_formula

        # PRINT FROM MODEL OBJECT
        print(f"  new (model):  {met.formula}")
        print("-" * 40)


print("GEM formulas updated and TFA-ready.")

Updating nadph_c
  old: C21H26N7O17P3
  new: C21H27N7O17P3
  ThermoDB:       C21H27N7O17P3
  new (model):  C21H27N7O17P3
----------------------------------------
Updating accoa_c
  old: C23H34N7O17P3S
  new: C23H35N7O17P3S
  ThermoDB:       C23H35N7O17P3S
  new (model):  C23H35N7O17P3S
----------------------------------------
Updating 3hbcoa__R_c
  old: C25H38N7O18P3S
  new: C25H39N7O18P3S
  ThermoDB:       C25H39N7O18P3S
  new (model):  C25H39N7O18P3S
----------------------------------------
Updating nadp_c
  old: C21H25N7O17P3
  new: C21H26N7O17P3
  ThermoDB:       C21H26N7O17P3
  new (model):  C21H26N7O17P3
----------------------------------------
Updating aacoa_c
  old: C25H36N7O18P3S
  new: C25H37N7O18P3S
  ThermoDB:       C25H37N7O18P3S
  new (model):  C25H37N7O18P3S
----------------------------------------
Updating coa_c
  old: C21H32N7O16P3S
  new: C21H33N7O16P3S
  ThermoDB:       C21H33N7O16P3S
  new (model):  C21H33N7O16P3S
----------------------------------------
GEM formula

## TFA preparation and conversion


In [22]:

palustris_model_lexicon_path = PHB_MODEL_TFA_DIR / "lexicon_auto.csv"
lexicon_palustris = read_lexicon(palustris_model_lexicon_path)

compartment_data = read_compartment_data(
    str(PHB_MODEL_TFA_DIR / "compartment_data.json")
)

mytfa_rpalustris = ThermoModel(thermo_data, cobra_model)

annotate_from_lexicon(mytfa_rpalustris, lexicon_palustris)
apply_compartment_data(mytfa_rpalustris, compartment_data)

## Prepare

In [23]:
# Prepare TFA model

mytfa_rpalustris.prepare()

## Find metabolites and reactions without thermo data

In [24]:
bad_mets = []

for m in mytfa_rpalustris.metabolites:
    if hasattr(m, "thermo"):
        if m.thermo.deltaGf_tr >= 1e6:
            bad_mets.append(m)

print("Metabolites without thermo:", len(bad_mets))

Metabolites without thermo: 289


In [25]:
bad_rxns = []

for rxn in mytfa_rpalustris.reactions:
    if any(m in bad_mets for m in rxn.metabolites):
        bad_rxns.append(rxn)

print("Reactions affected:", len(bad_rxns))

Reactions affected: 527


## Turn off reactions that lack thermo data

In [26]:
for rxn in bad_rxns:
    if hasattr(rxn, "thermo"):
        rxn.thermo["computed"] = False

## Convert

In [27]:
mytfa_rpalustris.convert()#add_displacement = True)


## Coverage

In [28]:
thermo_rxns = [
    r.id for r in mytfa_rpalustris.reactions
    if hasattr(r, "thermo") and r.thermo["computed"]
]

fba_rxns = [
    r.id for r in mytfa_rpalustris.reactions
    if hasattr(r, "thermo") and not r.thermo["computed"]
]

print("Thermo reactions:", len(thermo_rxns))
print("FBA fallback reactions:", len(fba_rxns))

Thermo reactions: 1417
FBA fallback reactions: 1304


## Check that tfa was executed

In [29]:
# Check that "DG_" variables exist
len([v for v in mytfa_rpalustris.variables if "DG_" in v.name])

1419

## Check PHB functions thermo data

In [30]:
PHB_RXNS = [
    "PHBS_syn",
    "ACACT1r",
    "ACACCT",
    "AACOAR_syn",
    "HACD1_2",
    "HACD1",
    "HACD1i",
    "KAT1"
]

for r in PHB_RXNS:
    rxn = mytfa_rpalustris.reactions.get_by_id(r)

    print("\nReaction:", r)

    if hasattr(rxn, "thermo"):
        print("include thermo data:", r in thermo_rxns)
        print("ΔG°:", rxn.thermo["deltaGR"])
        print("ΔG° uncertainty:", rxn.thermo["deltaGRerr"])
        print("computed:", rxn.thermo["computed"])
        print("transport:", rxn.thermo["isTrans"])
    else:
        print("No thermo data")


Reaction: PHBS_syn
include thermo data: False
ΔG°: 1000.0
ΔG° uncertainty: 1000.0
computed: False
transport: True

Reaction: ACACT1r
include thermo data: True
ΔG°: 9.06445594322821
ΔG° uncertainty: 1.087366929789572
computed: True
transport: False

Reaction: ACACCT
include thermo data: True
ΔG°: -1.540445621776371
ΔG° uncertainty: 0.5236430081649138
computed: True
transport: False

Reaction: AACOAR_syn
include thermo data: True
ΔG°: -1.9038808744587072
ΔG° uncertainty: 1.079693011925149
computed: True
transport: False

Reaction: HACD1_2
include thermo data: False
ΔG°: 1000.0
ΔG° uncertainty: 1000.0
computed: False
transport: False

Reaction: HACD1
include thermo data: True
ΔG°: -2.2909134854636477
ΔG° uncertainty: 1.079693011925149
computed: True
transport: False

Reaction: HACD1i
include thermo data: True
ΔG°: 2.2909134854636477
ΔG° uncertainty: 1.079693011925149
computed: True
transport: False

Reaction: KAT1
include thermo data: True
ΔG°: -9.06445594322821
ΔG° uncertainty: 1.087366

## Check objective functions

In [31]:
# FBA reaction
phb_fba_rxn = cobra_model.reactions.get_by_id('PHBS_syn')
print(f"FBA bounds: {phb_fba_rxn.bounds}")
print(f"FBA reaction: {phb_fba_rxn.reaction}")

# TFA reaction
phb_tfa_rxn = mytfa_rpalustris.reactions.get_by_id('PHBS_syn')
print(f"TFA bounds: {phb_tfa_rxn.bounds}")
print(f"TFA reaction: {phb_tfa_rxn.reaction}")


FBA bounds: (0.0, 0.3978)
FBA reaction: 3hbcoa__R_c + phbg_c --> PHB_c + coa_c
TFA bounds: (0.0, 0.3978)
TFA reaction: 3hbcoa__R_c + phbg_c --> PHB_c + coa_c


## Test Run FBA and TFA

In [156]:
# Solve both models
sol_fba = cobra_model.optimize()
sol_tfa = mytfa_rpalustris.optimize()

# Compare Objective values
print(sol_fba.objective_value)
print(sol_tfa.objective_value)


DEBUG:thermomodel_None:'slim_optimize' ((), {}) 52.70 sec


0.12375000000000189
0.1237499999999816


## Compare fluxes of key reactions

In [32]:
# Compare flux of key reactions in FBA and TFA
rows = []
for r in PHB_RXNS:
    rows.append({
        "Reaction": r,
        "FBA Flux": sol_fba.fluxes[r],
        "TFA Flux": sol_tfa.fluxes[r]
    })

df = pd.DataFrame(rows)
print(df)

NameError: name 'sol_fba' is not defined

## FVA PHB reactions

In [168]:
# FVA for FBA optimization

PHB_RXNS = [
    "PHBS_syn",
    "ACACT1r",
    "ACACCT",
    "AACOAR_syn",
    "HACD1_2",
    "HACD1",
    "HACD1i",
    "KAT1"
]

# run FVA
fva_fba = flux_variability_analysis(cobra_model, PHB_RXNS, fraction_of_optimum=1.0)
fva_fba

In [ ]:
# FVA for TFA optimization

fva_tfa = flux_variability_analysis(mytfa_rpalustris, PHB_RXNS, fraction_of_optimum=1.0)

DEBUG:thermomodel_None:'slim_optimize' ((), {'error_value': None, 'message': 'There is no optimal solution for the chosen objective!'}) 89.42 sec
DEBUG:thermomodel_None:'slim_optimize' ((), {}) 254.29 sec
DEBUG:thermomodel_None:'slim_optimize' ((), {}) 832.86 sec
DEBUG:thermomodel_None:'slim_optimize' ((), {}) 124.12 sec
DEBUG:thermomodel_None:'slim_optimize' ((), {}) 168.42 sec


In [ ]:
fva_tfa

## Report DG_ variables

In [161]:
dg_vars = [v for v in mytfa_rpalustris.variables if v.name.startswith("DG_")]

print("Thermodynamically constrained reactions:", len(dg_vars))
print("Total reactions:", len(mytfa_rpalustris.reactions))
print("Fraction constrained:",
      len(dg_vars) / len(mytfa_rpalustris.reactions))

Thermodynamically constrained reactions: 1417
Total reactions: 2721
Fraction constrained: 0.5207644248438075


## Report specific DG_ values

In [165]:
for v in mytfa_rpalustris.variables:
    if "DG_" in v.name:
        print(v.name, sol_tfa.raw[v.name])

BG_MADG_reverse_90e03 0.0
BG_MBDG_reverse_25438 0.0
DG_QULNS -11.623985681943154
DG_FUM -1.1092780300495884
DG_VPAMTr -1.9999771606751153
DG_NDPK7 11.037396093831456
DG_GTPCI -34.35325130501178
DG_MTHFC 8.11936439306578
DG_GCATENEC -15.185125041493105
DG_GARFT -1.0213067764095687
DG_UDPG4E -8.406275779637213
DG_ASPTA -8.830458334455077
DG_PRAGSr 10.442621478470315
DG_ACKr 11.16151367208828
DG_UDCPDPS -166.06207272656158
DG_SHKK 6.17245069682761
DG_Htex -2.0
DG_PNTK 6.146022850389787
DG_ACGK 7.004097698254102
DG_LEUabcpp -1.8732913934763804
DG_G5SD 8.961867352729051
DG_DTMPK 11.113118144182858
DG_HSTPT -1.170032945313963
DG_PGK 13.321757205510707
DG_IMPD -8.937397840648007
DG_CHRPL -51.521514024145745
DG_GCALDDy -7.51105483025265
DG_AIRC2 -18.450021263869033
DG_NNATr -3.7705856139442875
DG_HISTDb -19.549508154932504
DG_ADCL -36.00028233763307
DG_CYTK1 4.706841918765537
DG_THRS 1.0408768038922718
DG_NAMNPP -13.633111887641963
DG_PERD -2.7504535593003068
DG_DAPDC -11.116695695343676
DG_AN

# * TORUN: R palustris model multiple PHB syn values

In [ ]:
# ============================================================
# USER PARAMETERS
# ============================================================

import logging
import pandas as pd
import numpy as np
from cobra.flux_analysis import pfba

logging.getLogger("thermomodel_None").setLevel(logging.ERROR)
logging.getLogger("pytfa").setLevel(logging.ERROR)

PHB_SYN_UB_VALUES = [0.3978, 1000]  # 0.3978,
PHB_REACTIONS = [
    "ACACT1r",
    "ACACCT",
    "AACOAR_syn",
    "HACD1_2",
    "HACD1",
    "HACD1i",
    "KAT1",
]

SUB_TO_RXN_BASE = {
    "ace": "EX_ac_e",
    "lac": "EX_lac__L_e",
    "ppa": "EX_ppa_e",
    "but": "EX_but_e",
    "ibt": "EX_ibt_e",
    "mal": "EX_mal__L_e",
    "hxa": "EX_hxa_e",
    "oct": "EX_octa_e",
    "nh4": "EX_nh4_e",
    "hco3": "EX_hco3_e",
}

EXCHANGE_RXNS = list(SUB_TO_RXN_BASE.values())

# ============================================================
# SOLVER SETTINGS
# ============================================================
def apply_solver_settings(model):
    model.solver = "glpk"
    model.solver.configuration.tolerances.feasibility = 1e-9
    model.solver.configuration.presolve = True

# Apply solver settings once to original models
apply_solver_settings(cobra_model)
apply_solver_settings(mytfa_rpalustris)


# ============================================================
# MAIN LOOP OVER PHB SYNTHESIS UB VALUES
# ============================================================
for PHB_SYN_UB in PHB_SYN_UB_VALUES:

    print("\n==============================")
    print(f"PHB_SYN_UB_VALUE: {PHB_SYN_UB}")
    print("==============================")

    PHB_SYN_TAG = f"{PHB_SYN_UB:g}"

    INPUT_CSV = PHB_PARETO_DIR / f"04_pareto_best_all_strategies_{PHB_SYN_TAG}.csv"
    OUT_CSV = PHB_TFA_DIR / f"05_FBA_TFA_PFBA_detailed_results_all_strategies_checked_{PHB_SYN_TAG}.csv"


    # Keep ONLY SUP_and_CON

    df = df[df["optimization_strategy"] == "SUP_and_CON"].copy()

    results = []

    for pareto_id, row in df.iterrows():

        strategy = row["optimization_strategy"]
        suffix = "_sup"

    #df = pd.read_csv(INPUT_CSV)
    #results = []

    #for pareto_id, row in df.iterrows():

        #strategy = row["optimization_strategy"]

        #if "SUP_and_CON" in strategy:
        #    suffix = "_sup"
        #elif strategy == "CON":
        #    suffix = "_con"
        #elif strategy == "SUP":
        #    suffix = "_sup"
        #elif strategy == "PFBA":
        #    suffix = "_con_pfba"
        #elif strategy == "SUP_and_PFBA":
        #    suffix = "_sup"
        #else:
        #    print(f'Other optimization strategy: {strategy}')
        #    suffix = ""

        # -----------------------------
        # COPY COBRA MODEL (FBA/pFBA)
        # -----------------------------
        m = cobra_model.copy()

        #tfa_model = mytfa_rpalustris.copy()

        # -----------------------------
        # UPDATE EXCHANGE BOUNDS
        # -----------------------------
        input_values_used = {}
        for base_col, rxn_id in SUB_TO_RXN_BASE.items():
            col_name = f"{base_col}{suffix}"
            val_used = np.nan

            if col_name in row:
                val = row[col_name]
                if pd.notna(val) and float(val) != 0.0 and rxn_id in m.reactions:
                    rxn = m.reactions.get_by_id(rxn_id)
                    rxn.lower_bound = -abs(float(val))
                    rxn.upper_bound = 0.0
                    val_used = float(val)

            input_values_used[f"{base_col}_supplied"] = 0.0 if pd.isna(val_used) else val_used

            # Also update TFA bounds
            if rxn_id in mytfa_rpalustris.reactions:
                rxn_tfa = mytfa_rpalustris.reactions.get_by_id(rxn_id)
                rxn_tfa.lower_bound = -abs(val_used) if pd.notna(val_used) else rxn_tfa.lower_bound
                rxn_tfa.upper_bound = 0.0


        # -----------------------------
        # OBJECTIVE & NO GROWTH
        # -----------------------------
        phb_rxn = m.reactions.get_by_id("PHBS_syn")
        phb_rxn.lower_bound = 0.0
        phb_rxn.upper_bound = PHB_SYN_UB
        m.objective = phb_rxn

        biomass = m.reactions.get_by_id("BIOMASS__1")
        biomass.lower_bound = 0.0
        biomass.upper_bound = 0.0

        # Update TFA objective and PHB bounds
        phb_tfa = mytfa_rpalustris.reactions.get_by_id("PHBS_syn")
        phb_tfa.lower_bound = 0.0
        phb_tfa.upper_bound = PHB_SYN_UB
        mytfa_rpalustris.objective = phb_tfa

        biomass_tfa = mytfa_rpalustris.reactions.get_by_id("BIOMASS__1")
        biomass_tfa.lower_bound = 0.0
        biomass_tfa.upper_bound = 0.0

        # -----------------------------
        # FBA
        # -----------------------------
        fba_sol = m.optimize()

        # -----------------------------
        # pFBA
        # -----------------------------
        pfba_sol = pfba(m)

        # -----------------------------
        # TFA
        # -----------------------------
        try:
            tfa_sol = mytfa_rpalustris.optimize()
        except Exception as e:
            print(f"TFA optimization failed for Pareto solution {pareto_id}: {e}")
            tfa_sol = None

        # -----------------------------
        # COLLECT RESULTS
        # -----------------------------
        result = {
            "PHB_SYN_UB": PHB_SYN_UB,
            "PHB_SYN_TAG": PHB_SYN_TAG,
            "pareto_id": pareto_id,
            "PHB_pFBA": pfba_sol.fluxes.get("PHBS_syn", np.nan),
            "PHB_FBA": fba_sol.fluxes.get("PHBS_syn", np.nan),
            "PHB_TFA": tfa_sol.fluxes.get("PHBS_syn", np.nan),
            "Biomass_pFBA": pfba_sol.fluxes.get("BIOMASS__1", np.nan),
            "Biomass_FBA": fba_sol.fluxes.get("BIOMASS__1", np.nan),
            "Biomass_TFA": tfa_sol.fluxes.get("BIOMASS__1", np.nan),
            "status_pFBA": pfba_sol.status,
            "status_FBA": fba_sol.status,
            "status_TFA": tfa_sol.status,
            "optimization_strategy": strategy
        }

        # PHB reactions
        for r in PHB_REACTIONS:
            result[f"{r}_pFBA"] = pfba_sol.fluxes.get(r, np.nan)
            result[f"{r}_FBA"] = fba_sol.fluxes.get(r, np.nan)
            result[f"{r}_TFA"] = tfa_sol.fluxes.get(r, np.nan)

        # Exchange reactions
        for base_col, rxn_id in SUB_TO_RXN_BASE.items():
            result[f"{base_col}_pFBA"] = pfba_sol.fluxes.get(rxn_id, np.nan)
            result[f"{base_col}_FBA"]  = fba_sol.fluxes.get(rxn_id, np.nan)
            result[f"{base_col}_TFA"]  = tfa_sol.fluxes.get(rxn_id, np.nan)

        result.update(input_values_used)
        print(pd.Series(result))
        print("-" * 60)
        results.append(result)

    # Save results
    pd.DataFrame(results).to_csv(OUT_CSV, index=False)
    print(f"\nDONE — Results saved to:\n{OUT_CSV}")


PHB_SYN_UB_VALUE: 0.3978
PHB_SYN_UB         0.3978
PHB_SYN_TAG        0.3978
pareto_id               2
PHB_pFBA           0.3978
PHB_FBA            0.3978
                   ...   
mal_supplied    -0.238157
hxa_supplied          0.0
oct_supplied       -0.112
nh4_supplied          0.0
hco3_supplied         0.0
Length: 74, dtype: object
------------------------------------------------------------

DONE — Results saved to:
/content/drive/MyDrive/metabolic_modelling/phb-optimization-rpalustris/results/phb/tfa/05_FBA_TFA_PFBA_detailed_results_all_strategies_checked_0.3978.csv

PHB_SYN_UB_VALUE: 1000


# Main E. coli tutorial

In [ ]:
!pip install sympy==1.10.1

  Using cached sympy-1.10.1-py3-none-any.whl.metadata (12 kB)
Using cached sympy-1.10.1-py3-none-any.whl (6.4 MB)
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
modelseedpy 0.4.2 requires sympy>=1.12.0, but you have sympy 1.10.1 which is incompatible.
optlang 1.8.3 requires sympy>=1.12.0, but you have sympy 1.10.1 which is incompatible.
torch 2.10.0+cpu requires sympy>=1.13.3, but you have sympy 1.10.1 which is incompatible.


In [ ]:
from pytfa.io import import_matlab_model, load_thermoDB,                    \
                            read_lexicon, annotate_from_lexicon,            \
                            read_compartment_data, apply_compartment_data

case = 'full' # 'reduced' or full'

# Load reaction DB
print("Loading thermo data...")

#thermo_data = load_thermoDB('/Users/hector/git/pytfa/data/thermo_data.thermodb')


thermo_data['metabolites'] = {
    str(k): v
    for k, v in thermo_data['metabolites'].items()
}

print("Done !")

if case == 'reduced':
    cobra_model = import_matlab_model(str(ec_model_mat_path))
    mytfa = pytfa.ThermoModel(thermo_data, cobra_model)
    biomass_rxn = 'Ec_biomass_iJO1366_WT_53p95M'
elif case == 'full':
    # We import pre-compiled data as it is faster for bigger models
    cobra_model = load_json_model(ec_model_json_path)

    lexicon = read_lexicon(ec_model_lexicon_path)
    compartment_data = read_compartment_data(str(ec_model_comp_path))

    # Initialize the cobra_model
    mytfa = pytfa.ThermoModel(thermo_data, cobra_model)

    # Annotate the cobra_model
    annotate_from_lexicon(mytfa, lexicon)
    apply_compartment_data(mytfa, compartment_data)

    biomass_rxn = 'Ec_biomass_iJO1366_WT_53p95M'

mytfa.name = 'tutorial_basics'
mytfa.solver = "glpk"
mytfa.objective = biomass_rxn

# Solver settings

def apply_solver_settings(model, solver = "glpk"):
    model.solver = "glpk"
    # model.solver.configuration.verbosity = 1
    model.solver.configuration.tolerances.feasibility = 1e-9
    if solver == 'optlang_gurobi':
        model.solver.problem.Params.NumericFocus = 3
    model.solver.configuration.presolve = True

apply_solver_settings(mytfa)


## FBA
fba_solution = cobra_model.optimize()
fba_value = fba_solution.objective_value
# fva = flux_variability_analysis(mytfa)

## TFA conversion
mytfa.prepare()
mytfa.convert()#add_displacement = True)

## Info on the cobra_model
mytfa.summary()

## Optimality
tfa_solution = mytfa.optimize()
tfa_value = tfa_solution.objective_value

# It might happen that the model is infeasible. In this case, we can relax
# thermodynamics constraints:

#if tfa_value < 0.1:
#    from pytfa.optim.relaxation import relax_dgo

#    mytfa.reactions.get_by_id(biomass_rxn).lower_bound = 0.5*fba_value
#    relaxed_model, slack_model, relax_table = relax_dgo(mytfa)

 #   original_model, mytfa = mytfa, relaxed_model

 #   print('Relaxation: ')
 #   print(relax_table)

  #  tfa_solution = mytfa.optimize()
  #  tfa_value = tfa_solution.objective_value

# Report
print('FBA Solution found : {0:.5g}'.format(fba_value))
print('TFA Solution found : {0:.5g}'.format(tfa_value))


solver_results = dict()

In [ ]:
# It might happen that the model is infeasible. In this case, we can relax
# thermodynamics constraints:

if tfa_value < 0.1:
    from pytfa.optim.relaxation import relax_dgo

    mytfa.reactions.get_by_id(biomass_rxn).lower_bound = 0.5*fba_value
    relaxed_model, slack_model, relax_table = relax_dgo(mytfa)

    original_model, mytfa = mytfa, relaxed_model

    print('Relaxation: ')
    print(relax_table)

    tfa_solution = mytfa.optimize()
    tfa_value = tfa_solution.objective_value

# Report
print('FBA Solution found : {0:.5g}'.format(fba_value))
print('TFA Solution found : {0:.5g}'.format(tfa_value))


solver_results = dict()

In [ ]:
mytfa.compartments

In [ ]:
print(f"TBA:{tfa_value}")
print(f"FBA:{fba_value}")

TBA:0.8109971377524022
FBA:0.8109621653343299


# Check TFA E.coli tutorial

In [ ]:
for v in mytfa.variables.values():
  print(v)

In [ ]:
len([v.name for v in mytfa.variables if v.name.startswith("DGo_")])


416

In [ ]:
[c.name for c in mytfa.constraints if c.name.startswith("G_")][:10]


['G_5MTRtex',
 'G_ACALD',
 'G_ACALDtex',
 'G_ACALDtpp',
 'G_ACKr',
 'G_ACONTa',
 'G_ACONTb',
 'G_ACS',
 'G_ACSERtex',
 'G_ACt2rpp']

In [ ]:
[c.name for c in mytfa.constraints if "DG_" in str(c.expression)][:10]


['G_5MTRtex',
 'FU_5MTRtex',
 'BU_5MTRtex',
 'G_ACALD',
 'FU_ACALD',
 'BU_ACALD',
 'G_ACALDtex',
 'FU_ACALDtex',
 'BU_ACALDtex',
 'G_ACALDtpp']

In [ ]:
[c.name for c in mytfa.constraints if c.name.startswith(("FU_", "BU_", "SU_", "UF_"))][:10]


['SU_DM_4CRSOL',
 'UF_DM_4CRSOL',
 'SU_DM_5DRIB',
 'UF_DM_5DRIB',
 'SU_DM_AMOB',
 'UF_DM_AMOB',
 'SU_DM_MTHTHF',
 'UF_DM_MTHTHF',
 'SU_Ec_biomass_iJO1366_WT_53p95M',
 'UF_Ec_biomass_iJO1366_WT_53p95M']

In [ ]:
assert any(c.name.startswith("G_") for c in mytfa.constraints)
assert any(v.name.startswith("DGo_") for v in mytfa.variables)
assert any(v.type == "binary" for v in mytfa.variables)
print("✅ TFBA is active")


✅ TFBA is active


In [ ]:
for c in mytfa.constraints:
  print(c)


10fthf_c: 0.0 <= -0.000223*Ec_biomass_iJO1366_WT_53p95M + 0.000223*Ec_biomass_iJO1366_WT_53p95M_reverse_55db7 + 1.0*LMPD_1_10fthf_c - 1.0*LMPD_1_10fthf_c_reverse_e2a7c + 1.0*LMPD_2_10fthf_c - 1.0*LMPD_2_10fthf_c_reverse_0c02d + 1.0*LMPD_3_10fthf_c - 1.0*LMPD_3_10fthf_c_reverse_4cbc5 + 1.0*LMPD_4_10fthf_c - 1.0*LMPD_4_10fthf_c_reverse_ebfa8 <= 0.0
13dpg_c: 0.0 <= 1.0*GAPD - 1.0*GAPD_reverse_459c1 + 1.0*PGK - 1.0*PGK_reverse_02696 <= 0.0
2ddg6p_c: 0.0 <= -1.0*EDA + 1.0*EDA_reverse_81f1b + 1.0*EDD - 1.0*EDD_reverse_007a2 <= 0.0
2dmmq8_c: 0.0 <= 1.0*FRD3 - 1.0*FRD3_reverse_78134 - 1.0*G3PD7 + 1.0*G3PD7_reverse_74364 - 1.0*GLYCTO4 + 1.0*GLYCTO4_reverse_9c086 - 1.0*NADH18pp + 1.0*NADH18pp_reverse_8cf33 - 1.0*NADH9 + 1.0*NADH9_reverse_91511 - 1.0*NADPHQR4 + 1.0*NADPHQR4_reverse_ebd03 <= 0.0
2dmmql8_c: 0.0 <= -0.000223*Ec_biomass_iJO1366_WT_53p95M + 0.000223*Ec_biomass_iJO1366_WT_53p95M_reverse_55db7 - 1.0*FRD3 + 1.0*FRD3_reverse_78134 + 1.0*G3PD7 - 1.0*G3PD7_reverse_74364 + 1.0*GLYCTO4 - 1.0*

In [ ]:
thermo_data['metabolites'].keys()


In [ ]:
[r.id for r in mytfa.reactions if r.thermo is not None]


# R. palustris model

Previously constrained

In [ ]:
# Load models

# Directory where the model lives
phbv_model_dir = PHBV_MODEL_DIR / "phbv"
phbv_model_dir.mkdir(parents=True, exist_ok=True)

# Full path to the SBML file
rp_lex_mat_path = PHBV_MODEL_DIR / "model_rpalustris_PHBV_constrained.xml"

# Load model
model_phbv = io.read_sbml_model(
    str(rp_lex_mat_path),
    use_fbc=False
)

print("Model loaded successfully")

Model loaded successfully


In [ ]:
r_phbv  = model_phbv.reactions.get_by_id("PHBVS_syn")     # PHBV copolymer
model_phbv.objective = r_phbv
model_phbv.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
actn__R_e,EX_actn__R_e,0.8502,4,79.08%
cinnm_e,EX_cinnm_e,0.09995,9,20.92%
o2_e,EX_o2_e,0.1999,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
h2_c,DM_h2_c,-0.5001,0,0.00%
h2o_e,EX_h2o_e,-0.0002,0,0.00%
hco3_e,EX_hco3_e,-0.09985,1,99.60%
PHBV_c,SK_phbv_c,-1,0,0.00%
2obut_c,sink_2obut_c,-0.0001,4,0.40%


In [ ]:


#rp_model_mat_path = project_root_palustris / "iDT1294.mat" # Load iDT1294.mat file (base model)
rp_lex_mat_path = PHB_MODEL_TFA_DIR / "lexicon_iDT1294.csv" # load lexicon with 8 reactions
rp_comp_mat_path = PHB_MODEL_TFA_DIR / "compartment_data.json" # "compartments_data_iDT1294_test.json"


cobra_model_mat = model_phbv #load_matlab_model(str(rp_model_mat_path))



# Correct formulas

In [ ]:
import re
from collections import Counter

# --------------------------------
# INPUT DATA
# --------------------------------
Thermodb_meta_interest = [
  "C21H27N7O17P3",
  "C21H27N7O14P2",
  "C23H35N7O17P3S",
  "C21H26N7O14P2",
  "C25H39N7O18P3S",
  "C21H26N7O17P3",
  "C25H37N7O18P3S",
  "C21H33N7O16P3S"
]

GEM_meta_interest = [
  "C21H26N7O17P3",
  "C21H27N7O14P2",
  "C23H34N7O17P3S",
  "C21H26N7O14P2",
  "C25H38N7O18P3S",
  "C21H25N7O17P3",
  "C25H36N7O18P3S",
  "C21H32N7O16P3S"
]

# GEM metabolite IDs in SAME ORDER
gem_met_ids = [
  "nadph_c",
  "nadh_c",
  "accoa_c",
  "nad_c",
  "3hbcoa__R_c",
  "nadp_c",
  "aacoa_c",
  "coa_c",
]

# --------------------------------
# FORMULA PARSING UTILITIES
# --------------------------------
def parse_formula(formula):
  """Convert formula string to Counter"""
  return Counter({
      elem: int(count) if count else 1
      for elem, count in re.findall(r'([A-Z][a-z]?)(\d*)', formula)
  })

def counter_to_formula(counter):
  """Convert Counter back to formula string (Hill order, omit 1s)"""
  parts = []

  # Hill system: C, H, then others alphabetically
  if 'C' in counter:
      parts.append(f"C{counter['C']}" if counter['C'] != 1 else "C")
  if 'H' in counter:
      parts.append(f"H{counter['H']}" if counter['H'] != 1 else "H")

  for el in sorted(counter):
      if el in ('C', 'H'):
          continue
      count = counter[el]
      parts.append(f"{el}{count}" if count != 1 else el)

  return ''.join(parts)


def compare_formulas(thermo_f, gem_f):
  """Return element-wise difference GEM − ThermoDB"""
  t = parse_formula(thermo_f)
  g = parse_formula(gem_f)
  elements = set(t) | set(g)
  return {el: g.get(el, 0) - t.get(el, 0) for el in elements}

def update_gem_formula(gem_f, thermo_f):
  """Add missing elements from ThermoDB into GEM formula"""
  g = parse_formula(gem_f)
  t = parse_formula(thermo_f)

  for el, count in t.items():
      if g.get(el, 0) < count:
          g[el] = count

  return counter_to_formula(g)

# --------------------------------
# APPLY UPDATES TO MODEL
# --------------------------------
for met_id, thermo_f, gem_f in zip(gem_met_ids,
                                  Thermodb_meta_interest,
                                  GEM_meta_interest):

    diff = compare_formulas(thermo_f, gem_f)

    # Safety check (CRITICAL for TFA)
    for el in diff:
        if el in ("C", "N", "P") and diff[el] != 0:
            raise ValueError(
                f"Unsafe formula mismatch for {met_id}: element {el}"
            )

    new_formula = update_gem_formula(gem_f, thermo_f)

    if new_formula != gem_f:
        met = cobra_model_mat.metabolites.get_by_id(met_id) ## update cobra model

        print(f"Updating {met_id}")
        print(f"  old: {gem_f}")
        print(f"  new: {new_formula}")
        print(f"  ThermoDB:       {thermo_f}")

        met.formula = new_formula

        # PRINT FROM MODEL OBJECT
        print(f"  new (model):  {met.formula}")
        print("-" * 40)


print("✅ GEM formulas updated and TFA-ready.")

NameError: name 'cobra_model_mat' is not defined

In [ ]:
annotated = [
    m for m in mytfa.metabolites
    if "seed_id" in m.annotation
]

print("Annotated metabolites:", len(annotated))


NameError: name 'mytfa' is not defined

In [ ]:
rxns_with_annotated_mets = []

for r in mytfa.reactions:
    mets = r.metabolites
    if all(
        ("seed_id" in m.annotation) and (m.formula is not None)
        for m in mets
    ):
        rxns_with_annotated_mets.append(r.id)

print("Thermo-eligible reactions:", len(rxns_with_annotated_mets))


NameError: name 'mytfa' is not defined

# Load medium constraints

In [ ]:
# Apply medium constraints from Montiel-Corona 2022.
def apply_media_from_excel(model, excel_path, close_exchanges=True):
    """
    Apply media constraints from an Excel file with columns:
    Reaction | LowerBound | UpperBound
    """

    #media_df = pd.read_csv(excel_path)
    media_df = pd.read_excel(excel_path)


    required_cols = {"Reaction", "LowerBound", "UpperBound"}
    if not required_cols.issubset(media_df.columns):
        raise ValueError(
            f"Excel file must contain columns: {required_cols}"
        )

    # Optionally close all exchanges first
    if close_exchanges:
        for rxn in model.exchanges:
            rxn.lower_bound = 0
            rxn.upper_bound = 1000

    # Apply media bounds
    for _, row in media_df.iterrows():
        rxn_id = row["Reaction"]

        if rxn_id not in model.reactions:
            raise KeyError(f"Reaction {rxn_id} not found in model")

        rxn = model.reactions.get_by_id(rxn_id)
        rxn.lower_bound = float(row["LowerBound"])
        rxn.upper_bound = float(row["UpperBound"])

    # Check reaction directly associated to PHB production
    EX_acetate_rxn = model.reactions.get_by_id("EX_ac_e")
    print(EX_acetate_rxn.bounds)
    print(EX_acetate_rxn.reaction)

    return model


In [ ]:
# Load medium conditions from Pareto results
project_root = Path(os.getcwd())
excel_path_pareto_medium = project_root / "phbv" / "checkpoint_output" / "Pareto_result_PHBV_model_medium_montiel_buitron_2022_exchange_bounds.xlsx"
print(excel_path_pareto_medium)

/content/driveDL/MyDrive/Genome_scale_metabolic_models_PNSB/Model_Tec_Campos_2023_RPiDT1294/rpalustris_pha_optimization/models/phbv/checkpoint_output/Pareto_result_PHBV_model_medium_montiel_buitron_2022_exchange_bounds.xlsx


In [ ]:
cobra_model_mat

Name,iDT1294
Memory address,7bfc5aed8530
Number of metabolites,2128
Number of reactions,2739
Number of genes,1294
Number of groups,118
Objective expression,1.0*PHBVS_syn - 1.0*PHBVS_syn_reverse_aec1c
Compartments,"c, u, p, e"


In [ ]:
#cobra_model_mat2 = cobra_model_mat.copy()
apply_media_from_excel(cobra_model_mat, excel_path_pareto_medium)

(-1.7, 1000.0)
ac_e <=> 


Name,iDT1294
Memory address,7c8aa3424e00
Number of metabolites,2128
Number of reactions,2739
Number of genes,1294
Number of groups,118
Objective expression,1.0*PHBVS_syn - 1.0*PHBVS_syn_reverse_aec1c
Compartments,"c, u, p, e"


# Add compartments to cobra model

In [ ]:
cobra_model.compartments = {
        'c': {
            'pH': 7.0,
            'ionicStr': 0.25,
            'c_min': 1e-6,
            'c_max': 0.02,
            'membranePot': 0.0,
        },
        'e': {
            'pH': 7.0,
            'ionicStr': 0.25,
            'c_min': 1e-6,
            'c_max': 0.02,
            'membranePot': 0.0,
        },
        'p': {
            'pH': 7.0,
            'ionicStr': 0.25,
            'c_min': 1e-6,
            'c_max': 0.02,
            'membranePot': 0.15,   # periplasm vs cytosol (example)
        },
        'u': {
            'pH': 7.0,
            'ionicStr': 0.25,
            'c_min': 1e-6,
            'c_max': 0.02,
            'membranePot': 0.0,
        }
    }

# Multiple points FBA (version 1)

In [ ]:
model_phbv2 = model_phbv.copy()

In [ ]:
"""
Full corrected script: converts experimental values → fluxes and runs FBA per row.
Features / changes:
 - Uses per-row Time (d) -> TIME_H_ROW
 - Uses INITIAL_BIOMASS as the mean of the provided inoculum list
 - Uses effective biomass X_eff = (INITIAL_BIOMASS + biomass_final)/2 for normalization
 - Supports substrates: acetate, hexanoate, fumarate, lactate, propionate, butyrate, succinate
 - Converts g COD/L (fed) -> mg COD/L -> substrate mg/L -> mmol/L -> mmol·gDW⁻¹·h⁻¹
 - Converts PHB mg/L -> mmol/L -> mmol·gDW⁻¹·h⁻¹ using X_eff and TIME_H_ROW
 - Keeps your medium / photon scaling / reaction checks / CSV outputs
 - Assumes `model_phbv2` (COBRA model) is already loaded in the session
"""

# -------------------------
# USER: input dataset path
# -------------------------
CSV_PATH = "/content/driveDL/My Drive/Genome_scale_metabolic_models_PNSB/Model_Tec_Campos_2023_RPiDT1294/rpalustris_pha_optimization/data/Buitron2025_dataset.csv"

# -------------------------
# CONSTANTS (mg / mmol)
# -------------------------
PHB_MW        = 86.0     # mg/mmol (3HB monomer)
PHV_MW  = 100.12  # mg/mmol (3HV monomer)
ACETATE_MW    = 59.04
HEXANOATE_MW  = 116.16
FUMARATE_MW   = 116.07
LACTATE_MW    = 90.08
PROPIONATE_MW = 74.08
BUTYRATE_MW   = 88.11
SUCCINATE_MW  = 118.09

# COD to acetate mass equivalence (mg COD per mg acetate)
COD_TO_ACETATE_RATIO = 1.066  #Constant

# If Time(d) missing fallback (hours)
FALLBACK_TIME_H = 72.0

# Initial inoculum values provided (mg DW/L)

INITIAL_BIOMASS = 0.094  # gDW/L

# -------------------------
# Load dataset and slice as in original
# -------------------------
df = pd.read_csv(CSV_PATH)

# Select rows 1 to 34 (the original used iloc[1:35])
df = df.iloc[1:35].reset_index(drop=True)

# Drop rows 17 and 18 (indices 15 and 16 after reset)
df = df.drop(index=[15, 16]).reset_index(drop=True)

print("Loaded dataset shape:", df.shape)
print(f"Using INITIAL_BIOMASS (mean inoculum) = {INITIAL_BIOMASS:.6f} gDW/L")

# -------------------------
# Medium conservative upperbounds (unchanged)
# -------------------------
nh4_from_acetate = 0.06590
nh4_from_nh4cl   = 0.07598
total_nh4_uptake = nh4_from_acetate + nh4_from_nh4cl

medium_montiel_buitron_2022_conservative_upperbound = {
    "EX_ac_e"   : 0.2477,
    "EX_nh4_e"  : total_nh4_uptake,
    "EX_pi_e"   : 0.03733,
    "EX_mg2_e"  : 0.01604,
    "EX_na1_e"  : 0.06955,
    "EX_ca2_e"  : 0.003455,
    "EX_fe3_e"  : 0.0004147,
    "EX_zn2_e"  : 0.000002337,
    "EX_mn2_e"  : 0.000000620,
    "EX_bo3_e"  : 0.00001972,
    "EX_cu2_e"  : 0.000000234,
    "EX_ni2_e"  : 0.000000345,
    "EX_mobd_e" : 0.000000559,
    "EX_co2_e"  : 0,
    "EX_o2_e"   : 0,
    # photons (kept)
    "EX_photon410_e": 100,
    "EX_photon430_e": 100,
    "EX_photon450_e": 100,
    "EX_photon470_e": 100,
    "EX_photon490_e": 100,
    "EX_photon510_e": 100,
    "EX_photon530_e": 100,
    "EX_photon550_e": 100,
    "EX_photon570_e": 100,
    "EX_photon590_e": 100,
    "EX_photon610_e": 100,
    "EX_photon630_e": 100,
    "EX_photon650_e": 100,
    "EX_photon670_e": 100,
    "EX_photon690_e": 100
}

# -------------------------
# Safe photon scaling (unchanged)
# -------------------------
def safe_scale_photons(model, illum_value, ref=1000.0):

    def photons_off():
        for rxn in model.reactions:
            if rxn.id.startswith("EX_photon"):
                model.reactions.get_by_id(rxn.id).lower_bound = 0

    # NaN or empty
    if pd.isna(illum_value):
        photons_off()
        return

    illum_str = str(illum_value).lower()

    if illum_str in ["continuous", "light", "constant", "on"]:
        for rxn_id, ub in medium_montiel_buitron_2022_conservative_upperbound.items():
            if rxn_id.startswith("EX_photon") and rxn_id in model.reactions:
                model.reactions.get_by_id(rxn_id).lower_bound = -ub
        return

    extracted = "".join(ch for ch in illum_str if ch.isdigit() or ch == ".")

    if extracted == "":
        photons_off()
        return

    try:
        num = float(extracted)
    except:
        photons_off()
        return

    if num == 0:
        photons_off()
        return

    scale = num / ref

    for rxn_id, ub in medium_montiel_buitron_2022_conservative_upperbound.items():
        if rxn_id.startswith("EX_photon") and rxn_id in model.reactions:
            model.reactions.get_by_id(rxn_id).lower_bound = -scale * ub

# -------------------------
# Helper to safe-get flux from solution
# -------------------------
def get_flux(sol, rxn_id):
    try:
        return float(sol.fluxes.get(rxn_id, np.nan))
    except Exception:
        return np.nan

# -------------------------
# Output directories
# -------------------------
os.makedirs("medium_rows", exist_ok=True)
os.makedirs("fluxes_rows", exist_ok=True)

# -------------------------
# Pre-check reaction ids in GEM
# -------------------------
r_ac_exists     = "EX_ac_e"        in model_phbv2.reactions
r_hxa_exists    = "EX_hxa_e"       in model_phbv2.reactions
r_fum_exists    = "EX_fum_e"       in model_phbv2.reactions
r_lac_exists    = "EX_lac__L_e"    in model_phbv2.reactions
r_ppa_exists    = "EX_ppa_e"       in model_phbv2.reactions
r_but_exists    = "EX_but_e"       in model_phbv2.reactions
r_succ_exists   = "EX_succ_e"      in model_phbv2.reactions
r_phb_exists    = "PHBS_syn"       in model_phbv2.reactions
r_phv_exists     = "PHVS_syn"       in model_phbv2.reactions
r_phbv_exists    = "PHBVS_syn"       in model_phbv2.reactions
r_3hvcoa_exists    = "3hvcoa_syn"       in model_phbv2.reactions
r_biomass_exists= "BIOMASS__1"     in model_phbv2.reactions

print(
    f"Model reaction presence: "
    f"EX_ac_e={r_ac_exists}, EX_hxa_e={r_hxa_exists}, EX_fum_e={r_fum_exists}, "
    f"EX_lac__L_e={r_lac_exists}, EX_ppa_e={r_ppa_exists}, EX_but_e={r_but_exists}, EX_succ_e={r_succ_exists}, "
    f"PHBS_syn={r_phb_exists}, BIOMASS__1={r_biomass_exists}, PHBVS_syn={r_phb_exists}"
)

# -------------------------
# MAIN LOOP
# -------------------------
results = []

for idx, row in df.iterrows():

    row_id = idx + 1
    print(f"\n=== Row {row_id} / {len(df)} ===")

    # copy model for this row
    m = model_phbv2.copy()

    # reset exchange lower bounds to 0 then apply conservative upperbounds
    for ex in m.exchanges:
        ex.lower_bound = 0.0

    for rxn_id, ub in medium_montiel_buitron_2022_conservative_upperbound.items():
        if rxn_id in m.reactions:
            m.reactions.get_by_id(rxn_id).lower_bound = -ub

    # -------------------------
    # Final biomass (from dataset)
    # -------------------------
    biomass_mgL = row.get("Biomass (mg dw/L)", np.nan)
    if pd.isna(biomass_mgL) or biomass_mgL == 0:
        print(f"  WARNING: Biomass missing or zero in row {row_id}; setting biomass_final_gL = NaN")
        biomass_final_gL = np.nan
    else:
        biomass_final_gL = biomass_mgL / 1000.0  # mg -> gDW/L

    # -------------------------
    # Row-specific time handling (Time (d) -> hours)
    # -------------------------
    time_days = row.get("Time (d)", np.nan)
    if pd.isna(time_days) or time_days <= 0:
        print(f"  WARNING: Missing or invalid Time (d) in row {row_id}; using fallback {FALLBACK_TIME_H} h")
        t_hours = FALLBACK_TIME_H
    else:
        t_hours = float(time_days) * 24

    # -------------------------
    # Effective biomass (X_eff = (initial_inoculum + final)/2)
    # -------------------------
    if pd.isna(biomass_final_gL):
        X_eff = np.nan
    else:
        X_eff = (INITIAL_BIOMASS + biomass_final_gL) / 2 #biomass_final_gL #

    print(f"  Time (h) = {t_hours:.2f}, biomass_final_gL = {biomass_final_gL}, X_eff = {X_eff}")

    # -------------------------
    # PHB experimental conversion mg/L -> q_phb (mmol·gDW⁻¹·h⁻¹)
    # -------------------------
    phb_mgL = row.get("PHB (mg/L)", np.nan)
    if pd.isna(phb_mgL) or pd.isna(X_eff) or X_eff == 0:
        q_phb = np.nan
    else:
        phb_mmolL = phb_mgL / PHB_MW
        q_phb = phb_mmolL / (X_eff * t_hours)

    # -------------------------
    # PHV experimental conversion mg/L -> q_phv (mmol·gDW⁻¹·h⁻¹)
    # -------------------------
    phv_mgL = row.get("PHV (mg/L)", np.nan)

    if pd.isna(phv_mgL) or pd.isna(X_eff) or X_eff == 0:
        q_phv = np.nan
    else:
        phv_mmolL = phv_mgL / PHV_MW
        q_phv = phv_mmolL / (X_eff * t_hours)

    # -------------------------
    # Experimental PHBV flux (PHB + PHV)
    # -------------------------
    if not pd.isna(q_phb) and not pd.isna(q_phv):
        q_phbv = q_phb + q_phv
    else:
        q_phbv = np.nan


    # -------------------------
    # COD-fed -> substrate uptake flux (q_sub)
    # -------------------------
    COD_gL_fed = row.get("g substrate COD/L", np.nan)   # g COD / L fed
    substrate_type = str(row.get("Type of substrate", "")).strip().lower()

    q_sub = np.nan
    sub_rxn_id = None

    if not pd.isna(COD_gL_fed) and COD_gL_fed > 0 and not pd.isna(X_eff) and X_eff > 0:
        # convert g COD/L -> mg COD/L
        cod_mgL = COD_gL_fed * 1000.0

        # for acetate we convert COD to acetate mass using COD_TO_ACETATE_RATIO
        if substrate_type in ["acetate", "ac", "acetic acid"]:
            substrate_mgL = cod_mgL / COD_TO_ACETATE_RATIO
            substrate_mmolL = substrate_mgL / ACETATE_MW
            sub_rxn_id = "EX_ac_e"

        elif substrate_type in ["hexanoate", "hexanoic acid", "hxa", "hex"]:
            substrate_mgL = cod_mgL
            substrate_mmolL = substrate_mgL / HEXANOATE_MW
            sub_rxn_id = "EX_hxa_e"

        elif substrate_type in ["fumarate", "fum"]:
            substrate_mgL = cod_mgL
            substrate_mmolL = substrate_mgL / FUMARATE_MW
            sub_rxn_id = "EX_fum_e"

        elif substrate_type in ["lactate", "lac", "lactic acid"]:
            substrate_mgL = cod_mgL
            substrate_mmolL = substrate_mgL / LACTATE_MW
            sub_rxn_id = "EX_lac__L_e"

        elif substrate_type in ["propionate", "ppa", "propionic acid", "prop"]:
            substrate_mgL = cod_mgL
            substrate_mmolL = substrate_mgL / PROPIONATE_MW
            sub_rxn_id = "EX_ppa_e"

        elif substrate_type in ["butyrate", "but", "butyric acid"]:
            substrate_mgL = cod_mgL
            substrate_mmolL = substrate_mgL / BUTYRATE_MW
            sub_rxn_id = "EX_but_e"

        elif substrate_type in ["succinate", "succ", "succinic acid"]:
            substrate_mgL = cod_mgL
            substrate_mmolL = substrate_mgL / SUCCINATE_MW
            sub_rxn_id = "EX_succ_e"

        else:
            # default conservative: treat as acetate COD
            substrate_mgL = cod_mgL / COD_TO_ACETATE_RATIO
            substrate_mmolL = substrate_mgL / ACETATE_MW
            sub_rxn_id = "EX_ac_e"
            print(f"  NOTE row {row_id}: substrate_type '{substrate_type}' not recognized -> assuming acetate conversion")

        # compute q_sub (mmol·gDW⁻¹·h⁻¹)
        try:
            q_sub = substrate_mmolL / (X_eff * t_hours)
        except Exception as e:
            print(f"  ERROR computing q_sub row {row_id}: {e}")
            q_sub = np.nan

    else:
        print(f"  WARNING row {row_id}: Missing COD_fed or X_eff -> q_sub set to NaN")

    # -------------------------
    # Apply substrate flux bounds if reaction exists
    # -------------------------
    if sub_rxn_id and sub_rxn_id in m.reactions and not pd.isna(q_sub):
        # set both bounds to -q_sub to force uptake (negative uptake in COBRA convention)
        m.reactions.get_by_id(sub_rxn_id).lower_bound = -q_sub
        m.reactions.get_by_id(sub_rxn_id).upper_bound = -q_sub
    else:
        if sub_rxn_id and sub_rxn_id not in m.reactions:
            print(f"  Reaction {sub_rxn_id} not present in model; skipping substrate constraint for row {row_id}")

    # -------------------------
    # Illumination -> photon scaling
    # -------------------------
    illum = row.get("Illumination", np.nan)
    safe_scale_photons(m, illum)

    # -------------------------
    # PHB physiological cap and objective (Option A)
    # -------------------------

#PHVS_syn: imbalance -> {'H': 2.0, 'O': 1.0}
#PHBS_syn: imbalance -> {'C': 1.0, 'H': 2.0}
#PHBVS_syn: imbalance -> {}

    phb_rxn_id = 'PHBVS_syn'
    mu_max = 0.08 # growth rate
    f = 0.50       # fraction of the maximum growth-associated carbon/flux that is diverted to PHB synthesis

    v_phbv_max = 1 #mu_max * f * 1000 / PHV_MW
    v_phbv_max_eff = mu_max * f * 1000 / PHV_MW + PHB_MW
    print(f" Calculated v_phbv_max (mmol/gDW/h) = {v_phbv_max_eff:.6f}")
    print(f" v_phbv_max (mmol/gDW/h) = {v_phbv_max:.6f}")

    if r_phb_exists and phb_rxn_id in m.reactions:
        phb_rxn = m.reactions.get_by_id(phb_rxn_id)
        phb_rxn.lower_bound = 0.0
        phb_rxn.upper_bound = v_phbv_max
        m.objective = phb_rxn # define OBJECTIVE
    else:
        print(f"WARNING: PHBS_syn not found in model for row {row_id}")

    # -------------------------
    # Minimal biomass requirement (if present)
    # Enabling biomass bounds turn predictions infeasible (probably because biomass and PHB produciton are competing objectives)
    # -------------------------
    if r_biomass_exists and "BIOMASS__1" in m.reactions:
        biomass_rxn = m.reactions.get_by_id("BIOMASS__1")
        biomass_rxn.lower_bound = 0 #0.001626
        biomass_rxn.upper_bound = 0

    # optional model fixes
    #if "EX_h2_e" in m.reactions:
    #    m.reactions.get_by_id("EX_h2_e").lower_bound = 0
    #    m.reactions.get_by_id("EX_h2_e").upper_bound = 1000.0

    #if "EX_no3_e" in m.reactions:
    #    no3 = m.reactions.get_by_id("EX_no3_e")
    #    no3.lower_bound = -1000.0
    #    no3.upper_bound = 0

    #if "ATPM" in m.reactions:
    #   m.reactions.get_by_id("ATPM").lower_bound = 10


    # Disable homopolymer synthetases to avoid competition (temporary)
      #"PHVS_syn"
    #for rid in ["PHVS_syn", "PHBVS_syn"]:
    #    if rid in m.reactions:
    #        m.reactions.get_by_id(rid).lower_bound = 0
    #        m.reactions.get_by_id(rid).upper_bound = 0

    m.summary()

    # Finla check of key reactions before optimization
    EX_ac_e_rxn = m.reactions.get_by_id('EX_ac_e') #PDH #PHBS_syn #EX_ac_e
    print(f"{EX_ac_e_rxn.id}: lower = {EX_ac_e_rxn.lower_bound}, upper = {EX_ac_e_rxn.upper_bound}") # bounds
    # -------------------------
    # Run FBA
    # -------------------------
    sol = m.optimize()
    if sol.status != "optimal":
        print(f"  Row {row_id}: no optimal solution (status={sol.status}); saving NaNs")
        fluxes = sol.fluxes if hasattr(sol, "fluxes") else pd.Series()
    else:
        fluxes = sol.fluxes

    # Save model

    # Directory where the model lives
    phb_model_dir = project_root / "phbv"

    # Ensure folder exists
    phb_model_dir.mkdir(parents=True, exist_ok=True)

    # Target file
    phb_model_file = phb_model_dir / "model_rpalustris_PHBV_constrained_FBA_loop.xml"

    # Save COBRA model
    cobra.io.write_sbml_model(m, str(phb_model_file))

    # -------------------------
    # Save medium CSV for this row
    # -------------------------
    try:
        medium_df = pd.DataFrame({
            "reaction": [rxn.id for rxn in m.exchanges],
            "lb":       [rxn.lower_bound for rxn in m.exchanges],
            "ub":       [rxn.upper_bound for rxn in m.exchanges],
            "flux":     [float(fluxes.get(rxn.id, np.nan)) for rxn in m.exchanges]
        })
        medium_df.to_csv(f"medium_rows/medium_row_{row_id}.csv", index=False)
    except Exception as e:
        print(f"  Error saving medium_row_{row_id}.csv: {e}")

    # -------------------------
    # Save full flux vector for this row
    # -------------------------
    try:
        fluxes_df = pd.DataFrame({
            "reaction": fluxes.index.astype(str),
            "flux": fluxes.values
        })
        fluxes_df.to_csv(f"fluxes_rows/fluxes_row_{row_id}.csv", index=False)
    except Exception as e:
        print(f"  Error saving fluxes_row_{row_id}.csv: {e}")

    # -------------------------
    # Collect outputs for summary table
    # -------------------------
    biomass_flux = get_flux(sol, "BIOMASS__1") if r_biomass_exists else np.nan
    #phb_flux_model = get_flux(sol, "PHBS_syn") if r_phb_exists else np.nan
    #phv_flux_model = get_flux(sol, "PHVS_syn") if r_phv_exists else np.nan
    phbv_flux_model = get_flux(sol, "PHBVS_syn") if r_phbv_exists else np.nan
    hv_flux_model = get_flux(sol, "3hvcoa_syn") if r_phbv_exists else np.nan
    hb_flux_model = get_flux(sol, "AACOAR_syn") if r_phbv_exists else np.nan
    PHBV_model_accumulation = get_flux(sol, "DM_PHBV_c") if r_phbv_exists else np.nan
    sub_flux_model = get_flux(sol, sub_rxn_id) if sub_rxn_id and sub_rxn_id in m.reactions else np.nan

    # top 10 absolute fluxes
    try:
      top10 = fluxes.abs().sort_values(ascending=False).head(10)

      records = []
      for rxn_id in top10.index:
          rxn = m.reactions.get_by_id(rxn_id)

          records.append({
              "reaction_id": rxn_id,
              "flux": float(fluxes.get(rxn_id, np.nan)),
              "name": rxn.name if rxn.name else "Unnamed",
              "subsystem": rxn.subsystem if rxn.subsystem else "Unassigned"
          })

      # Final DataFrame with full metadata
      top10_df = pd.DataFrame(records)

      # JSON/dict format for later use if needed
      top10_dict = {row["reaction_id"]: row["flux"] for row in records}

      # store for wordcloud later
      top10_subsystems = list(top10_df["subsystem"])

      # Save top 10 flux details per row
      top10_df.to_csv(f"fluxes_rows/top10_fluxes_row_{row_id}.csv", index=False)

    except Exception as e:
        print(f"Error extracting top10 fluxes row {row_id}: {e}")
        top10_df = pd.DataFrame(columns=["reaction_id","flux","name","subsystem"])
        top10_dict = {}
        top10_subsystems = []

    print(f"PHB_exp_flux  : {q_phb}")
    print(f"PHV_exp_flux  : {q_phv}")
    print(f"PHBV_exp_flux : {q_phbv}")
    #print(f"PHB_model_flux: {phb_flux_model}")
    #print(f"PHV_model_flux: {phv_flux_model}")
    print(f"3HV_model_flux: {hv_flux_model}")
    print(f"3HB_model_flux: {hb_flux_model}")
    print(f"PHBV_model_flux:{phbv_flux_model}")
    print(f"PHBV_model_accumulation:{PHBV_model_accumulation}")
    print(f"Biomass_model_flux:{biomass_flux}")

    results.append({
    "Row": row_id,

    # original experiment columns
    "Biomass_mgL": biomass_mgL,
    "COD_gL_fed": COD_gL_fed,
    "Substrate_type": substrate_type,
    "Time_h": t_hours,
    "PHB_mgL": phb_mgL,
    "PHV_mgL": phv_mgL,

    # biomass
    "biomass_initial_gL": INITIAL_BIOMASS,
    "biomass_final_gL": biomass_final_gL,
    "biomass_eff_gL": X_eff,

    # experimental fluxes
    "PHB_exp_flux": q_phb,
    "PHV_exp_flux": q_phv,
    "PHBV_exp_flux": q_phbv,
    "Substrate_exp_flux": q_sub,
    "sub_rxn_id": sub_rxn_id,

    # model outputs
    #"PHB_model_flux": phb_flux_model,
    #"PHV_model_flux": sol.fluxes.get("PHVS_syn", np.nan) if sol.status == "optimal" else np.nan,
    "PHBV_model_flux": sol.fluxes.get("PHBVS_syn", np.nan) if sol.status == "optimal" else np.nan,
    "PHBV_model_accumulation": sol.fluxes.get("DM_PHBV_c", np.nan) if sol.status == "optimal" else np.nan,
    "3HV_model_flux": sol.fluxes.get("3hvcoa_syn", np.nan) if sol.status == "optimal" else np.nan,
    "3HB_model_flux": sol.fluxes.get("AACOAR_syn", np.nan) if sol.status == "optimal" else np.nan,
    "Biomass_model_flux": biomass_flux,
    "Substrate_model_flux": sub_flux_model,
    })


# -------------------------
# SAVE SUMMARY CSV and MERGE WITH ORIGINAL ROWS
# -------------------------
results_df = pd.DataFrame(results)
results_df.to_csv("FBA_PHBV_results_summary.csv", index=False)

df_merged = pd.concat([df.reset_index(drop=True), results_df.reset_index(drop=True)], axis=1)
df_merged.to_csv(phbv_model_dir/"Buitron_dataset_augmented_with_FBA_fluxes_PHBV_before_tfa.csv", index=False)

print("\nSaved: FBA_PHBV_results_summary.csv and Buitron_dataset_augmented_with_FBA_fluxes_PHBV.csv")


Loaded dataset shape: (32, 23)
Using INITIAL_BIOMASS (mean inoculum) = 0.094000 gDW/L
Model reaction presence: EX_ac_e=True, EX_hxa_e=True, EX_fum_e=True, EX_lac__L_e=True, EX_ppa_e=True, EX_but_e=True, EX_succ_e=True, PHBS_syn=True, BIOMASS__1=True, PHBVS_syn=True

=== Row 1 / 32 ===
  Time (h) = 120.00, biomass_final_gL = 2.467, X_eff = 1.2805
 Calculated v_phbv_max (mmol/gDW/h) = 0.399521
 v_phbv_max (mmol/gDW/h) = 1.000000
EX_ac_e: lower = -0.5304604174858837, upper = -0.5304604174858837
PHB_exp_flux  : 0.0064745784259419015
PHV_exp_flux  : 0.0007780589103306411
PHBV_exp_flux : 0.007252637336272543
3HV_model_flux: 0.050236250077610144
3HB_model_flux: 0.20094500031044055
PHBV_model_flux:0.25118125038805067
PHBV_model_accumulation:0.25118125038805067
Biomass_model_flux:0.0

=== Row 2 / 32 ===
  Time (h) = 120.00, biomass_final_gL = 1.15, X_eff = 0.622
 Calculated v_phbv_max (mmol/gDW/h) = 0.399521
 v_phbv_max (mmol/gDW/h) = 1.000000
EX_ac_e: lower = -1.0920491392133023, upper = -1.09

# Multiple points fba and tfa (version 2)


In [ ]:
import cobra
import pytfa
import pandas as pd
import numpy as np
import os
from pathlib import Path
from cobra.flux_analysis import flux_variability_analysis
from pytfa.io import read_lexicon, annotate_from_lexicon, read_compartment_data, apply_compartment_data

# ===============================
# USER INPUT
# ===============================
CSV_PATH = "/content/driveDL/My Drive/Genome_scale_metabolic_models_PNSB/Model_Tec_Campos_2023_RPiDT1294/rpalustris_pha_optimization/data/Buitron2025_dataset.csv"
INITIAL_BIOMASS = 0.094     # gDW/L
FALLBACK_TIME_H = 72.0

# ===============================
# CONSTANTS
# ===============================
PHB_MW = 86.0
ACETATE_MW = 59.04
COD_TO_ACETATE_RATIO = 1.066

# ===============================
# LOAD DATA
# ===============================
df = pd.read_csv(CSV_PATH)
df = df.iloc[1:35].drop(index=[15, 16]).reset_index(drop=True)

# ===============================
# MEDIUM (unchanged)
# ===============================
medium_ub = {
    "EX_ac_e": 0.2477,
    "EX_nh4_e": 0.14188,
    "EX_pi_e": 0.03733,
    "EX_o2_e": 0,
    "EX_co2_e": 0,
    **{f"EX_photon{x}_e": 10 for x in range(410, 700, 20)}
}

# ===============================
# PHOTON SCALING
# ===============================
def scale_photons(model, illum, ref=1000):
    for rxn in model.reactions:
        if rxn.id.startswith("EX_photon"):
            rxn.lower_bound = 0

    if pd.isna(illum):
        return

    s = str(illum).lower()
    if s in ["continuous", "light", "on"]:
        for rid, ub in medium_ub.items():
            if rid.startswith("EX_photon") and rid in model.reactions:
                model.reactions.get_by_id(rid).lower_bound = -ub
        return

    try:
        val = float("".join(c for c in s if c.isdigit() or c == "."))
        scale = val / ref
        for rid, ub in medium_ub.items():
            if rid.startswith("EX_photon") and rid in model.reactions:
                model.reactions.get_by_id(rid).lower_bound = -scale * ub
    except:
        pass

# ===============================
# SOLVER SETTINGS
# ===============================
def apply_solver_settings(model):
    model.solver = "glpk"
    model.solver.configuration.tolerances.feasibility = 1e-9
    model.solver.configuration.presolve = True

# ===============================
# LOAD TFA METADATA
# ===============================
lexicon = read_lexicon(str(rp_lex_mat_path))
comp_data = read_compartment_data(str(rp_comp_mat_path))

# ===============================
# OUTPUT
# ===============================
results = []

# ===============================
# MAIN LOOP
# ===============================
for i, row in df.iterrows():
    print(f"\n--- ROW {i+1} ---")

    # ---------------------------
    # BUILD COBRA MODEL
    # ---------------------------
    m = model_phbv2.copy() # DEFINE MODEL

    for ex in m.exchanges:
        ex.lower_bound = 0

    for rid, ub in medium_ub.items():
        if rid in m.reactions:
            m.reactions.get_by_id(rid).lower_bound = -ub

    # ---------------------------
    # TIME & BIOMASS
    # ---------------------------
    t_h = row.get("Time (d)", np.nan)
    t_h = FALLBACK_TIME_H if pd.isna(t_h) else float(t_h) * 24

    biomass_final = row.get("Biomass (mg dw/L)", np.nan)
    biomass_final = np.nan if pd.isna(biomass_final) else biomass_final / 1000
    X_eff = (INITIAL_BIOMASS + biomass_final) / 2 if not pd.isna(biomass_final) else np.nan

    # ---------------------------
    # SUBSTRATE UPTAKE
    # ---------------------------
    COD = row.get("g substrate COD/L", np.nan)
    if not pd.isna(COD) and X_eff > 0:
        mmol_L = (COD * 1000 / COD_TO_ACETATE_RATIO) / ACETATE_MW
        q_sub = mmol_L / (X_eff * t_h)
        if "EX_ac_e" in m.reactions:
            rxn = m.reactions.get_by_id("EX_ac_e")
            rxn.lower_bound = -q_sub
            rxn.upper_bound = -q_sub

    # ---------------------------
    # PHOTONS
    # ---------------------------
    scale_photons(m, row.get("Illumination", np.nan))

    # ---------------------------
    # PHB CAP & OBJECTIVE
    # ---------------------------
    phb_rxn_id = 'PHBVS_syn'
    mu_max = 0.08 # growth rate
    f = 0.50       # fraction of the maximum growth-associated carbon/flux that is diverted to PHB synthesis

    v_phbv_max = 1 #mu_max * f * 1000 / PHV_MW
    v_phbv_max_eff = mu_max * f * 1000 / PHV_MW + PHB_MW
    print(f" Calculated v_phbv_max (mmol/gDW/h) = {v_phbv_max_eff:.6f}")
    print(f" v_phbv_max (mmol/gDW/h) = {v_phbv_max:.6f}")

    if r_phb_exists and phb_rxn_id in m.reactions:
        phb_rxn = m.reactions.get_by_id(phb_rxn_id)
        phb_rxn.lower_bound = 0.0
        phb_rxn.upper_bound = v_phbv_max
        m.objective = phb_rxn # define OBJECTIVE
    else:
        print(f"WARNING: PHBS_syn not found in model for row {row_id}")

    # ===========================
    # FBA
    # ===========================
    fba_sol = m.optimize()

    # ===========================
    # TFA (ROW-SPECIFIC)
    # ===========================
    mytfa = pytfa.ThermoModel(thermo_data, m)
    annotate_from_lexicon(mytfa, lexicon)
    apply_compartment_data(mytfa, comp_data)

    mytfa.objective = "PHBS_syn"
    apply_solver_settings(mytfa)

    mytfa.prepare()
    tfa_sol = mytfa.optimize()

    # ===========================
    # STORE RESULTS
    # ===========================
    results.append({
        "row": i+1,
        "PHB_FBA": fba_sol.fluxes.get("PHBS_syn", np.nan) if fba_sol.status == "optimal" else np.nan,
        "PHB_TFA": tfa_sol.fluxes.get("PHBS_syn", np.nan) if tfa_sol.status == "optimal" else np.nan,
        "Biomass_FBA": fba_sol.fluxes.get("BIOMASS__1", np.nan) if fba_sol.status == "optimal" else np.nan,
        "Biomass_TFA": tfa_sol.fluxes.get("BIOMASS__1", np.nan) if tfa_sol.status == "optimal" else np.nan,
        "status_FBA": fba_sol.status,
        "status_TFA": tfa_sol.status
    })

# ===============================
# SAVE
# ===============================
results_df = pd.DataFrame(results)
results_df.to_csv(project_root_palustris / "FBA_vs_TFA_PHB_results.csv", index=False)

print("\nDONE: FBA_vs_TFA_PHB_results.csv")



--- ROW 1 ---
 Calculated v_phbv_max (mmol/gDW/h) = 86.399521
 v_phbv_max (mmol/gDW/h) = 1.000000


2026-01-13 17:26:16,134 - thermomodel_None - INFO - # Model initialized with units kcal/mol and temperature 298.15 K
INFO:thermomodel_None:# Model initialized with units kcal/mol and temperature 298.15 K
2026-01-13 17:26:16,141 - thermomodel_None - WARNING - gam6p_c  not found in annotations
2026-01-13 17:26:16,146 - thermomodel_None - WARNING - cgly_c  not found in annotations
2026-01-13 17:26:16,150 - thermomodel_None - WARNING - achms_c  not found in annotations
2026-01-13 17:26:16,155 - thermomodel_None - WARNING - pcox_u  not found in annotations
2026-01-13 17:26:16,160 - thermomodel_None - WARNING - octe9ACP_c  not found in annotations
2026-01-13 17:26:16,163 - thermomodel_None - WARNING - fdp_c  not found in annotations
2026-01-13 17:26:16,168 - thermomodel_None - WARNING - 5caiz_c  not found in annotations
2026-01-13 17:26:16,170 - thermomodel_None - WARNING - pep_c  not found in annotations
2026-01-13 17:26:16,174 - thermomodel_None - WARNING - hgbam_c  not found in annotation

TypeError: string indices must be integers, not 'str'

# Multiple points fba tfba (version 3)

Checkpoints to save data each iteration



In [ ]:

from tqdm import tqdm
for i in tqdm(range(10000)):
    pass


# -------------------------
r_ac_exists     = "EX_ac_e"        in cobra_model_mat.reactions
r_hxa_exists    = "EX_hxa_e"       in cobra_model_mat.reactions
r_fum_exists    = "EX_fum_e"       in cobra_model_mat.reactions
r_lac_exists    = "EX_lac__L_e"    in cobra_model_mat.reactions
r_ppa_exists    = "EX_ppa_e"       in cobra_model_mat.reactions
r_but_exists    = "EX_but_e"       in cobra_model_mat.reactions
r_succ_exists   = "EX_succ_e"      in cobra_model_mat.reactions
r_phb_exists    = "PHBS_syn"       in cobra_model_mat.reactions
r_phv_exists     = "PHVS_syn"       in cobra_model_mat.reactions
r_phbv_exists    = "PHBVS_syn"       in cobra_model_mat.reactions
r_3hvcoa_exists    = "3hvcoa_syn"       in cobra_model_mat.reactions
r_biomass_exists= "BIOMASS__1"     in cobra_model_mat.reactions

# ===============================
# USER INPUT
# ===============================
#CSV_PATH = "/content/driveDL/My Drive/Genome_scale_metabolic_models_PNSB/Model_Tec_Campos_2023_RPiDT1294/rpalustris_pha_optimization/data/Buitron2025_dataset.csv"
OUT_CSV = project_root_palustris / "FBA_vs_TFA_PHB_results.csv"

INITIAL_BIOMASS = 0.094
FALLBACK_TIME_H = 72.0

# ===============================
# CONSTANTS
# ===============================
PHB_MW = 86.0
PHV_MW  = 100.12  # mg/mmol (3HV monomer)
ACETATE_MW = 59.04
COD_TO_ACETATE_RATIO = 1.066

# ===============================
# LOAD DATA
# ===============================
df = pd.read_csv(phbv_model_dir / "subset_pareto_optimal_conditions_for_tfa.csv") #pd.read_csv(CSV_PATH)
df = df.iloc[1:35].drop(index=[15, 16]).reset_index(drop=True)

# ===============================
# MEDIUM
# ===============================
medium_ub = {
    "EX_ac_e": 0.2477,
    "EX_nh4_e": 0.14188,
    "EX_pi_e": 0.03733,
    "EX_o2_e": 0,
    "EX_co2_e": 0,
    **{f"EX_photon{x}_e": 10 for x in range(410, 700, 20)}
}

PHBV_REACTIONS = [
    "ACACT1r",
    "ACACCT",
    "AACOAR_syn",
    "HACD1_2",
    "HACD1",
    "HACD1i",
    "KAT1",
]

# ===============================
# PHOTON SCALING
# ===============================
def scale_photons(model, illum, ref=1000):
    for rxn in model.reactions:
        if rxn.id.startswith("EX_photon"):
            rxn.lower_bound = 0

    if pd.isna(illum):
        return

    s = str(illum).lower()

    if s in ["continuous", "light", "on"]:
        for rid, ub in medium_ub.items():
            if rid.startswith("EX_photon") and rid in model.reactions:
                model.reactions.get_by_id(rid).lower_bound = -ub
        return

    try:
        val = float("".join(c for c in s if c.isdigit() or c == "."))
        scale = val / ref
        for rid, ub in medium_ub.items():
            if rid.startswith("EX_photon") and rid in model.reactions:
                model.reactions.get_by_id(rid).lower_bound = -scale * ub
    except Exception:
        pass

# ===============================
# SOLVER SETTINGS
# ===============================
def apply_solver_settings(model):
    model.solver = "glpk"
    model.solver.configuration.tolerances.feasibility = 1e-9
    model.solver.configuration.presolve = True

# ===============================
# LOAD TFA METADATA
# ===============================
lexicon = read_lexicon(str(rp_lex_mat_path))
comp_data = read_compartment_data(str(rp_comp_mat_path))


# ============================================================
# OUTPUT HEADER
# ============================================================
BASE_COLUMNS = [
    "row",
    "Time_d",
    "Time_h",
    "Biomass_final_gDW_L",
    "X_eff_gDW_L",
    "COD_g_L",
    "Illumination",
    "q_ac_mmol_gDW_h",
    "EX_ac_e_lb",
    "EX_ac_e_ub",
    "PHBV_FBA",
    "PHBV_TFA",
    "Biomass_FBA",
    "Biomass_TFA",
    "status_FBA",
    "status_TFA",
]

PHBV_FBA_COLS = [f"{r}_FBA" for r in PHBV_REACTIONS]
PHBV_TFA_COLS = [f"{r}_TFA" for r in PHBV_REACTIONS]

ALL_COLUMNS = BASE_COLUMNS + PHBV_FBA_COLS + PHBV_TFA_COLS

if not OUT_CSV.exists():
    pd.DataFrame(columns=ALL_COLUMNS).to_csv(OUT_CSV, index=False)



# ===============================
# MAIN LOOP (APPEND EACH ROW)
# ===============================
for i, row in df.iterrows():
    print(f"\n--- ROW {i+1} ---")

    try:
        # ---------------------------
        # BUILD MODEL
        # ---------------------------
        m = cobra_model_mat.copy()

        for ex in m.exchanges:
            ex.lower_bound = 0

        for rid, ub in medium_ub.items():
            if rid in m.reactions:
                m.reactions.get_by_id(rid).lower_bound = -ub

        # ---------------------------
        # TIME & BIOMASS
        # ---------------------------
        t_h = row.get("Time (d)", np.nan)
        t_h = FALLBACK_TIME_H if pd.isna(t_h) else float(t_h) * 24

        biomass_final = row.get("Biomass (mg dw/L)", np.nan)
        biomass_final = biomass_final / 1000 if not pd.isna(biomass_final) else np.nan

        X_eff = (INITIAL_BIOMASS + biomass_final) / 2 if not pd.isna(biomass_final) else np.nan

        # ---------------------------
        # COD → ACETATE FLUX
        # ---------------------------
        COD = row.get("g substrate COD/L", np.nan)
        q_ac = np.nan

        if not pd.isna(COD) and X_eff > 0:
            mmol_L = (COD * 1000 / COD_TO_ACETATE_RATIO) / ACETATE_MW
            q_ac = mmol_L / (X_eff * t_h)

            rxn = m.reactions.get_by_id("EX_ac_e")
            rxn.lower_bound = -q_ac
            rxn.upper_bound = -q_ac

        # ---------------------------
        # PHOTONS
        # ---------------------------
        scale_photons(m, row.get("Illumination", np.nan))

        # ---------------------------
        # PHB CAP & OBJECTIVE
        # ---------------------------
        phb_rxn_id = 'PHBVS_syn'
        mu_max = 0.08 # growth rate
        f = 0.50       # fraction of the maximum growth-associated carbon/flux that is diverted to PHB synthesis

        v_phbv_max = 1 #mu_max * f * 1000 / PHV_MW
        v_phbv_max_eff = mu_max * f * 1000 / PHV_MW + PHB_MW
        print(f" Calculated v_phbv_max (mmol/gDW/h) = {v_phbv_max_eff:.6f}")
        print(f" v_phbv_max (mmol/gDW/h) = {v_phbv_max:.6f}")

        if r_phbv_exists and phb_rxn_id in m.reactions:
            phb_rxn = m.reactions.get_by_id(phb_rxn_id)
            phb_rxn.lower_bound = 0.0
            phb_rxn.upper_bound = v_phbv_max
            m.objective = phb_rxn # define OBJECTIVE
        else:
            print(f"WARNING: PHBS_syn not found in model for row {row_id}")

        #m.reactions.get_by_id("ATPM").lower_bound = 20

        apply_solver_settings(m)

        # ---------------------------
        # FBA
        # ---------------------------
        fba_sol = m.optimize()

        # ---------------------------
        # TFA
        # ---------------------------
        mytfa = pytfa.ThermoModel(thermo_data, m)
        annotate_from_lexicon(mytfa, lexicon)
        apply_compartment_data(mytfa, comp_data)

        mytfa.objective = "PHBVS_syn"
        apply_solver_settings(mytfa)
        mytfa.prepare()
        tfa_sol = mytfa.optimize()

        result = {
            "row": i + 1,
            "X_eff": X_eff,
            "COD_g_L": COD,
            "q_ac_mmol_gDW_h": q_ac,
            "PHBV_FBA": fba_sol.fluxes.get("PHBVS_syn", np.nan),
            "PHBV_TFA": tfa_sol.fluxes.get("PHBVS_syn", np.nan),
            "Biomass_FBA": fba_sol.fluxes.get("BIOMASS__1", np.nan),
            "Biomass_TFA": tfa_sol.fluxes.get("BIOMASS__1", np.nan),
            "status_FBA": fba_sol.status,
            "status_TFA": tfa_sol.status,
        }

    except Exception as e:
        print(f"Row {i+1} failed: {e}")
        result = {
            "row": i + 1,
            "X_eff": np.nan,
            "COD_g_L": np.nan,
            "q_ac_mmol_gDW_h": np.nan,
            "PHB_FBA": np.nan,
            "PHB_TFA": np.nan,
            "Biomass_FBA": np.nan,
            "Biomass_TFA": np.nan,
            "status_FBA": "error",
            "status_TFA": "error",
        }

    # ---------------------------
    # APPEND IMMEDIATELY
    # ---------------------------
    pd.DataFrame([result]).to_csv(
        OUT_CSV,
        mode="a",
        header=False,
        index=False
    )

    for i, row in tqdm(df.iterrows(), total=len(df), desc="Running FBA/TFA"):
    tqdm.write(f"--- ROW {i+1} ---")

print("\nDONE — results safely saved per row.")


# Dumbell plot TFBA vs FBA  

In [ ]:
# Load TFBA and FBA results
df_results_fba_tfba = pd.read_csv(phbv_model_dir / "FBA_vs_TFA_PHB_results.csv")
len(df_results_fba_tfba)

df = df_results_fba_tfba.copy()

# Keep only successful runs
df = df[
    (df["status_FBA"] == "optimal") &
    (df["status_TFA"] == "optimal")
].reset_index(drop=True)

# Sort by TFA flux for clean ordering
df = df.sort_values("PHBV_TFA").reset_index(drop=True)

# Best solution = highest TFA PHBV flux
best_idx = df["PHBV_TFA"].idxmax()



# Opt: Quantify thermodynamic thightening


In [ ]:
# Quantify thermodynamic thightening
df["flux_loss_pct"] = 100 * (df["PHBV_FBA"] - df["PHBV_TFA"]) / df["PHBV_FBA"]


# *Multiple points fba tfba (version 4).

Include operational and calculated parameters, fba, tfba, and their reaction fluxes

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import cobra
import pytfa
import pandas as pd
import numpy as np
from pathlib import Path

from pytfa.io import (
    read_lexicon,
    annotate_from_lexicon,
    read_compartment_data,
    apply_compartment_data,
)

# ============================================================
# REQUIRED PRE-LOADED OBJECTS (ASSUMED TO EXIST)
# ============================================================
# cobra_model_mat
# thermo_data
# rp_lex_mat_path
# rp_comp_mat_path
# project_root_palustris
# ============================================================

# ============================================================
# USER INPUT
# ============================================================

#model_phbv2 = model_phbv.copy()
#CSV_PATH = "/content/driveDL/My Drive/Genome_scale_metabolic_models_PNSB/Model_Tec_Campos_2023_RPiDT1294/rpalustris_pha_optimization/data/Buitron2025_dataset.csv"

OUT_CSV = PHB_MODEL_TFA_DIR / "FBA_TFA_PHB_detailed_results_phbv_20260308.csv"

INITIAL_BIOMASS = 0.094      # gDW/L
FALLBACK_TIME_H = 72.0

# ============================================================
# CONSTANTS
# ============================================================
PHB_MW = 86.0
PHV_MW  = 100.12  # mg/mmol (3HV monomer)
ACETATE_MW = 59.04
COD_TO_ACETATE_RATIO = 1.066

PHB_REACTIONS = [
    "ACACT1r",
    "ACACCT",
    "AACOAR_syn",
    "HACD1_2",
    "HACD1",
    "HACD1i",
    "KAT1",
]

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv(PHBV_MODEL_DIR / "subset_pareto_optimal_conditions_for_tfa.csv")
df = df.iloc[1:35].drop(index=[15, 16]).reset_index(drop=True)

# ============================================================
# MEDIUM
# ============================================================
medium_ub = {
    "EX_ac_e": 0.2477,
    "EX_nh4_e": 0.14188,
    "EX_pi_e": 0.03733,
    "EX_o2_e": 0,
    "EX_co2_e": 0,
    **{f"EX_photon{x}_e": 10 for x in range(410, 700, 20)},
}

# ============================================================
# PHOTON SCALING
# ============================================================
def scale_photons(model, illum, ref=1000):
    for rxn in model.reactions:
        if rxn.id.startswith("EX_photon"):
            rxn.lower_bound = 0

    if pd.isna(illum):
        return

    s = str(illum).lower()
    if s in ["continuous", "light", "on"]:
        for rid, ub in medium_ub.items():
            if rid.startswith("EX_photon") and rid in model.reactions:
                model.reactions.get_by_id(rid).lower_bound = -ub
        return

    try:
        val = float("".join(c for c in s if c.isdigit() or c == "."))
        scale = val / ref
        for rid, ub in medium_ub.items():
            if rid.startswith("EX_photon") and rid in model.reactions:
                model.reactions.get_by_id(rid).lower_bound = -scale * ub
    except Exception:
        pass

# ============================================================
# SOLVER SETTINGS
# ============================================================
def apply_solver_settings(model):
    model.solver = "glpk"
    model.solver.configuration.tolerances.feasibility = 1e-9
    model.solver.configuration.presolve = True

# ============================================================
# LOAD TFA METADATA
# ============================================================
lexicon = read_lexicon(str(rp_lex_mat_path))
comp_data = read_compartment_data(str(rp_comp_mat_path))

# ============================================================
# OUTPUT HEADER
# ============================================================
BASE_COLUMNS = [
    "row",
    "Time_d",
    "Time_h",
    "Biomass_final_gDW_L",
    "X_eff_gDW_L",
    "COD_g_L",
    "Illumination",
    "q_ac_mmol_gDW_h",
    "EX_ac_e_lb",
    "EX_ac_e_ub",
    "PHB_FBA",
    "PHB_TFA",
    "Biomass_FBA",
    "Biomass_TFA",
    "status_FBA",
    "status_TFA",
]

PHB_FBA_COLS = [f"{r}_FBA" for r in PHB_REACTIONS]
PHB_TFA_COLS = [f"{r}_TFA" for r in PHB_REACTIONS]

ALL_COLUMNS = BASE_COLUMNS + PHB_FBA_COLS + PHB_TFA_COLS

if not OUT_CSV.exists():
    pd.DataFrame(columns=ALL_COLUMNS).to_csv(OUT_CSV, index=False)

# ============================================================
# MAIN LOOP (APPEND PER ROW)
# ============================================================
for i, row in df.iterrows():
    print(f"\n--- ROW {i+1} ---")

    try:
        # ---------------------------
        # RAW INPUT
        # ---------------------------
        time_d = row.get("Time (d)", np.nan)
        illum = row.get("Illumination", np.nan)
        COD = row.get("g substrate COD/L", np.nan)

        biomass_final = row.get("Biomass (mg dw/L)", np.nan)
        biomass_final = biomass_final / 1000 if not pd.isna(biomass_final) else np.nan

        # ---------------------------
        # DERIVED PARAMETERS
        # ---------------------------
        time_h = FALLBACK_TIME_H if pd.isna(time_d) else float(time_d) * 24
        X_eff = (INITIAL_BIOMASS + biomass_final) / 2 if not pd.isna(biomass_final) else np.nan

        q_ac = np.nan
        ex_lb = np.nan
        ex_ub = np.nan

        # ---------------------------
        # BUILD MODEL
        # ---------------------------
        m = cobra_model_mat.copy()

        for ex in m.exchanges:
            ex.lower_bound = 0

        for rid, ub in medium_ub.items():
            if rid in m.reactions:
                m.reactions.get_by_id(rid).lower_bound = -ub

        # ---------------------------
        # COD → ACETATE
        # ---------------------------
        if not pd.isna(COD) and X_eff > 0:
            mmol_L = (COD * 1000 / COD_TO_ACETATE_RATIO) / ACETATE_MW
            q_ac = mmol_L / (X_eff * time_h)

            rxn = m.reactions.get_by_id("EX_ac_e")
            rxn.lower_bound = -q_ac
            rxn.upper_bound = -q_ac
            ex_lb = rxn.lower_bound
            ex_ub = rxn.upper_bound

        # ---------------------------
        # PHOTONS
        # ---------------------------
        scale_photons(m, illum)

        # ---------------------------
        # PHB OBJECTIVE
        # ---------------------------
        mu_max = 0.08
        f = 0.20
        v_phb_max = mu_max * f * 1000 / PHB_MW

        phb_rxn = m.reactions.get_by_id("PHBVS_syn")
        phb_rxn.lower_bound = 0
        phb_rxn.upper_bound = v_phb_max
        m.objective = phb_rxn

        m.reactions.get_by_id("ATPM").lower_bound = 20
        apply_solver_settings(m)

        # ---------------------------
        # FBA
        # ---------------------------
        fba_sol = m.optimize()

        # ---------------------------
        # TFA
        # ---------------------------
        mytfa = pytfa.ThermoModel(thermo_data, m)
        annotate_from_lexicon(mytfa, lexicon)
        apply_compartment_data(mytfa, comp_data)
        mytfa.objective = "PHBVS_syn"
        apply_solver_settings(mytfa)
        mytfa.prepare()
        tfa_sol = mytfa.optimize()

        # ---------------------------
        # COLLECT RESULTS
        # ---------------------------
        result = {
            "row": i + 1,
            "Time_d": time_d,
            "Time_h": time_h,
            "Biomass_final_gDW_L": biomass_final,
            "X_eff_gDW_L": X_eff,
            "COD_g_L": COD,
            "Illumination": illum,
            "q_ac_mmol_gDW_h": q_ac,
            "EX_ac_e_lb": ex_lb,
            "EX_ac_e_ub": ex_ub,
            "PHBV_FBA": fba_sol.fluxes.get("PHBVS_syn", np.nan),
            "PHBV_TFA": tfa_sol.fluxes.get("PHBVS_syn", np.nan),
            "Biomass_FBA": fba_sol.fluxes.get("BIOMASS__1", np.nan),
            "Biomass_TFA": tfa_sol.fluxes.get("BIOMASS__1", np.nan),
            "status_FBA": fba_sol.status,
            "status_TFA": tfa_sol.status,
        }

        for r in PHB_REACTIONS:
            result[f"{r}_FBA"] = fba_sol.fluxes.get(r, np.nan)
            result[f"{r}_TFA"] = tfa_sol.fluxes.get(r, np.nan)

    except Exception as e:
        print(f" Row {i+1} failed: {e}")
        result = {c: np.nan for c in ALL_COLUMNS}
        result["row"] = i + 1
        result["status_FBA"] = "error"
        result["status_TFA"] = "error"

    # ---------------------------
    # APPEND TO CSV
    # ---------------------------
    pd.DataFrame([result])[ALL_COLUMNS].to_csv(
        OUT_CSV,
        mode="a",
        header=False,
        index=False,
    )

    print(f"✓ Row {i+1} saved")

print("\nDONE — all rows processed safely.")


In [ ]:
import cobra
import pytfa
import pandas as pd
import numpy as np
from pathlib import Path
from pytfa.io import (
    read_lexicon,
    annotate_from_lexicon,
    read_compartment_data,
    apply_compartment_data
)

# ============================================================
# USER INPUT
# ============================================================
CSV_PATH = "/content/driveDL/My Drive/Genome_scale_metabolic_models_PNSB/Model_Tec_Campos_2023_RPiDT1294/rpalustris_pha_optimization/data/Buitron2025_dataset.csv"
OUTPUT_CSV = Path("FBA_TFA_PHB_complete_dataset.csv")

INITIAL_BIOMASS = 0.094   # gDW/L
FALLBACK_TIME_H = 72.0

UPTAKE_FACTORS = [1, 10, 100]

# ============================================================
# CONSTANTS
# ============================================================
PHB_MW = 86.0
ACETATE_MW = 59.04
COD_TO_ACETATE_RATIO = 1.066

PHB_REACTIONS = [
    "ACACT1r",
    "ACACCT",
    "AACOAR_syn",
    "HACD1_2",
    "HACD1",
    "HACD1i",
    "KAT1"
]

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv(CSV_PATH)
df = df.iloc[1:35].drop(index=[15, 16]).reset_index(drop=True)

# ============================================================
# MEDIUM
# ============================================================
medium_ub = {
    "EX_ac_e": 0.2477,
    "EX_nh4_e": 0.14188,
    "EX_pi_e": 0.03733,
    "EX_o2_e": 0,
    "EX_co2_e": 0,
    **{f"EX_photon{x}_e": 10 for x in range(410, 700, 20)}
}

# ============================================================
# HELPERS
# ============================================================
def apply_solver_settings(model):
    model.solver = "glpk"
    model.solver.configuration.tolerances.feasibility = 1e-9
    model.solver.configuration.presolve = True


def scale_photons(model, illum, ref=1000):
    for rxn in model.reactions:
        if rxn.id.startswith("EX_photon"):
            rxn.lower_bound = 0

    if pd.isna(illum):
        return

    s = str(illum).lower()
    if s in ["continuous", "light", "on"]:
        for rid, ub in medium_ub.items():
            if rid.startswith("EX_photon") and rid in model.reactions:
                model.reactions.get_by_id(rid).lower_bound = -ub
        return

    try:
        val = float("".join(c for c in s if c.isdigit() or c == "."))
        scale = val / ref
        for rid, ub in medium_ub.items():
            if rid.startswith("EX_photon") and rid in model.reactions:
                model.reactions.get_by_id(rid).lower_bound = -scale * ub
    except Exception:
        pass


# ============================================================
# LOAD TFA METADATA
# ============================================================
lexicon = read_lexicon(str(rp_lex_mat_path))
comp_data = read_compartment_data(str(rp_comp_mat_path))

# ============================================================
# RESUME LOGIC
# ============================================================
if OUTPUT_CSV.exists():
    existing = pd.read_csv(OUTPUT_CSV)
    done = set(zip(existing.row, existing.uptake_factor))
    results = existing.to_dict("records")
else:
    done = set()
    results = []

# ============================================================
# MAIN LOOP
# ============================================================
for i, row in df.iterrows():

    base_metadata = row.to_dict()  # ALL original CSV columns

    for factor in UPTAKE_FACTORS:

        if (i + 1, factor) in done:
            continue

        print(f"ROW {i+1} | uptake ×{factor}")

        m = cobra_model_mat.copy()

        for ex in m.exchanges:
            ex.lower_bound = 0
        for rid, ub in medium_ub.items():
            if rid in m.reactions:
                m.reactions.get_by_id(rid).lower_bound = -ub

        # Time & biomass
        t_h = row.get("Time (d)", np.nan)
        t_h = FALLBACK_TIME_H if pd.isna(t_h) else float(t_h) * 24

        biomass_final = row.get("Biomass (mg dw/L)", np.nan)
        biomass_final = biomass_final / 1000 if not pd.isna(biomass_final) else np.nan

        X_eff = (INITIAL_BIOMASS + biomass_final) / 2 if not pd.isna(biomass_final) else np.nan

        # Acetate uptake
        COD = row.get("g substrate COD/L", np.nan)
        q_ac = lb = ub = np.nan

        if not pd.isna(COD) and X_eff > 0:
            mmol_L = (COD * 1000 / COD_TO_ACETATE_RATIO) / ACETATE_MW
            q_ac = mmol_L / (X_eff * t_h) * factor
            rxn = m.reactions.get_by_id("EX_ac_e")
            rxn.lower_bound = -q_ac
            rxn.upper_bound = -q_ac
            lb, ub = rxn.lower_bound, rxn.upper_bound

        scale_photons(m, row.get("Illumination", np.nan))

        # PHB objective
        v_phb_max = 0.08 * 0.20 * 1000 / PHB_MW
        phb = m.reactions.get_by_id("PHBS_syn")
        phb.lower_bound = 0
        phb.upper_bound = 1000# v_phb_max ### IMPORTANT!
        m.objective = phb
        m.reactions.get_by_id("ATPM").lower_bound = 20

        apply_solver_settings(m)

        fba = m.optimize()

        # TFA
        mytfa = pytfa.ThermoModel(thermo_data, m)
        annotate_from_lexicon(mytfa, lexicon)
        apply_compartment_data(mytfa, comp_data)
        mytfa.objective = "PHBS_syn"
        apply_solver_settings(mytfa)
        mytfa.prepare()
        tfa = mytfa.optimize()

        record = {
            **base_metadata,
            "row": i + 1,
            "uptake_factor": factor,
            "Time_h": t_h,
            "Biomass_final_gDW_L": biomass_final,
            "X_eff_gDW_L": X_eff,
            "COD_g_L": COD,
            "q_ac_mmol_gDW_h": q_ac,
            "EX_ac_e_lb": lb,
            "EX_ac_e_ub": ub,
            "PHB_FBA": fba.fluxes.get("PHBS_syn", np.nan) if fba.status == "optimal" else np.nan,
            "Biomass_FBA": fba.fluxes.get("BIOMASS__1", np.nan) if fba.status == "optimal" else np.nan,
            "PHB_TFA": tfa.fluxes.get("PHBS_syn", np.nan) if tfa.status == "optimal" else np.nan,
            "Biomass_TFA": tfa.fluxes.get("BIOMASS__1", np.nan) if tfa.status == "optimal" else np.nan,
            "status_FBA": fba.status,
            "status_TFA": tfa.status,
        }

        for r in PHB_REACTIONS:
            record[f"{r}_FBA"] = fba.fluxes.get(r, np.nan) if fba.status == "optimal" else np.nan
            record[f"{r}_TFA"] = tfa.fluxes.get(r, np.nan) if tfa.status == "optimal" else np.nan

        results.append(record)
        pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)

print("DONE — complete dataset saved")


In [ ]:
too_wide = [
    rxn.id for rxn in cobra_model_mat.reactions
    if rxn.upper_bound > 1000 or rxn.lower_bound < -1000
]

print(f"Reactions still exceeding ±1000 bounds: {len(too_wide)}")


Reactions still exceeding ±1000 bounds: 423


# One datapoint

Build themo model rpalustris for one data point (e.g., the one resulting from Pareto optimality)

In [ ]:
from pytfa.io import import_matlab_model, load_thermoDB,                    \
                            read_lexicon, annotate_from_lexicon,            \
                            read_compartment_data, apply_compartment_data

# =====================================
# ADD THERMODYNAMIC COMPARTMENT DATA
# (REPLACES read_compartment_data)
# =====================================

case = 'full' # 'reduced' or full'

# Load reaction DB
print("Loading thermo data...")

#thermo_data = load_thermoDB('/Users/hector/git/pytfa/data/thermo_data.thermodb')

thermo_data['metabolites'] = {
    str(k): v
    for k, v in thermo_data['metabolites'].items()
}

print("Done !")

if case == 'reduced':
    cobra_model = cobra_model_mat
    mytfa = pytfa.ThermoModel(thermo_data, cobra_model)
    biomass_rxn = 'PHBVS_syn'
elif case == 'full':
    # We import pre-compiled data as it is faster for bigger models

    cobra_model = cobra_model_mat

    lexicon = read_lexicon(str(rp_lex_mat_path))
    #lexicon = pd.read_csv(rp_lex_mat_path, index_col=0)
    compartment_data = read_compartment_data(str(rp_comp_mat_path))

    # Initialize the cobra_model
    mytfa = pytfa.ThermoModel(thermo_data, cobra_model)
    # Annotate the cobra_model
    annotate_from_lexicon(mytfa, lexicon)
    apply_compartment_data(mytfa, compartment_data)

    rxn = mytfa.reactions.get_by_id("PHBVS_syn")

    rxn.lower_bound = 0.0      # e.g. no negative growth
    rxn.upper_bound = 1.0   # or any max growth rate you want

    biomass_rxn = 'PHBVS_syn'

mytfa.name = 'RP_iDT1294'
mytfa.solver = "glpk"
mytfa.objective = biomass_rxn

# Solver settings

def apply_solver_settings(model, solver = "glpk"):
    model.solver = "glpk"
    # model.solver.configuration.verbosity = 1
    model.solver.configuration.tolerances.feasibility = 1e-9
    if solver == 'optlang_gurobi':
        model.solver.problem.Params.NumericFocus = 3
    model.solver.configuration.presolve = True

apply_solver_settings(mytfa)


## FBA
cobra_model.objective = biomass_rxn
fba_solution = cobra_model.optimize()
fba_value = fba_solution.objective_value
#fva = flux_variability_analysis(mytfa)


## TFA conversion
mytfa.prepare()

mytfa.solver.update()
print(len(mytfa.constraints))
print(len(mytfa.variables))






In [ ]:
rxn = cobra_model.reactions.get_by_id('PHBVS_syn')
print("PHBS_syn bounds:", rxn.bounds)
print("PHBS_syn stoichiometry:", rxn.reaction)

NameError: name 'cobra_model' is not defined

# Save TFBA model

In [ ]:
from cobra.io import write_sbml_model

mytfa.convert() # convert

phbv_tfba_dir = project_root / "tfba"
phbv_tfba_dir.mkdir(parents=True, exist_ok=True)

cobra.io.write_sbml_model(
    mytfa,
    phbv_tfba_dir / "model_GEM_PHBV.xml"
)

print("Model saved successfully")


# Run TFBA

In [ ]:
# Optimize TFA
tfa_sol = mytfa.optimize()


DEBUG:thermomodel_None:'slim_optimize' ((), {}) 335.95 sec


# Compare FBA and TFBA solutions

In [ ]:

## Optimality
tfa_value = tfa_sol.objective_value

# It might happen that the model is infeasible. In this case, we can relax
# thermodynamics constraints:

#if tfa_value < 0.1:
#    from pytfa.optim.relaxation import relax_dgo

 #   mytfa.reactions.get_by_id(biomass_rxn).lower_bound = 0.5*fba_value
 #   relaxed_model, slack_model, relax_table = relax_dgo(mytfa)

 #   original_model, mytfa = mytfa, relaxed_model

  #  print('Relaxation: ')
  #  print(relax_table)

   # tfa_solution = mytfa.optimize()
  #  tfa_value = tfa_solution.objective_value

# Report
print('FBA Solution found : {0:.5g}'.format(fba_value))
print('TFA Solution found : {0:.5g}'.format(tfa_value))


solver_results = dict()

FBA Solution found : 1
TFA Solution found : 1


## Check value of PHB_reactions in FBA and TFBA

In [ ]:
PHB_REACTIONS = [
    "ACACT1r",
    "ACACCT",
    "AACOAR_syn",
    "HACD1_2",
    "HACD1",
    "HACD1i",
    "KAT1"
]

thermo_rxns = {r.id for r in mytfa.reactions if r.thermo is not None}

for r_id in PHB_REACTIONS:
    print(
        f"{r_id:12s}",
        "✅ thermo" if r_id in thermo_rxns else "❌ no thermo"
    )


ACACT1r      ✅ thermo
ACACCT       ✅ thermo
AACOAR_syn   ✅ thermo
HACD1_2      ✅ thermo
HACD1        ✅ thermo
HACD1i       ✅ thermo
KAT1         ✅ thermo


In [ ]:
for r_id in PHB_REACTIONS:

  rxn_id = r_id
  flux = fba_solution.fluxes[rxn_id]
  print(f"FBA Flux of {rxn_id}: {flux}")


FBA Flux of ACACT1r: 0.6000000000000003
FBA Flux of ACACCT: 0.0
FBA Flux of AACOAR_syn: 0.8000000000000004
FBA Flux of HACD1_2: 0.0
FBA Flux of HACD1: -0.20000000000000015
FBA Flux of HACD1i: 0.0
FBA Flux of KAT1: 0.0


In [ ]:
for r_id in PHB_REACTIONS:

  rxn_id = r_id
  flux = tfa_sol.fluxes[rxn_id]
  print(f"TFBA Flux of {rxn_id}: {flux}")

TFBA Flux of ACACT1r: 0.6000000000000001
TFBA Flux of ACACCT: 0.0
TFBA Flux of AACOAR_syn: 0.8
TFBA Flux of HACD1_2: 0.0
TFBA Flux of HACD1: 0.0
TFBA Flux of HACD1i: 0.2
TFBA Flux of KAT1: 0.0


In [ ]:
assert any(c.name.startswith("G_") for c in mytfa.constraints)
assert any(v.name.startswith("DGo_") for v in mytfa.variables)
assert any(v.type == "binary" for v in mytfa.variables)
print("✅ TFBA is active")

✅ TFBA is active


In [ ]:
for v in mytfa.variables.values():
  print(v)

  #0.0 <= DM_4CRSOL <= 1000.0


0.0 <= QULNS <= 1000.0
0.0 <= QULNS_reverse_66da1 <= 0.0
0.0 <= ORNDC <= 1000.0
0.0 <= ORNDC_reverse_63596 <= 0.0
0.0 <= MSBENZMT <= 1000.0
0.0 <= MSBENZMT_reverse_a902a <= 0.0
0.0 <= DESAT18a <= 1000.0
0.0 <= DESAT18a_reverse_fd859 <= 1000.0
0.0 <= FUM <= 1000.0
0.0 <= FUM_reverse_d3642 <= 1000.0
0.0 <= PHYFXOR <= 1000.0
0.0 <= PHYFXOR_reverse_84960 <= 0.0
0.0 <= VPAMTr <= 1000.0
0.0 <= VPAMTr_reverse_872bd <= 0.0
0.0 <= GLYCL <= 1000.0
0.0 <= GLYCL_reverse_e418f <= 0.0
0.0 <= NDPK7 <= 1000.0
0.0 <= NDPK7_reverse_9dc79 <= 1000.0
0.0 <= GTPCI <= 1000.0
0.0 <= GTPCI_reverse_1ee86 <= 0.0
0.0 <= MTHFC <= 1000.0
0.0 <= MTHFC_reverse_f6fcc <= 1000.0
0.0 <= GCATENEC <= 1000.0
0.0 <= GCATENEC_reverse_ae4a5 <= 1000.0
0.0 <= ORNTA <= 1000.0
0.0 <= ORNTA_reverse_5adff <= 1000.0
0.0 <= UPPDC1 <= 1000.0
0.0 <= UPPDC1_reverse_cb592 <= 0.0
0.0 <= GARFT <= 1000.0
0.0 <= GARFT_reverse_7ecb6 <= 0.0
0.0 <= H4THDPR <= 1000.0
0.0 <= H4THDPR_reverse_617be <= 0.0
0.0 <= UDPG4E <= 1000.0
0.0 <= UDPG4E_revers

In [ ]:
[v.name for v in mytfa.variables if v.name.startswith("LC_")]



['LC_gam6p_c',
 'LC_cgly_c',
 'LC_achms_c',
 'LC_pcox_u',
 'LC_octe9ACP_c',
 'LC_fdp_c',
 'LC_5caiz_c',
 'LC_pep_c',
 'LC_coa_c',
 'LC_hgbam_c',
 'LC_nac_c',
 'LC_r3mmal_c',
 'LC_leu__L_c',
 'LC_ahcys_c',
 'LC_ru5p__D_c',
 'LC_14dhncoa_c',
 'LC_h2o_cx_c',
 'LC_h2s_c',
 'LC_orot_c',
 'LC_g1p_c',
 'LC_3hdecACP_c',
 'LC_na1_c',
 'LC_2pglyc_c',
 'LC_ala_B_c',
 'LC_glyc3p_c',
 'LC_pqh2_um_p',
 'LC_26dap_LL_c',
 'LC_ddcaACP_c',
 'LC_2pglyc_cx_c',
 'LC_trdox_c',
 'LC_3oddecACP_c',
 'LC_glu__D_c',
 'LC_h_cx_c',
 'LC_dutp_c',
 'LC_paps_c',
 'LC_mg2_c',
 'LC_hdeACP_c',
 'LC_o2_c',
 'LC_anth_c',
 'LC_dhpt_c',
 'LC_thex2eACP_c',
 'LC_fmn_c',
 'LC_pppi_c',
 'LC_udp_c',
 'LC_cit_c',
 'LC_acACP_c',
 'LC_uama_c',
 'LC_eig3p_c',
 'LC_3c4mop_c',
 'LC_lys__L_c',
 'LC_13dpg_c',
 'LC_gmp_c',
 'LC_utp_c',
 'LC_trdrd_c',
 'LC_so4_c',
 'LC_5apru_c',
 'LC_hmppp9_c',
 'LC_pydx5p_c',
 'LC_2dhp_c',
 'LC_ade_c',
 'LC_asp__L_c',
 'LC_2sephchc_c',
 'LC_acg5p_c',
 'LC_gdp_c',
 'LC_hgbyr_c',
 'LC_palmACP_c',
 'LC_pser

# Sensitivity analysis

In [ ]:
for phbv_flux in [0.25, 0.5, 1.0, 1.5, 10, 100, 1000]:

    with model:
        # Fix PHBV synthesis flux
        rxn = cobra_model_mat.reactions.PHBVS_syn
        rxn.lower_bound = phbv_flux
        rxn.upper_bound = phbv_flux

        # ---------- FBA ----------
        fba_sol = cobra_model_mat.optimize()
        fba_value = fba_sol.objective_value

        # ---------- TFA ----------
        tfa_sol = mytfa.optimize()
        tfa_value = tfa_sol.objective_value

        print(f"PHBV flux = {phbv_flux:.2f}")
        print(f"  FBA solution: {fba_value:.5g}")
        print(f"  TFA solution: {tfa_value:.5g}")
        print(f"  TFA feasible: {tfa_sol.status == 'optimal'}")


# Find reactions for building lexicon for iDT1294

In [ ]:
# Add PHB_REACTIONS

# Acetyl-CoA C-acetyltransferase
#Acetyl-CoA:acetoacetyl-CoA transferase
#Acetoacetyl CoA reductase
#3 hydroxyacyl CoA dehydrogenase  acetoacetyl CoA
#3-ketoacyl-CoA thiolase

PHB_REACTIONS = [
    "ACACT1r",
    "ACACCT",
    "AACOAR_syn",
    "HACD1_2",
    "HACD1",
    "HACD1i",
    "KAT1"
]

phb_mets = set()

for rxn_id in PHB_REACTIONS:
    rxn = cobra_model_mat.reactions.get_by_id(rxn_id)
    for met in rxn.metabolites:
        phb_mets.add(met)

for met in sorted(phb_mets, key=lambda m: m.id):
    print(f"{met.id}\t{met.name}\t{met.formula}\t{met.compartment}")




3hbcoa__R_c	(R)-3-Hydroxybutyryl-CoA	C25H39N7O18P3S	c
3hbcoa_c	(S)-3-Hydroxybutanoyl-CoA	C25H38N7O18P3S	c
3hbycoa_c	S  3 Hydroxybutyryl CoA C25H38N7O18P3S	C25H38N7O18P3S	c
aacoa_c	Acetoacetyl-CoA	C25H37N7O18P3S	c
ac_c	Acetate	C2H3O2	c
acac_c	Acetoacetate	C4H5O3	c
accoa_c	Acetyl-CoA	C23H35N7O17P3S	c
coa_c	Coenzyme A	C21H33N7O16P3S	c
h_c	H+	H	c
nad_c	Nicotinamide adenine dinucleotide	C21H26N7O14P2	c
nadh_c	Nicotinamide adenine dinucleotide - reduced	C21H27N7O14P2	c
nadp_c	Nicotinamide adenine dinucleotide phosphate	C21H26N7O17P3	c
nadph_c	Nicotinamide adenine dinucleotide phosphate - reduced	C21H27N7O17P3	c


In [ ]:
import pandas as pd
from pytfa.io import annotate_from_lexicon

lexicon = pd.read_csv("phb_lexicon.csv")

annotate_from_lexicon(
    mytfa,
    lexicon,
    inplace=True
)


## Check TFA R. palustris

In [ ]:
[c.name for c in mytfa.constraints if c.name.startswith(("FU_", "BU_", "SU_", "UF_"))][:10]


['SU_QULNS',
 'UF_QULNS',
 'SU_ORNDC',
 'UF_ORNDC',
 'SU_MSBENZMT',
 'UF_MSBENZMT',
 'SU_DESAT18a',
 'UF_DESAT18a',
 'SU_FUM',
 'UF_FUM']

In [ ]:
mytfa.description = "R. palustris iDT1294 TFA model"
mytfa.print_info()


                                          value
key                                            
name                            tutorial_basics
description      R. palustris iDT1294 TFA model
num constraints                           10287
num variables                             13008
num metabolites                            2124
num reactions                              2721
                            value
key                              
num metabolites(thermo)  8.000000
num reactions(thermo)    0.000000
pct metabolites(thermo)  0.376648
pct reactions(thermo)    0.000000


In [ ]:
for v in mytfa.variables.values():
  print(v)

In [ ]:
for c in mytfa.constraints.values():
    if rxn.id in c.name:
        print(c)


In [ ]:
len([v.name for v in mytfa.variables if v.name.startswith("DGo_")])


0

In [ ]:
[c.name for c in mytfa.constraints if c.name.startswith("G_")][:10]


[]

In [ ]:
fba_fluxes = cobra_model.optimize().fluxes
tfa_fluxes = mytfa.optimize().fluxes

diff = (fba_fluxes - tfa_fluxes).abs()
print("Max flux difference:", diff.max())


In [ ]:
print(type(thermo_data["metabolites"]))
print("ThermoDB metabolites:", len(thermo_data["metabolites"]))
print(list(thermo_data["metabolites"].keys())[:10])


<class 'dict'>
ThermoDB metabolites: 18307
['cpd00805', 'cpd11364', 'cpd18424', 'cpd17488', 'cpd03902', 'cpd14767', 'cpd05076', 'cpd00475', 'cpd17550', 'cpd08432']


In [ ]:
# Convert DataFrame → dict lexicon
lexicon_dict = (
    lexicon['seed_id']
    .dropna()
    .astype(str)
    .to_dict()
)

print(type(lexicon_dict))
print(len(lexicon_dict))
print(list(lexicon_dict.items())[:5])


<class 'dict'>
8
[('glc__D_c', 'cpd00027'), ('pyr_c', 'cpd00020'), ('accoa_c', 'cpd00022'), ('atp_c', 'cpd00002'), ('adp_c', 'cpd00008')]


In [ ]:
import pandas as pd

rows = []
for m in cobra_model_mat.metabolites:
    rows.append({
        "met_id": m.id,
        "name": m.name,
        "formula": m.formula,
        "charge": m.charge,
        "compartment": m.compartment
    })

df = pd.DataFrame(rows)
df.to_csv("metabolites_for_mapping.csv", index=False)


In [ ]:
from collections import Counter

print("Metabolites:", len(model.metabolites))
print("With formula:", sum(1 for m in model.metabolites if m.formula))
print("With annotation:", sum(1 for m in model.metabolites if m.annotation))


Metabolites: 2124
With formula: 2118
With annotation: 0


In [ ]:
# Load COBRA model

# Path to base model
base_model_dir = project_root / "base"
base_model_dir.mkdir(parents=True, exist_ok=True)

# Full path to the SBML file
base_model_path = base_model_dir / "model_rpalustris_comp_fixed.xml"

# Load model
model = read_sbml_model(
    str(base_model_path),
    use_fbc=False
)

print(f"Reactions: {len(model.reactions)}")
print(f"Metabolites: {len(model.metabolites)}")


Reactions: 2721
Metabolites: 2124


# Load Buitrons medium constraints

In [ ]:
# Apply medium constraints from Montiel-Corona 2022.
def apply_media_from_excel(model, excel_path, close_exchanges=True):
    """
    Apply media constraints from an Excel file with columns:
    Reaction | LowerBound | UpperBound
    """

    media_df = pd.read_csv(excel_path)

    required_cols = {"Reaction", "LowerBound", "UpperBound"}
    if not required_cols.issubset(media_df.columns):
        raise ValueError(
            f"Excel file must contain columns: {required_cols}"
        )

    # Optionally close all exchanges first
    if close_exchanges:
        for rxn in model.exchanges:
            rxn.lower_bound = 0
            rxn.upper_bound = 1000

    # Apply media bounds
    for _, row in media_df.iterrows():
        rxn_id = row["Reaction"]

        if rxn_id not in model.reactions:
            raise KeyError(f"Reaction {rxn_id} not found in model")

        rxn = model.reactions.get_by_id(rxn_id)
        rxn.lower_bound = float(row["LowerBound"])
        rxn.upper_bound = float(row["UpperBound"])

    # Check reaction directly associated to PHB production
    EX_acetate_rxn = model.reactions.get_by_id("EX_ac_e")
    print(EX_acetate_rxn.bounds)
    print(EX_acetate_rxn.reaction)

    return model


In [ ]:
# Load model

excel_path =  project_root / "phbv" / "checkpoint_output" / "PHBV_model_medium_montiel_buitron_2022_exchange_bounds.xlsx"
print(excel_path)

/content/driveDL/My Drive/Genome_scale_metabolic_models_PNSB/Model_Tec_Campos_2023_RPiDT1294/rpalustris_pha_optimization/models/phbv/checkpoint_output/PHBV_model_medium_montiel_buitron_2022_exchange_bounds.xlsx


In [ ]:

apply_media_from_excel(model, excel_path)

(-0.2477, 1000.0)
ac[e] <=> 


Name,M_iDT1294
Memory address,799a68b4e810
Number of metabolites,2124
Number of reactions,2721
Number of genes,1294
Number of groups,117
Objective expression,1.0*BIOMASS__1 - 1.0*BIOMASS__1_reverse_063c7
Compartments,"cytoplasm, unknownCompartment4, periplasm, exterior"


# Define PHBS_syn as objective

In [ ]:
# Get the reaction
rxn = model.reactions.get_by_id('PHBS_syn')
# Set it as objective
model.objective = rxn
# Set lower and upper bounds
rxn.lower_bound = 0.02
rxn.upper_bound = 0.18

model.summary() #  is the default objective function

Metabolite,Reaction,Flux,C-Number,C-Flux
ac[e],EX_ac_e,0.2477,2,100.00%
photon690[e],EX_photon690_e,0.626,0,0.00%
phbg[c],SK_phbg_c,0.1237,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
PHB[c],DM_PHB_c,-0.1237,4,99.92%
h2o[e],EX_h2o_e,-5E-05,0,0.00%
o2[e],EX_o2_e,-0.1238,0,0.00%
2obut[c],sink_2obut_c,-0.0001,4,0.08%


# Assing deltaG0_prime to important reactions

In [ ]:
tmodel.reactions.get_by_id("AACOAR_syn")

Reaction identifier,AACOAR_syn
Name,Acetoacetyl CoA reductase
Memory address,0x799a736e3260
Stoichiometry,aacoa[c] + h[c] + nadph[c] <=> 3hbcoa__R[c] + nadp[c] Acetoacetyl-CoA + H+ + Nicotinamide adenine dinucleotide phosphate - reduced <=> _R-3-Hydroxybutyryl-CoA + Nicotinamide adenine dinucleotide phosphate
GPR,RPE_RS00840
Lower bound,-1000
Upper bound,1000


In [ ]:
from pytfa.optim.variables import DeltaG

PHB_REACTION_THERMO = {
    "ACACT1r": {      # 2 acetyl-CoA ⇌ acetoacetyl-CoA + CoA
        "delta_g0": +18.0,
        "uncertainty": 5.0
    },

    "AACOAR_syn": {   # acetoacetyl-CoA + NADPH → 3HB-CoA + NADP+
        "delta_g0": -25.0,
        "uncertainty": 5.0
    },

    "PHBS_syn": {     # polymerization (CoA-releasing)
        "delta_g0": -35.0,
        "uncertainty": 7.0
    },

    "ACACCT": {       # CoA transfer / activation
        "delta_g0": -10.0,
        "uncertainty": 5.0
    }
}


for rxn_id in PHB_REACTION_THERMO:
    rxn = tmodel.reactions.get_by_id(rxn_id)
    dg_vars = [v for v in tmodel.variables.values() if isinstance(v, DeltaG) and v.reaction == rxn]
    print(rxn_id, "ΔG vars:", dg_vars)

ACACT1r ΔG vars: []
AACOAR_syn ΔG vars: []
PHBS_syn ΔG vars: []
ACACCT ΔG vars: []


In [ ]:
thermo_data2['cues'].update({
    "AACOAR_syn": {"delta_g0": -25.0, "uncertainty": 5.0},
    "PHBS_syn":    {"delta_g0": -20.0, "uncertainty": 5.0},
    "ACACT1r":     {"delta_g0": -10.0, "uncertainty": 5.0},
    "ACACCT":      {"delta_g0": -15.0, "uncertainty": 5.0},
})


In [ ]:
# -----------------------------
# Minimal PHB-focused TFA setup
# -----------------------------

import cobra
from pytfa import ThermoModel
from pytfa.io import load_thermoDB

# -----------------------------
# 1️⃣ Load thermodynamic database (optional, here we skip it for manual ΔG°)
# -----------------------------
# thermo_data = load_thermoDB()  # only needed if you want full database

# -----------------------------
# 2️⃣ Define your minimal thermo_data
# -----------------------------
# Include only metabolites and reactions relevant for PHB production
# Make sure metabolite formulas are correct (already done)

thermo_data = {
    "name": "PHB_minimal",
    "units": {"energy": "kJ/mol"},
    "metabolites": {
        m.id: {"formula": m.formula}
        for m in model.metabolites
        if m.formula is not None and m.id.endswith("[c]")
    },
    "cues": {
        "AACOAR_syn": {"deltaG0_prime": -25.0, "deltaG0_uncertainty": 5.0},
        "ACACT1r":    {"deltaG0_prime": -10.0, "deltaG0_uncertainty": 5.0},
    }
}

tmodel = ThermoModel(thermo_data, model)

tmodel.compartments["c"] = {"pH": 7.0, "ionicStr": 0.25}



# Adjust constraints since some reactions have values ><1000
for rxn in tmodel.reactions:
    if rxn.lower_bound < -1000:
        rxn.lower_bound = -1000
    if rxn.upper_bound > 1000:
        rxn.upper_bound = 1000

# -----------------------------
# 5️⃣ Add concentration bounds for key metabolites
# -----------------------------
bounds = {
    "nadph[c]": (1e-6, 1e-3),
    "nadp[c]":  (1e-6, 1e-3),
    "h[c]":     (1e-8, 1e-6),
    "coa[c]":   (1e-6, 1e-3),
    "aacoa[c]": (1e-6, 1e-3),
    "3hbcoa__R[c]": (1e-6, 1e-3)
}

for met_id, (lb, ub) in bounds.items():
    met = tmodel.metabolites.get_by_id(met_id)
    met.concentration = (lb, ub)


# -----------------------------
# 6️⃣ Prepare and convert TFA model
# -----------------------------
tmodel.prepare()  # prepare metabolite concentrations
tmodel.convert()  # add ΔG variables & thermodynamic constraints



In [ ]:
from pytfa.optim.variables import DeltaG

rxn = tmodel.reactions.get_by_id("AACOAR_syn")
dg_vars = [
    v for v in tmodel.variables.values()
    if isinstance(v, DeltaG) and v.reaction == rxn
]

print(dg_vars)


[]


In [ ]:
from pytfa.optim.variables import DeltaG

for rxn_id in ["AACOAR_syn", "ACACT1r"]:
    rxn = tmodel.reactions.get_by_id(rxn_id)
    dg_vars = [
        v for v in tmodel.variables.values()
        if isinstance(v, DeltaG) and v.reaction == rxn
    ]
    print(rxn_id, "ΔG vars:", dg_vars)


AACOAR_syn ΔG vars: []
ACACT1r ΔG vars: []


In [ ]:
rxn = tmodel.reactions.get_by_id("AACOAR_syn")

print("Stoichiometry:", rxn.metabolites)
print("Is transport:", rxn.id.endswith("t") or rxn.boundary)
print("All formulas present:",
      all(m.formula is not None for m in rxn.metabolites))
print("Compartments:",
      {m.compartment for m in rxn.metabolites})


Stoichiometry: {<Metabolite h[c] at 0x799a5e8bd580>: -1.0, <Metabolite nadph[c] at 0x799a5e8bd0d0>: -1.0, <Metabolite aacoa[c] at 0x799a5ea840e0>: -1.0, <Metabolite nadp[c] at 0x799a5ea98a40>: 1.0, <Metabolite 3hbcoa__R[c] at 0x799a5ea86900>: 1.0}
Is transport: False
All formulas present: True
Compartments: {'c'}


In [ ]:
from pytfa.optim.variables import DeltaG

rxn = tmodel.reactions.get_by_id("AACOAR_syn")
dg_vars = [
    v for v in tmodel.variables.values()
    if isinstance(v, DeltaG) and v.reaction == rxn
]

print(dg_vars)


[]


In [ ]:
phb_mets = [
    "aacoa[c]",
    "accoa[c]",
    "coa[c]",
    "nadph[c]",
    "nadp[c]",
    "3hbcoa__R[c]",
    "acac[c]",
    "ac[c]",
    "h[c]"
]

print("\n=== PHB metabolites: seed_id check ===")
for mid in phb_mets:
    met = tmodel.metabolites.get_by_id(mid)
    sid = met.annotation.get("seed_id", None)
    print(f"{mid:15s} | seed_id: {sid}")



=== PHB metabolites: seed_id check ===
aacoa[c]        | seed_id: None
accoa[c]        | seed_id: None
coa[c]          | seed_id: None
nadph[c]        | seed_id: None
nadp[c]         | seed_id: None
3hbcoa__R[c]    | seed_id: None
acac[c]         | seed_id: None
ac[c]           | seed_id: None
h[c]            | seed_id: None


In [ ]:
# Reactions of interest
phb_reactions = ["AACOAR_syn", "ACACT1r"]

thermo_mets = thermo_data["metabolites"]  # ThermoDB compounds dict

def normalize(s):
    return s.lower().replace("-", "").replace("_", "").replace(" ", "")

print("\n=== PHB Reactions: On-the-fly ThermoDB mapping ===")

for rxn_id in phb_reactions:
    if rxn_id not in model.reactions:
        print(f"\n❌ Reaction {rxn_id} not in model")
        continue

    rxn = model.reactions.get_by_id(rxn_id)
    print(f"\nReaction: {rxn_id}")
    print(rxn.build_reaction_string())
    print("-" * 70)

    for met in rxn.metabolites:
        print(f"\nMetabolite: {met.id}")
        print(f"  Name:        {met.name}")
        print(f"  Formula:     {met.formula}")
        print(f"  Compartment: {met.compartment}")

        formula_hits = []
        name_hits = []

        for seed_id, tmeta in thermo_mets.items():
            # Formula match
            if met.formula and tmeta.get("formula") == met.formula:
                formula_hits.append((seed_id, tmeta.get("name")))

            # Name match (loose)
            if met.name and normalize(met.name) in normalize(tmeta.get("name", "")):
                name_hits.append((seed_id, tmeta.get("name")))

        if formula_hits:
            print("  🔬 ThermoDB formula matches:")
            for sid, name in formula_hits[:5]:
                print(f"     - seed_id: {sid} | {name}")
        else:
            print("  ❌ No ThermoDB formula matches")

        if name_hits:
            print("  🧬 ThermoDB name matches:")
            for sid, name in name_hits[:5]:
                print(f"     - seed_id: {sid} | {name}")
        else:
            print("  ❌ No ThermoDB name matches")



=== PHB Reactions: On-the-fly ThermoDB mapping ===

Reaction: AACOAR_syn
aacoa[c] + h[c] + nadph[c] --> 3hbcoa__R[c] + nadp[c]
----------------------------------------------------------------------

Metabolite: h[c]
  Name:        H+
  Formula:     H
  Compartment: c
  🔬 ThermoDB formula matches:
     - seed_id: h_cx[c] | None
     - seed_id: h[c] | None
  ❌ No ThermoDB name matches

Metabolite: nadph[c]
  Name:        Nicotinamide adenine dinucleotide phosphate - reduced
  Formula:     C21H27N7O17P3
  Compartment: c
  🔬 ThermoDB formula matches:
     - seed_id: nadph[c] | None
  ❌ No ThermoDB name matches

Metabolite: aacoa[c]
  Name:        Acetoacetyl-CoA
  Formula:     C25H37N7O18P3S
  Compartment: c
  🔬 ThermoDB formula matches:
     - seed_id: aacoa[c] | None
  ❌ No ThermoDB name matches

Metabolite: nadp[c]
  Name:        Nicotinamide adenine dinucleotide phosphate
  Formula:     C21H26N7O17P3
  Compartment: c
  🔬 ThermoDB formula matches:
     - seed_id: nadp[c] | None
  ❌ No 

In [ ]:
# List of reactions of interest
phb_reactions = ["AACOAR_syn", "ACACT1r"] #"PHBS_syn","ACACCT"

print("=== PHB Reactions Metabolites ===")
for rxn_id in phb_reactions:
    if rxn_id not in model.reactions:
        print(f"Reaction {rxn_id} not in model")
        continue

    rxn = model.reactions.get_by_id(rxn_id)
    print(f"\nReaction: {rxn_id} | Equation: {rxn.build_reaction_string()}")
    print(f"{'Metabolite ID':<20} | {'Formula':<15} | {'Compartment'}")
    print("-"*55)

    for met, coeff in rxn.metabolites.items():
        # check if formula exists
        formula = met.formula if met.formula is not None else "None"
        print(f"{met.id:<20} | {formula:<15} | {met.compartment}")


=== PHB Reactions Metabolites ===

Reaction: AACOAR_syn | Equation: aacoa[c] + h[c] + nadph[c] --> 3hbcoa__R[c] + nadp[c]
Metabolite ID        | Formula         | Compartment
-------------------------------------------------------
h[c]                 | H               | c
nadph[c]             | C21H27N7O17P3   | c
aacoa[c]             | C25H37N7O18P3S  | c
nadp[c]              | C21H26N7O17P3   | c
3hbcoa__R[c]         | C25H39N7O18P3S  | c

Reaction: ACACT1r | Equation: 2.0 accoa[c] --> aacoa[c] + coa[c]
Metabolite ID        | Formula         | Compartment
-------------------------------------------------------
accoa[c]             | C23H35N7O17P3S  | c
coa[c]               | C21H33N7O16P3S  | c
aacoa[c]             | C25H37N7O18P3S  | c


In [ ]:
tmodel.compartments

{'c': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02},
 'u': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02},
 'p': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02},
 'e': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}}

In [ ]:
# Print all compartments defined in the GEM
print("=== Model Compartments ===")
for comp_id, comp in model.compartments.items():
    print(f"{comp_id}: {comp}")

print("\n=== Metabolites and their compartments ===")
for met in model.metabolites:
    # ID and compartment
    print(f"{met.id:20} | {met.compartment:5} | formula: {met.formula}")


In [ ]:
## Optimality
tfa_solution = mytfa.optimize()

In [ ]:
from pytfa.optim.variables import DeltaG

for rxn_id in PHB_REACTION_THERMO:
    rxn = mytfa.reactions.get_by_id(rxn_id)

    dg_vars = [
        v for v in mytfa.variables.values()
        if isinstance(v, DeltaG) and v.reaction == rxn
    ]

    print(rxn_id, "ΔG vars:", dg_vars)


ACACT1r ΔG vars: []
AACOAR_syn ΔG vars: []
PHBS_syn ΔG vars: []
ACACCT ΔG vars: []


# Check up thermodata

In [ ]:
thermo_data.keys()


dict_keys(['name', 'units', 'metabolites', 'cues'])

In [ ]:
# CHECK THERMODATA STRUCTURE
#thermo_data['metabolites'].items()

In [ ]:
# CHECK METABOLITES FORMULAS (OR NAME)
names = [entry['formula'] for entry in thermo_data['metabolites'].values()]
thermo_formulas = [str(f) for f in names if f is not None]
print(thermo_formulas)

['C6H12O9P', 'C37H56N8O9', 'CH3Br', 'C19H15N3O7', 'C21H26N2O6Cl2', 'NA', 'C18H16N6O8S3', 'C5H10O8P', 'C30H50O4', 'C22H29N7O5', 'C15H28N2', 'C27H46O', 'C22H16N4O', 'C7H17N4O', 'C36H61N7O17P3S', 'C19H21N7O6', 'C30H51N7O7', 'C23H31NO7', 'C9H14N4O4', 'C10H14CaO6', 'C15H26O2', 'C3H4N2', 'H2Cu2O', 'C6H7O3RS', 'C8H11NO', 'C14H14O4', 'C16H20N2O2', 'C20H28O', 'C28H50O2', 'C16H17NO2', 'C40H60', 'C7H9N2', 'C11H18N2O2S', 'C20H30N6O12S2', 'C16H21N4O9', 'C15H16O2', 'C60H78OSn2', 'C12H19N2O', 'C9H11O11PR3', 'C18H26O4', 'C8H16N2O4Se2', 'C13H20O', 'C9H16N4OS', 'C6H14O6', 'C7H10N2O3', 'C10H12O2', 'C20H32', 'NA', 'NA', 'C27H34O5', 'C9H9O5', 'C15H20O3', 'C35H32N4O5Mg', 'C35H44O16', 'C12H6O4', 'CNR', 'C20H24O6', 'C8H18O4S4', 'C8H10O', 'NA', 'C23H18O8', 'C11H16O16P2R2', 'C22H24N3OCl', 'C4H8O2', 'C19H28O3', 'C40H71N3O15P2', 'C8H8NO4', 'C25H39N7O18P3S', 'C23H26O8', 'C30H50O', 'C16H16O11', 'C17H22N3', 'C11H17O2', 'NA', 'C19H11O2', 'C18H26NO6', 'C24H47NO10S', 'C13H16N2O3', 'C19H25N2O', 'C20H25N2OS', 'C34H63N2O6

In [ ]:
# Look for metabolites with specific composition

target_elements = ["C21", "N7", "P2"]

for entry in thermo_formulas:
    if all(elem in entry for elem in target_elements):
        print(entry)


C21H27N7O14P2
C21H44N7O18P2
C21H29N7O15P2
C21H43N7O18P2
C21H26N7O14P2
C21H33N7O13P2S
C21H44N7O17P2
C21H44N7O17P2
C21H44N7O18P2


In [ ]:
# LOOCKUP INFORMATION OF SPECIFIC METABOLITES USING THEIR FORMULA

metabolites_to_check = [
    "C21H27N7O14P2",
    "C21H29N7O15P2",
    "C21H26N7O14P2",
]

for met in metabolites_to_check:
    found = False
    for met_id, entry in thermo_data['metabolites'].items():
        name_lower = entry.get('formula', '').lower()
        # Exact match
        if met.lower() == name_lower:
            print(f"Metabolite ID: {met_id}")
            for key, value in entry.items():
                print(f"  {key}: {value}")
            print("-"*50)
            found = True
            break  # stop after the first exact match
    if not found:
        print(f"{met} ❌ NOT found in thermo_data")
        print("-"*50)

Metabolite ID: cpd00004
  pKa: [1.8, 2.56, 12.69, 13.31, 14.3]
  deltaGf_err: 4.2679
  mass_std: 663.0
  struct_cues: {'RWWNW': 2, 'mid_phos': 1, 'WNH2': 2, 'amide': 1, 'RWCHWW': 8, 'RWOW': 2, 'RWCHdblW': 5, 'WPO4nW': 1, 'TWWCdblW': 2, 'Origin': 1, 'WketoneW': 1, 'WCH2W': 2, 'RWdblNW': 3, 'RWCH2W': 1, 'OCCC': 1, 'HeteroAromatic': 2, 'PrimOH': 4, 'RWCdblWW': 2}
  id: cpd00004
  nH_std: 27
  name: NADH
  formula: C21H27N7O14P2
  deltaGf_std: -524.32
  error: Nil
  charge_std: -2
  other_names: ['NADH', 'DPNH', 'Nicotinamide adenine dinucleotide - reduced', 'Nicotinamideadeninedinucleotide-reduced', 'nadh']
--------------------------------------------------
Metabolite ID: cpd02951
  pKa: [1.8, 2.56, 12.69, 13.31, 14.3]
  deltaGf_err: 4.238
  mass_std: 681.0
  struct_cues: {'RWWNW': 2, 'mid_phos': 1, 'WNH2': 2, 'amide': 1, 'RWCHWW': 9, 'RWOW': 2, 'RWCHdblW': 3, 'WPO4nW': 1, 'TWWCdblW': 2, 'Origin': 1, 'WketoneW': 1, 'WCH2W': 2, 'RWdblNW': 3, 'RWCH2W': 2, 'OCCC': 1, 'HeteroAromatic': 2, 'Pr

# Load thermodynamics database

In [ ]:
# Download thermo database
!wget https://raw.githubusercontent.com/EPFL-LCSB/pytfa/master/data/thermo_data.thermodb -O thermo_data.thermodb

# Load thermodynamics
thermo_data = load_thermoDB("thermo_data.thermodb")

# Print thermo_data keys

print(f"Thermo Data loaded successfully:{thermo_data.keys()}")

--2026-01-04 16:01:23--  https://raw.githubusercontent.com/EPFL-LCSB/pytfa/master/data/thermo_data.thermodb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2184378 (2.1M) [application/octet-stream]
Saving to: ‘thermo_data.thermodb’

thermo_data.thermod 100%[===================>]   2.08M  --.-KB/s    in 0.07s   

2026-01-04 16:01:24 (31.0 MB/s) - ‘thermo_data.thermodb’ saved [2184378/2184378]

Thermo Data loaded successfully:dict_keys(['name', 'units', 'metabolites', 'cues'])


In [ ]:
type(thermo_data)

dict

In [ ]:
thermo_data.keys()

dict_keys(['name', 'units', 'metabolites', 'cues'])

In [ ]:
# Adjust model compartments as required by TFA

model.compartments = {
    'c': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    },
    'e': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    },

    'u': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    },
    'p': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    }
}

# Step 1: map metabolites to standard compartments
for met in model.metabolites:
    if met.compartment in ["c", "cytosol"]:
        met.compartment = "c"
    elif met.compartment in ["e", "extracellular"]:
        met.compartment = "e"

# Step 2: collect all compartments used by metabolites
used_compartments = set([met.compartment for met in model.metabolites])

# Step 3: ensure all compartments exist and are dicts
for c in used_compartments:
    val = model.compartments.get(c)
    # overwrite any string or missing compartment with dict
    model.compartments[c] = {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02
    }

print("Final compartments:", model.compartments)


Final compartments: {'c': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}, 'u': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}, 'p': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}, 'e': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}}


In [ ]:
rxn = model.reactions.get_by_id('AACOAR_syn')
rxn.lower_bound = -1000
rxn.upper_bound = 1000

for rxn in model.reactions:
    rxn.lower_bound = max(rxn.lower_bound, -1000)
    rxn.upper_bound = min(rxn.upper_bound, 1000)


tmodel = ThermoModel(
    thermo_data,
    model
)

for comp in tmodel.compartments:
    tmodel.compartments[comp] = {
        "pH": 7.0,
        "ionicStr": 0.25
    }

tmodel.solver = "glpk" #'optlang-cplex'


# Set physiological concentration ranges
tmodel.metabolites.get_by_id("nadph[c]").concentration = (1e-6, 1e-3)
tmodel.metabolites.get_by_id("nadp[c]").concentration  = (1e-6, 1e-3)
tmodel.metabolites.get_by_id("h[c]").concentration     = (1e-8, 1e-6)
tmodel.metabolites.get_by_id("coa[c]").concentration   = (1e-6, 1e-3)
#for met in tmodel.metabolites:
#    met.concentration = (1e-9, 1e-2)


tmodel.prepare()
tmodel.convert()



In [ ]:
# TFA OPTIMIZATION

sol = tmodel.optimize()



DEBUG:thermomodel_Model Exported from COBRA Toolbox:'slim_optimize' ((), {}) 441.55 sec


In [ ]:
print(sol.status)
print(sol.fluxes['AACOAR_syn'])

optimal
0.12374999999995748


In [ ]:
# FBA OPTIMIZATION

sol2 = model.optimize()
print(sol2.status)
print(sol2.fluxes['PHBS_syn'])


optimal
0.12375000000000087


In [ ]:
# Explain why thi is necessary??
rxn = tmodel.reactions.get_by_id('AACOAR_syn')
rxn.thermo['deltaG0_prime'] = (-25.0, 2.0)
rxn.thermo

{'isTrans': False,
 'computed': False,
 'deltaGR': 1000.0,
 'deltaGRerr': 1000.0,
 'deltaG0_prime': (-25.0, 2.0)}

In [ ]:
rxn = tmodel.reactions.get_by_id('AACOAR_syn')
rxn.thermo

{'isTrans': False,
 'computed': False,
 'deltaGR': 1000.0,
 'deltaGRerr': 1000.0,
 'deltaG0_prime': (-25.0, 2.0)}

# Formula normalization and matching

In [ ]:
gem_formulas = {
    m.id: m.formula
    for m in model.metabolites
    if m.formula is not None
}
gem_formulas

In [ ]:
# Normalize Formula using Hill notation
import re
from collections import defaultdict

def normalize_formula(formula):
    """
    Normalize chemical formula to Hill-like notation.
    Example: 'H12C6O6P1' -> 'C6H12O6P'
    """
    if formula is None:
        return None

    # Parse elements and counts
    tokens = re.findall(r'([A-Z][a-z]?)(\d*)', formula)
    if not tokens:
        return None

    elements = defaultdict(int)
    for el, count in tokens:
        elements[el] += int(count) if count else 1

    # Hill system: C, then H, then others alphabetically
    ordered = []
    if 'C' in elements:
        ordered.append(('C', elements.pop('C')))
    if 'H' in elements:
        ordered.append(('H', elements.pop('H')))
    for el in sorted(elements):
        ordered.append((el, elements[el]))

    return ''.join(f"{el}{cnt if cnt > 1 else ''}" for el, cnt in ordered)


In [ ]:
# Find
thermo_norm = set(
    normalize_formula(f) for f in thermo_formulas
    if normalize_formula(f) is not None
)

gem_norm = {
    mid: normalize_formula(f)
    for mid, f in gem_formulas.items()
    if normalize_formula(f) is not None
}


In [ ]:
matched = {
    mid: f for mid, f in gem_norm.items()
    if f in thermo_norm
}


In [ ]:
len(matched)

1412

In [ ]:
unmatched = {
    mid: f for mid, f in gem_norm.items()
    if f not in thermo_norm
}


In [ ]:
len(unmatched)

706

In [ ]:
for mid, f in list(matched.items())[:20]:
    print(mid, f)


gam6p[c] C6H13NO8P
cgly[c] C5H10N2O3S
achms[c] C6H11NO4
pcox_u R
octe9ACP[c] C18H33ORS
pep[c] C3H2O6P
hgbam[c] C45H58N6O12
nac[c] C6H4NO2
r3mmal[c] C5H6O5
leu__L[c] C6H13NO2
ahcys[c] C14H20N6O5S
h2o_cx[c] H2O
orot[c] C5H3N2O4
na1[c] Na
ala_B[c] C3H7NO2
pqh2_um[p] C53H82O2
26dap_LL[c] C7H14N2O4
ddcaACP[c] C12H23ORS
trdox[c] X
glu__D[c] C5H8NO4


In [ ]:
# CHANGE FORMULA MANUALLY

met = model.metabolites.get_by_id('coa[c]')
met.formula = 'C21H36N7O16P3S'
# keep the charge as is


# Match metabolites by name

In [ ]:
'Acetyl-CoA' in thermo_formulas

True

In [ ]:
metabolites_to_check = [
    "nadph",
    "nadh",
    "acetyl-coa",
    "nad+",
    "nadp+",
    "3-hydroxybutyryl-coa",
    "acetoacetyl-coa",
    "coa"
]

for met in metabolites_to_check:
    found = False
    for met_id, entry in thermo_data['metabolites'].items():
        name_lower = entry.get('name', '').lower()
        # Exact match
        if met.lower() == name_lower:
            print(f"Metabolite ID: {met_id}")
            for key, value in entry.items():
                print(f"  {key}: {value}")
            print("-"*50)
            found = True
            break  # stop after the first exact match
    if not found:
        print(f"{met} ❌ NOT found in thermo_data")
        print("-"*50)

Metabolite ID: cpd00005
  pKa: []
  deltaGf_err: 4.2579
  mass_std: 742.0
  struct_cues: {'RWWNW': 2, 'mid_phos': 1, 'prim_phos': 1, 'amide': 1, 'RWCHWW': 8, 'TWWCdblW': 2, 'RWCHdblW': 5, 'WPO4nW': 1, 'WNH2': 2, 'RWOW': 2, 'Origin': 1, 'WketoneW': 1, 'WCH2W': 2, 'RWdblNW': 3, 'RWCH2W': 1, 'OCCC': 1, 'HeteroAromatic': 2, 'PrimOH': 3, 'RWCdblWW': 2}
  id: cpd00005
  nH_std: 27
  name: NADPH
  formula: C21H27N7O17P3
  deltaGf_std: -736.82
  error: Nil
  charge_std: -3
  other_names: ['NADPH', 'TPNH', 'Nicotinamide adenine dinucleotide phosphate - reduced', 'Nicotinamideadeninedinucleotidephosphate-reduced', 'nadph']
--------------------------------------------------
Metabolite ID: cpd00004
  pKa: [1.8, 2.56, 12.69, 13.31, 14.3]
  deltaGf_err: 4.2679
  mass_std: 663.0
  struct_cues: {'RWWNW': 2, 'mid_phos': 1, 'WNH2': 2, 'amide': 1, 'RWCHWW': 8, 'RWOW': 2, 'RWCHdblW': 5, 'WPO4nW': 1, 'TWWCdblW': 2, 'Origin': 1, 'WketoneW': 1, 'WCH2W': 2, 'RWdblNW': 3, 'RWCH2W': 1, 'OCCC': 1, 'HeteroAromati

# VERY IMPORTANT CODE

In [ ]:
# ================================
# VERY IMPORTANT CODE
# CORRECT GEM FORMULAS TO MAKE COMPATIBLE WITH THERMODB (BEFORE RUNNING TFA)
# ================================

import re
from collections import Counter

# --------------------------------
# INPUT DATA
# --------------------------------
Thermodb_meta_interest = [
    "C21H27N7O17P3",
    "C21H27N7O14P2",
    "C23H35N7O17P3S",
    "C21H26N7O14P2",
    "C25H39N7O18P3S",
    "C21H26N7O17P3",
    "C25H37N7O18P3S",
    "C21H33N7O16P3S"
]

GEM_meta_interest = [
    "C21H26N7O17P3",
    "C21H27N7O14P2",
    "C23H34N7O17P3S",
    "C21H26N7O14P2",
    "C25H38N7O18P3S",
    "C21H25N7O17P3",
    "C25H36N7O18P3S",
    "C21H32N7O16P3S"
]

# GEM metabolite IDs in SAME ORDER
gem_met_ids = [
    "nadph_c",
    "nadh_c",
    "accoa_c",
    "nad_c",
    "3hbcoa__R_c",
    "nadp_c",
    "aacoa_c",
    "coa_c",
]

# --------------------------------
# FORMULA PARSING UTILITIES
# --------------------------------
def parse_formula(formula):
    """Convert formula string to Counter"""
    return Counter({
        elem: int(count) if count else 1
        for elem, count in re.findall(r'([A-Z][a-z]?)(\d*)', formula)
    })

def counter_to_formula(counter):
    """Convert Counter back to formula string (Hill order, omit 1s)"""
    parts = []

    # Hill system: C, H, then others alphabetically
    if 'C' in counter:
        parts.append(f"C{counter['C']}" if counter['C'] != 1 else "C")
    if 'H' in counter:
        parts.append(f"H{counter['H']}" if counter['H'] != 1 else "H")

    for el in sorted(counter):
        if el in ('C', 'H'):
            continue
        count = counter[el]
        parts.append(f"{el}{count}" if count != 1 else el)

    return ''.join(parts)


def compare_formulas(thermo_f, gem_f):
    """Return element-wise difference GEM − ThermoDB"""
    t = parse_formula(thermo_f)
    g = parse_formula(gem_f)
    elements = set(t) | set(g)
    return {el: g.get(el, 0) - t.get(el, 0) for el in elements}

def update_gem_formula(gem_f, thermo_f):
    """Add missing elements from ThermoDB into GEM formula"""
    g = parse_formula(gem_f)
    t = parse_formula(thermo_f)

    for el, count in t.items():
        if g.get(el, 0) < count:
            g[el] = count

    return counter_to_formula(g)

# --------------------------------
# APPLY UPDATES TO MODEL
# --------------------------------
for met_id, thermo_f, gem_f in zip(gem_met_ids,
                                   Thermodb_meta_interest,
                                   GEM_meta_interest):

    diff = compare_formulas(thermo_f, gem_f)

    # Safety check (CRITICAL for TFA)
    for el in diff:
        if el in ("C", "N", "P") and diff[el] != 0:
            raise ValueError(
                f"Unsafe formula mismatch for {met_id}: element {el}"
            )

    new_formula = update_gem_formula(gem_f, thermo_f)

    if new_formula != gem_f:
        met = mytfa.metabolites.get_by_id(met_id)

        print(f"Updating {met_id}")
        print(f"  old: {gem_f}")
        print(f"  new: {new_formula}")
        print(f"  ThermoDB:       {thermo_f}")

        met.formula = new_formula

        # PRINT FROM MODEL OBJECT
        print(f"  new (model):  {met.formula}")
        print("-" * 40)


print("✅ GEM formulas updated and TFA-ready.")


Updating nadph_c
  old: C21H26N7O17P3
  new: C21H27N7O17P3
  ThermoDB:       C21H27N7O17P3
  new (model):  C21H27N7O17P3
----------------------------------------
Updating accoa_c
  old: C23H34N7O17P3S
  new: C23H35N7O17P3S
  ThermoDB:       C23H35N7O17P3S
  new (model):  C23H35N7O17P3S
----------------------------------------
Updating 3hbcoa__R_c
  old: C25H38N7O18P3S
  new: C25H39N7O18P3S
  ThermoDB:       C25H39N7O18P3S
  new (model):  C25H39N7O18P3S
----------------------------------------
Updating nadp_c
  old: C21H25N7O17P3
  new: C21H26N7O17P3
  ThermoDB:       C21H26N7O17P3
  new (model):  C21H26N7O17P3
----------------------------------------
Updating aacoa_c
  old: C25H36N7O18P3S
  new: C25H37N7O18P3S
  ThermoDB:       C25H37N7O18P3S
  new (model):  C25H37N7O18P3S
----------------------------------------
Updating coa_c
  old: C21H32N7O16P3S
  new: C21H33N7O16P3S
  ThermoDB:       C21H33N7O16P3S
  new (model):  C21H33N7O16P3S
----------------------------------------
✅ GEM formu

In [ ]:
model.metabolites.get_by_id("coa[c]").summary()



NameError: name 'model' is not defined

In [ ]:
# Match metabolites of interest with

mets_to_check = [
    "nadph[c]",
    "nadh[c]",
    "accoa[c]",
    "nad[c]",
    "3hbcoa__R[c]",
    "nadp[c]",
    "aacoa[c]",
    "coa[c]"
]
results = []

for met_id in mets_to_check:
    try:
        met = model.metabolites.get_by_id(met_id)
        model_formula = met.formula
        in_thermo = model_formula in thermo_formulas
        results.append((met_id, met.formula, model_formula, in_thermo))
    except KeyError:
        results.append((met_id, None, None, False))
for met_id, raw, norm, hit in results:
    print(f"{met_id:12s} | model: {raw} | normalized: {norm} | in thermodb: {hit}")


NameError: name 'model' is not defined

# GEM to TFA prep

In [ ]:
# Adjust model compartments as required by TFA

cobra_model.compartments = {
    'c': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    },
    'e': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    },

    'u': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    },
    'p': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    }
}

# Step 1: map metabolites to standard compartments
for met in cobra_model.metabolites:
    if met.compartment in ["c", "cytosol"]:
        met.compartment = "c"
    elif met.compartment in ["e", "extracellular"]:
        met.compartment = "e"

# Step 2: collect all compartments used by metabolites
used_compartments = set([met.compartment for met in cobra_model.metabolites])

# Step 3: ensure all compartments exist and are dicts
for c in used_compartments:
    val = cobra_model.compartments.get(c)
    # overwrite any string or missing compartment with dict
    cobra_model.compartments[c] = {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02
    }

print("Final compartments:", cobra_model.compartments)


Final compartments: {'c': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}, 'u': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}, 'p': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}, 'e': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}}


## Run test

In [ ]:
thermo_data = {
    "name": "PHB_minimal",

    "units": {
        "energy": "kJ/mol"
    },

    "metabolites": {
        m.id: {"formula": m.formula}
        for m in model.metabolites
        if m.formula is not None and m.id.endswith("_c")
    },

    "cues": {
        "AACOAR_syn": {
            "delta_g0": -25.0,
            "uncertainty": 5.0
        }
    }
}


NameError: name 'model' is not defined

In [ ]:
thermo_data.keys()

dict_keys(['name', 'units', 'metabolites', 'cues'])

In [ ]:
for m in model.metabolites:
    m.id = m.id.replace("[c]", "_c")


In [ ]:
# Build thermo model

# Ensure the COBRA model has a description attribute
if not hasattr(model, "description"):
    model.description = ""

# Correct argument order for your pyTFA version
tmodel = ThermoModel(thermo_data, model)

for c, val in tmodel.compartments.items():
    if isinstance(val, dict):
        tmodel.compartments[c] = c  # just keep the key as a string



2025-12-30 06:15:07,156 - thermomodel_Model Exported from COBRA Toolbox - INFO - # Model initialized with units {'energy': 'kJ/mol'} and temperature 298.15 K
INFO:thermomodel_Model Exported from COBRA Toolbox:# Model initialized with units {'energy': 'kJ/mol'} and temperature 298.15 K


# Add missing metabolites seed ID

In [ ]:
# Adjust constraints since some reactions have values ><1000
for rxn in tmodel.reactions:
    if rxn.lower_bound < -1000:
        rxn.lower_bound = -1000
    if rxn.upper_bound > 1000:
        rxn.upper_bound = 1000

In [ ]:
# CHECK MODEL BEFORE TFA
model.summary() #  is the default objective function

Metabolite,Reaction,Flux,C-Number,C-Flux
ac[e],EX_ac_e,0.2477,2,100.00%
photon690[e],EX_photon690_e,0.626,0,0.00%
phbg_c,SK_phbg_c,0.1237,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
PHB_c,DM_PHB_c,-0.1237,4,99.92%
h2o[e],EX_h2o_e,-5E-05,0,0.00%
o2[e],EX_o2_e,-0.1238,0,0.00%
2obut_c,sink_2obut_c,-0.0001,4,0.08%


In [ ]:
model.metabolites.get_by_id('nadph[c]').summary()

Percent,Flux,Reaction,Definition
17.43%,0.396,ALDD2y,acald[c] + h2o[c] + nadp[c] <=> ac[c] + 2.0 h[c] + nadph[c]
4.76%,0.1081,DESAT16a,h[c] + nadph[c] + o2[c] + palmACP[c] <=> 2.0 h2o[c] + hdeACP[c] + nadp[c]
17.72%,0.4027,FNOR_1,2.0 fdxrd[c] + h[c] + nadp[c] <=> 2.0 fdxox[c] + nadph[c]
4.36%,0.099,G3PD2,glyc3p[c] + nadp[c] <=> dhap[c] + h[c] + nadph[c]
10.90%,0.2476,ICDHyr,icit[c] + nadp[c] --> akg[c] + co2[c] + nadph[c]
44.84%,1.019,NADTRHD,nad[c] + nadph[c] <=> nadh[c] + nadp[c]
Percent,Flux,Reaction,Definition
5.45%,-0.1237,AACOAR_syn,aacoa[c] + h[c] + nadph[c] --> 3hbcoa__R[c] + nadp[c]
17.43%,-0.3961,ASAD,aspsa[c] + nadp[c] + pi[c] <-- 4pasp[c] + h[c] + nadph[c]
61.86%,-1.406,C161SN,actACP[c] + 19.0 h[c] + 6.0 malACP[c] + 13.0 nadph[c] --> 6.0 ACP[c] + 6.0 co2[c] + 7.0 h2o[c] + hdeACP[c] + 13.0 nadp[c]


# Match reactions

In [ ]:
# CHECK PRECURSOR REACTIONS
precursor_rxns = [
    "AACOAR_syn",
    "PHBS_syn",
    "ACACT1r",
    "ACACCT",
]

for rxn_id in precursor_rxns:
    if any(rxn_id in v.name for v in dg_vars):
        print(f"{rxn_id}: thermo-constrained ✅")
    else:
        print(f"{rxn_id}: NOT thermo-constrained ❌")


AACOAR_syn: NOT thermo-constrained ❌
PHBS_syn: NOT thermo-constrained ❌
ACACT1r: NOT thermo-constrained ❌
ACACCT: NOT thermo-constrained ❌


In [ ]:
# Assuming your model is called 'model' and it's a COBRApy model

# Get the metabolite
met = model.metabolites.get_by_id("3hbcoa__R[c]")

# Find reactions where this metabolite is produced (stoichiometry > 0)
producing_reactions = [rxn for rxn in met.reactions if rxn.get_coefficient(met) > 0]

# Print them
for rxn in producing_reactions:
    print(rxn.id, rxn.reaction)


AACOAR_syn aacoa[c] + h[c] + nadph[c] --> 3hbcoa__R[c] + nadp[c]


In [ ]:


# Re-run prepare and convert
tmodel.prepare()
tmodel.convert()


2025-12-30 06:15:19,195 - thermomodel_Model Exported from COBRA Toolbox - INFO - # Model preparation starting...
INFO:thermomodel_Model Exported from COBRA Toolbox:# Model preparation starting...
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite gam6p_c (D-Glucosamine 6-phosphate) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite cgly_c (Cys Gly C5H10N2O3S) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite achms_c (O Acetyl L homoserine C6H11NO4) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite pcox_u (Plastocyanin(Cu2+)) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite octe9ACP_c (Cis-octadec-9-enoyl-[acyl-carrier protein] ((9Z)-n-C18:1)) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite fdp_c (D-Fructose 1,6-bisphosphate) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite 5caiz_c (5-phosphoribosyl-5-carbo

DEBUG:thermomodel_Model Exported from COBRA Toolbox:ENO : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PRFGS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PC8XM : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:3OAS60 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:GMPS2 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:RBFK : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:NTRIRfx : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:GTHPi : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:ARGSS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:SPTc : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:H2Otpp : thermo constraint NOT created
DEBUG:ther

DEBUG:thermomodel_Model Exported from COBRA Toolbox:GLUDGE_OLE_PALM : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DGDGS_OLE_PALM : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:H2Otcx : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:GLNTRS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:TYRTRS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:METTRS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:SERTRS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:GLYTRS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PROTRS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:CYSTRS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:ARGTRS : thermo const

DEBUG:thermomodel_Model Exported from COBRA Toolbox:PYDXO : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PYK2 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PYK3 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PYK4 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PYK5 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:R05224_1 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:RBCh : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:RBFSb : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:RBPC : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:SDPTA : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:SERD_L : thermo constraint NOT created
DEBUG:thermo

DEBUG:thermomodel_Model Exported from COBRA Toolbox:MOCOS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:MOGDS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:MPTS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:MPTSS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:MSAR : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:MSO3abcpp : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:MTHFR2 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:NADDP : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:NADH10 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:NADH16pp : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:NADH17pp : thermo constraint NOT created


DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_ethso3_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_etoh_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_feenter_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_for_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_fru_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_g3pc_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_g3pi_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_g3ps_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_galct__D_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_glc__D_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox

Streaming output truncated to the last 5000 lines.
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added constraint: UF_MDDCP1ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added constraint: UR_MDDCP1ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating only use constraints for reactionMDDCP4ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: FU_MDDCP4ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: BU_MDDCP4ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added constraint: SU_MDDCP4ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added constraint: UF_MDDCP4ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added constraint: UR_MDDCP4ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating only use constraints for reactionMDDCP5ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: FU_MDDCP5ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: BU_MDDCP5ex
DEBUG:the

In [ ]:
from pytfa.optim.variables import DeltaG

print("ΔG present:", DeltaG in tmodel.variables)


ΔG present: False


In [ ]:
rxn = tmodel.reactions.get_by_id("AACOAR_syn")

# Manually define standard Gibbs energy (kJ/mol)
tmodel.add_reaction_thermo(
    rxn,
    delta_g0=-25.0,    # literature estimate
    uncertainty=5.0
)


NameError: name 'tmodel' is not defined

In [ ]:
print("ΔG variables present:", DeltaG in tmodel.variables)

if DeltaG not in tmodel.variables:
    raise RuntimeError(
        "❌ No DeltaG variables created → thermo_data not matched"
    )


ΔG variables present: False


RuntimeError: ❌ No DeltaG variables created → thermo_data not matched

In [ ]:
tmodel.metabolites.get_by_id("nadph[c]").concentration = (1e-6, 1e-3)
tmodel.metabolites.get_by_id("nadp[c]").concentration  = (1e-6, 1e-3)
tmodel.metabolites.get_by_id("h[c]").concentration     = (1e-8, 1e-6)
tmodel.metabolites.get_by_id("coa[c]").concentration   = (1e-6, 1e-3)


KeyError: 'nadph_c'

In [ ]:
rxn_id = "AACOAR_syn"
rxn = tmodel.reactions.get_by_id(rxn_id)

dg = tmodel.variables[DeltaG][rxn]

print(f"{rxn_id} ΔG bounds (kJ/mol):")
print("  LB:", dg.lb)
print("  UB:", dg.ub)


In [ ]:
# Solve
# FBA max PHB
fba = model.optimize()

# TFA max PHB
tfa = tmodel.optimize()
print("Thermo-FBA objective:", tfa.objective_value)

DEBUG:thermomodel_Model Exported from COBRA Toolbox:'slim_optimize' ((), {}) 91.40 sec


Thermo-FBA objective: 0.18


In [ ]:
print(tfa.objective_value -fba.objective_value)


In [ ]:
import inspect
from pytfa import ThermoModel

print(inspect.signature(ThermoModel))


(thermo_data=None, model=<Model None at 0x7e7adc9688c0>, name=None, temperature=298.15, min_ph=3, max_ph=9)


In [ ]:
# Reaction ID in your GEM
rxn = tmodel.reactions.get_by_id("AACOAR_syn")


# Add standard Gibbs energy (kJ/mol)
tmodel.add_reaction_thermo(
    rxn,
    delta_g0=-25.0,   # literature-based estimate
    uncertainty=5.0
)

In [ ]:
# --------------------------------------
# 2. PREPARE + CONVERT (safe even if partial)
# --------------------------------------
tmodel.convert()

2025-12-30 02:59:25,442 - thermomodel_Model Exported from COBRA Toolbox - INFO - # Model preparation starting...
INFO:thermomodel_Model Exported from COBRA Toolbox:# Model preparation starting...
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite gam6p[c] (D-Glucosamine 6-phosphate) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite cgly[c] (Cys Gly C5H10N2O3S) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite achms[c] (O Acetyl L homoserine C6H11NO4) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite pcox_u (Plastocyanin_Cu2+) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite octe9ACP[c] (Cis-octadec-9-enoyl-_acyl-carrier protein __9Z-n-C18:1) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite fdp[c] (D-Fructose 1,6-bisphosphate) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite 5caiz[c] (5-phosphoribosyl-5-car

DEBUG:thermomodel_Model Exported from COBRA Toolbox:NTPP8 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PC11M : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:ARGabcpp : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PMDPHT : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DAPE : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:LYCOPC : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:RB15BPtcx : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:GLCBRAN3 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DB4PS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DXPRIi : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:GTPCII : thermo constraint NOT creat

DEBUG:thermomodel_Model Exported from COBRA Toolbox:PRMICI : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:CYTBD4um : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DM_co_c : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:CAT : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:GLUR : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:ALAR : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:SULR_2 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:NAt3pp : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:ADCS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:UGMDDS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:BIOMASS_CARB : thermo constraint NOT created

DEBUG:thermomodel_Model Exported from COBRA Toolbox:G3PAT1819Z_1 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:AGPAT160 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:AGPATACP_HDE_PALM : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:AGPAT161 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:AGPATACP_OLE_HDE : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:AGPATACP_OLE_PALM : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PAPA160 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PAPA_HDE_PALM : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PAPA161 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PAPA_OLE_HDE : thermo constraint NOT created
DEBUG:thermomodel_Model Exported 

DEBUG:thermomodel_Model Exported from COBRA Toolbox:DASYN182_9_12 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DASYN183_6_9_12 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DASYN183_9_12_15 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DASYN184_6_9_12_15 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DHDPS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DHORDi : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DPPS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EAR121y : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EAR141y : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EAR161y : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:

DEBUG:thermomodel_Model Exported from COBRA Toolbox:HACD4 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HACD5 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HACD6 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HACD7 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HACD8 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HADPCOADH3 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HBZOPT : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HEPT1 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HEPT2 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HG2abcpp : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HISTD : thermo constraint NOT created
D

DEBUG:thermomodel_Model Exported from COBRA Toolbox:OCOAT1 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:OOR3r : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:OPHHX : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:OXOAEL : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:OXPTNDH : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PACCOAL3 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PC : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PEPCK_re : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PGK_1 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PHAPC100 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PHAPC120 : thermo constraint NOT creat

Streaming output truncated to the last 5000 lines.
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating thermo variables for starch[e]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: LC_starch[e]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating thermo variables for zcarote[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: LC_zcarote[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating thermo variables for 1h12dhl[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: LC_1h12dhl[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating thermo variables for 34dhardv[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: LC_34dhardv[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating thermo variables for 34dhrdv[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: LC_34dhrdv[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating t

In [ ]:
DeltaG in tmodel.variables


False

In [ ]:
print(tmodel.variables.keys())


['QULNS', 'QULNS_reverse_66da1', 'ORNDC', 'ORNDC_reverse_63596', 'MSBENZMT', 'MSBENZMT_reverse_a902a', 'DESAT18a', 'DESAT18a_reverse_fd859', 'FUM', 'FUM_reverse_d3642', 'PHYFXOR', 'PHYFXOR_reverse_84960', 'VPAMTr', 'VPAMTr_reverse_872bd', 'GLYCL', 'GLYCL_reverse_e418f', 'NDPK7', 'NDPK7_reverse_9dc79', 'GTPCI', 'GTPCI_reverse_1ee86', 'MTHFC', 'MTHFC_reverse_f6fcc', 'GCATENEC', 'GCATENEC_reverse_ae4a5', 'ORNTA', 'ORNTA_reverse_5adff', 'UPPDC1', 'UPPDC1_reverse_cb592', 'GARFT', 'GARFT_reverse_7ecb6', 'H4THDPR', 'H4THDPR_reverse_617be', 'UDPG4E', 'UDPG4E_reverse_08c7f', 'GLCS3', 'GLCS3_reverse_5e7ed', 'ASPTA', 'ASPTA_reverse_36525', 'PRAGSr', 'PRAGSr_reverse_fd2d8', 'ACKr', 'ACKr_reverse_b49c0', 'KAS14', 'KAS14_reverse_25582', 'UDCPDPS', 'UDCPDPS_reverse_04082', 'GLNS', 'GLNS_reverse_59581', 'SHKK', 'SHKK_reverse_163fd', 'G1PACT', 'G1PACT_reverse_51580', '5DOAN', '5DOAN_reverse_c8e7d', 'OHPBAT', 'OHPBAT_reverse_7e72e', 'Htex', 'Htex_reverse_6f9a4', '3HAD60', '3HAD60_reverse_d7fb5', 'PGI', 

In [ ]:


# --------------------------------------
# 3. MANUAL SEED-ID ANNOTATION (COFACTORS ONLY)
# --------------------------------------
SEED_MAP1 = {
    "h[c]":     "cpd00067",
    "nadp[c]":  "cpd00005",
    "nadph[c]": "cpd00006",
    "coa[c]":   "cpd00010"
}

for met_id, seed_id in SEED_MAP1.items():
    met = tmodel.metabolites.get_by_id(met_id)
    met.annotation["seed_id"] = seed_id


conc_bounds = {
    "nadph[c]": (1e-6, 1e-3),
    "nadp[c]":  (1e-6, 1e-3),
    "h[c]":     (1e-8, 1e-6),
    "coa[c]":   (1e-6, 1e-3),
}

for met_id, bounds in conc_bounds.items():
    met = tmodel.metabolites.get_by_id(met_id)
    met.concentration = bounds

# --------------------------------------
# 5. FORCE THERMODYNAMIC FEASIBILITY
#    Acetoacetyl-CoA reductase (AACOAR)
# --------------------------------------
# Reaction: aacoa + nadph + H -> 3hbcoa + nadp
# Literature ΔG°′ ≈ −25 to −30 kJ/mol
# We enforce directionality conservatively


from pytfa.optim.variables import DeltaG

rxn = tmodel.reactions.get_by_id("AACOAR_syn")
dg = tmodel.variables[DeltaG][rxn]

# Set thermodynamic directionality
dg.upper_bound = -1.0   # kJ/mol
dg.lower_bound = -100.0

# --------------------------------------
# 6. OPTIONAL: OBJECTIVE (PHB / growth)
# --------------------------------------
# Example: maximize PHB precursor
# tmodel.objective = "PHB_SYN"

# --------------------------------------
# 7. RUN TFA OPTIMIZATION
# --------------------------------------
solution = tmodel.optimize()

dg_value = solution.raw[dg.name]
print("ΔG_AACOAR (kJ/mol):", dg_value)


# --------------------------------------
# 8. REPORT RESULTS
# --------------------------------------
print("TFA status:", solution.status)
print("Objective value:", solution.objective_value)

print("\n--- AACOAR thermodynamics ---")
print("Flux:", solution.fluxes["AACOAR_syn"])
print("ΔG (kJ/mol):", solution.raw[tmodel.reactions.AACOAR_syn.delta_g.name])

# Optional: check PHB precursor
if "3hbcoa__R[c]" in solution.fluxes:
    print("\n3HB-CoA flux:", solution.fluxes["3hbcoa__R[c]"])


KeyError: <class 'pytfa.optim.variables.DeltaG'>

In [ ]:
from pytfa.optim.variables import DeltaG

print("DeltaG variables:")
for r, var in tmodel.variables[DeltaG].items():
    if "AACOAR" in r.id:
        print(var.name)


DeltaG variables:


KeyError: <class 'pytfa.optim.variables.DeltaG'>

In [ ]:
from pytfa.optim.variables import DeltaG

phb_rxns = [
    "AACOAR_syn",
    "PHBS_syn",
    "ACACT1r",
    "ACACCT",
]

# Force thermo mode
tmodel.convert(add_thermo=True)

for r_id in phb_rxns:
    rxn = tmodel.reactions.get_by_id(r_id)

    dg = tmodel.variables[DeltaG][rxn]
    dg.lower_bound = -100
    dg.upper_bound = -5   # irreversible forward


TypeError: ThermoModel.convert() got an unexpected keyword argument 'add_thermo'

In [ ]:
def has_dGr_variable(tmodel, rxn_id):
    return any(v.name == f"DG_r_{rxn_id}" for v in tmodel.variables)

for rxn_id in ["ACACT1", "ACCOAC", "PHBS_syn"]:
    print(rxn_id, has_dGr_variable(tmodel, rxn_id))

def thermo_constraints_for_rxn(tmodel, rxn_id):
    return [
        c.name for c in tmodel.constraints
        if rxn_id in c.name and "thermo" in c.name.lower()
    ]

for rxn_id in ["ACACT1", "ACCOAC", "PHBS_syn"]:
    cons = thermo_constraints_for_rxn(tmodel, rxn_id)
    print(rxn_id, cons)


rows = []
for rxn_id in ["ACACT1", "ACCOAC", "PHBS_syn"]:
    rows.append({
        "reaction": rxn_id,
        "has_DG": has_dGr_variable(tmodel, rxn_id),
        "n_thermo_constraints": len(thermo_constraints_for_rxn(tmodel, rxn_id))
    })

pd.DataFrame(rows)


ACACT1 False
ACCOAC False
PHBS_syn False
ACACT1 []
ACCOAC []
PHBS_syn []


,reaction,has_DG,n_thermo_constraints
0,ACACT1,False,0
1,ACCOAC,False,0
2,PHBS_syn,False,0


In [ ]:
thermo_rxns = []

for rxn in tmodel.reactions:
    rxn_id = rxn.id

    # 1) check ΔG variable
    has_dg = f"DG_r_{rxn_id}" in [v.name for v in tmodel.variables]

    if not has_dg:
        continue

    # 2) check thermo constraints
    thermo_cons = [
        c for c in tmodel.constraints
        if rxn_id in c.name and "thermo" in c.name.lower()
    ]

    if len(thermo_cons) > 0:
        thermo_rxns.append(rxn_id)

print(f"Number of thermo-constrained reactions: {len(thermo_rxns)}")
thermo_rxns[:20]


Number of thermo-constrained reactions: 0


[]

In [ ]:
rxn = tmodel.reactions.get_by_id('PHBS_syn')
for met in rxn.metabolites:
    print(met.id, getattr(met, "seed_id", "no seed_id"))


3hbcoa__R[c] C12345
phbg[c] C23456
coa[c] C00010
PHB[c] C34567


In [ ]:
# List all variables (variable objects)
for var in tmodel.variables:
    print(var)


In [ ]:
rxn = tmodel.reactions.get_by_id('PHBS_syn')

if hasattr(rxn, 'deltaG_r') and rxn.deltaG_r in tmodel.variables:
    print(f"{rxn.id} now has a thermodynamic constraint (ΔG applied).")
else:
    print(f"{rxn.id} still does NOT have a thermodynamic constraint.")


PHBS_syn still does NOT have a thermodynamic constraint.


## Map to ModelSEED Database

*   https://modelseed.org/biochem/compounds/cpd00805

*   https://academic.oup.com/nar/article/49/D1/D575/5912569

*   https://github.com/ModelSEED/ModelSEEDpy/tree/dev/tests/biochem

*   https://github.com/ModelSEED/ModelSEEDDatabase?utm_source=chatgpt.com




In [13]:
import json
from pathlib import Path

db_path = Path(MODELS_DIR / "ModelSEEDDatabase"/"Biochemistry"/"compounds.json")
if not db_path.exists():
    raise FileNotFoundError(f"File not found: {db_path}")

with open(db_path) as f:
    compounds = json.load(f)


In [14]:
print(len(compounds))
print(compounds[100])

33992
{'abbreviation': 'r5p', 'abstract_compound': None, 'aliases': ["Name: CPD-15317; D-Ribose 5-phosphate; D-ribose 5'-phosphate; D-ribose 5-phosphate; D-ribose-5-P; D-ribose-5-phosphate; D-ribose-5-phosphoric acid; RIBOSE-5P; Ribose 5-phosphate; aldehydo-D-ribose 5-phosphate; alpha-D-Ribose5-phosphate; alpha-D-ribose-5-phosphate; keto-D-ribose 5-phosphate; ribose-5-P; ribose-5-phosphate; ribose-5-phosphoric acid; ribose-5P", 'AraCyc: RIBOSE-5P', 'BiGG: r5p', 'BrachyCyc: RIBOSE-5P', 'KEGG: C00117', 'MetaCyc: CPD-15317; CPD-15895; RIBOSE-5P'], 'charge': -2, 'comprised_of': None, 'deltag': -394.12, 'deltagerr': 1.32, 'formula': 'C5H9O8P', 'id': 'cpd00101', 'inchikey': 'KTVPXOYAKDPRHY-SOOFDHNKSA-L', 'is_cofactor': 0, 'is_core': 1, 'is_obsolete': 0, 'linked_compound': None, 'mass': 229.0, 'name': 'ribose-5-phosphate', 'notes': ['GC'], 'pka': '1:3:1.22;1:4:6.25;1:11:12.91;1:13:14.31;1:14:11.31', 'pkb': '1:9:-4.39;1:11:-3.68;1:13:-3.94;1:14:-4.39', 'smiles': 'O=P([O-])([O-])OC[C@H]1OC(O)[C

In [15]:
# Create mapping from BiGG, KEGG, Name, etc.
bigg_to_seed = {}

for c in compounds:
    seed_id = c["id"]
    aliases = c.get("aliases") or []
    for entry in aliases:
        entry = str(entry).strip()
        if entry.startswith(("BiGG:", "KEGG:", "Name:")):
            ids = entry.split(":", 1)[1]
            for alias in ids.split(";"):
                alias = alias.strip().lower()
                if alias:
                    if alias not in bigg_to_seed:
                        bigg_to_seed[alias] = seed_id

In [16]:
print(bigg_to_seed.get("h2o"))      # could give cpd00001 if present
print(bigg_to_seed.get("nad"))    # check compartmented version
print(bigg_to_seed.get("fe2"))    # check name alias
print(bigg_to_seed.get(""))   # check KEGG.get("h2o_c"))  # should return 'cpd00001'

cpd00001
cpd00003
cpd10515
None


In [ ]:
def base_met(m):
    return m.rsplit("_",1)[0]

lexicon = []

for m in cobra_model.metabolites:

    base = base_met(m.id)

    seed_id = bigg_to_seed.get(base)

    lexicon.append((m.id, seed_id))

In [ ]:
import pandas as pd

lexicon = pd.DataFrame(
    lexicon,
    columns=["metabolite_id","seed_id"]
)

In [ ]:
mapped = lexicon.seed_id.notna().sum()

print("coverage:", mapped/len(lexicon))

coverage: 0.6944444444444444


In [ ]:
lexicon[lexicon.seed_id.notna()].head(50)

,metabolite_id,seed_id
0,gam6p_c,cpd00288
1,cgly_c,cpd01017
2,achms_c,cpd03399
5,fdp_c,cpd19036
6,5caiz_c,cpd11310
7,pep_c,cpd00061
8,coa_c,cpd00010
9,hgbam_c,cpd03913
10,nac_c,cpd00218
11,r3mmal_c,cpd03593


In [ ]:
lexicon.to_csv("lexicon_20260308.csv", sep="\t", index=False)

## Precompiled database

In [ ]:
seed_db = {c["id"]: c for c in compounds}
seed_db["cpd00001"]

{'abbreviation': 'h2o',
 'abstract_compound': None,
 'aliases': ['Name: H20; H2O; H3O+; HO-; Hydroxide ion; OH; OH-; Water; hydrogen oxide; hydroxide; hydroxide ion; hydroxyl; hydroxyl ion; oxonium; water',
  'AraCyc: OH; WATER',
  'BiGG: h2o; oh1',
  'BrachyCyc: WATER',
  'KEGG: C00001; C01328',
  'MetaCyc: OH; OXONIUM; WATER'],
 'charge': 0,
 'comprised_of': None,
 'deltag': -37.54,
 'deltagerr': 0.18,
 'formula': 'H2O',
 'id': 'cpd00001',
 'inchikey': 'XLYOFNOQVPJJNP-UHFFFAOYSA-N',
 'is_cofactor': 0,
 'is_core': 1,
 'is_obsolete': 0,
 'linked_compound': None,
 'mass': 18.0,
 'name': 'H2O',
 'notes': ['GC', 'EQ', 'EQU'],
 'pka': '1:1:15.70',
 'pkb': '1:1:-1.80',
 'smiles': 'O',
 'source': 'Primary Database'}

In [ ]:
thermo_mets = []

for _, row in lexicon.iterrows():

    met_id = row["metabolite_id"]
    seed_id = row["seed_id"]

    if pd.isna(seed_id):
        continue

    c = seed_db.get(seed_id)

    if c is None:
        continue

    thermo_mets.append({
        "metabolite_id": met_id,
        "seed_id": seed_id,
        "dGf": c.get("deltag"),
        "dGf_err": c.get("deltagerr"),
        "formula": c.get("formula"),
        "charge": c.get("charge"),
        "pka": c.get("pka"),
        "pkb": c.get("pkb")
    })

In [ ]:
thermo_mets = pd.DataFrame(thermo_mets)

In [ ]:
print(len(thermo_mets))
print(len(cobra_model_mat.metabolites))

1475
2128


In [ ]:
thermo_mets = thermo_mets[thermo_mets["dGf"].notna()]

In [ ]:
import json

met_list = []

for _, row in thermo_mets.iterrows():

    met_id = row["metabolite_id"]

    met_list.append({
        "id": met_id,
        "name": cobra_model_mat.metabolites.get_by_id(met_id).name,
        "compartment": met_id.split("_")[-1],
        "charge": int(row["charge"]) if row["charge"] is not None else None,
        "formula": row["formula"]
    })

thermo_json = {"metabolites": met_list}

with open("thermo_metabolites.json","w") as f:
    json.dump(thermo_json, f, indent=2)

# *Save xml cobra model to json

In [17]:
from cobra.io import read_sbml_model, save_json_model

cobra_model = read_sbml_model(PHB_MODEL_DIR / "02_model_rpalustris_PHB_constrained.xml")

save_json_model(cobra_model, PHB_MODEL_DIR / "02_model_rpalustris_PHB_constrained.json")

In [18]:
cobra_model = load_json_model(
    PHB_MODEL_DIR / "02_model_rpalustris_PHB_constrained.json"
)

In [ ]:
import pandas as pd

lexicon_path = PHB_MODEL_TFA_DIR / "lexicon_iDT1294.csv"# "lexicon_20260308.csv"

# Read CSV properly
lexicon = pd.read_csv(
    lexicon_path,
    sep="\t",        # tab-separated file
    header=None,     # no header in the file
    skiprows=0       # skip extra header row if present, otherwise 0
)

# Rename columns
lexicon.columns = ["metabolite_id", "seed_id"]

# Replace missing seed_id with NaN
lexicon["seed_id"] = lexicon["seed_id"].replace("", pd.NA)

# Save a fixed version
lexicon.to_csv(PHB_MODEL_TFA_DIR / "lexicon_fixed.csv", index=False)

# Check
print(lexicon.head())
print(lexicon.columns)




In [ ]:
lexicon_path = PHB_MODEL_TFA_DIR / "lexicon_iDT1294.csv" # "lexicon_20260308.csv"
lexicon = pd.read_csv(lexicon_path)
lexicon.rename(columns={"metabolite_id": "met_id"}, inplace=True)
lexicon.head()

,Unnamed: 0,seed_id
0,3hbcoa__R_c,cpd00842
1,3hbcoa_c,cpd03043
2,3hbycoa_c,cpd00842
3,aacoa_c,cpd00279
4,ac_c,cpd00029


In [ ]:
compartment_data = read_compartment_data(
    str(PHB_MODEL_TFA_DIR / "compartment_data.json")
)

In [ ]:
thermo_data = load_thermoDB(PHB_MODEL_TFA_DIR / "thermo_data.thermodb")  # if loaded correctly, this should be a ThermoData
# but if you used json.load(), then thermo_data is a dict

In [ ]:
thermo_data['metabolites'].keys()

dict_keys([np.str_('cpd00805'), np.str_('cpd11364'), np.str_('cpd18424'), np.str_('cpd17488'), np.str_('cpd03902'), np.str_('cpd14767'), np.str_('cpd05076'), np.str_('cpd00475'), np.str_('cpd17550'), np.str_('cpd08432'), np.str_('cpd07669'), np.str_('cpd02398'), np.str_('cpd19498'), np.str_('cpd01442'), np.str_('cpd11437'), np.str_('cpd00087'), np.str_('cpd11395'), np.str_('cpd04959'), np.str_('cpd15625'), np.str_('cpd18361'), np.str_('cpd06546'), np.str_('cpd00372'), np.str_('cpd18691'), np.str_('cpd19841'), np.str_('cpd00872'), np.str_('cpd17787'), np.str_('cpd19729'), np.str_('cpd00304'), np.str_('cpd14524'), np.str_('cpd10597'), np.str_('cpd14588'), np.str_('cpd01231'), np.str_('cpd04691'), np.str_('cpd00111'), np.str_('cpd04068'), np.str_('cpd09476'), np.str_('cpd11126'), np.str_('cpd16375'), np.str_('cpd15649'), np.str_('cpd09999'), np.str_('cpd03405'), np.str_('cpd09059'), np.str_('cpd18413'), np.str_('cpd01057'), np.str_('cpd19453'), np.str_('cpd19201'), np.str_('cpd08677'), np

In [ ]:
from pytfa.io import load_thermoDB
from pathlib import Path
import pandas as pd

# --- 1️⃣ Load thermoDB ---
thermo_data = load_thermoDB(PHB_MODEL_TFA_DIR / "thermo_data.thermodb")


# --- 2️⃣ Load lexicon & compartment data ---
lexicon = read_lexicon(PHB_MODEL_TFA_DIR / "lexicon_fixed.csv")
compartment_data = read_compartment_data(
    str(PHB_MODEL_TFA_DIR / "compartment_data.json")
)

# Lexicon seed IDs as Python strings
lexicon_seed_ids = set(map(str, lexicon['seed_id'].dropna()))

# Thermo seed IDs as Python strings
thermo_seed_ids = set(map(str, thermo_data['metabolites'].keys()))

present_in_thermo = lexicon_seed_ids & thermo_seed_ids
missing_in_thermo = lexicon_seed_ids - thermo_seed_ids

print(f"Seed IDs present in thermo_data: {present_in_thermo}")
print(f"Seed IDs missing in thermo_data: {missing_in_thermo}")



Seed IDs present in thermo_data: {'cpd15432', 'cpd00054', 'cpd15325', 'cpd15660', 'cpd16747', 'cpd00044', 'cpd00086', 'cpd15349', 'cpd16750', 'cpd00235', 'cpd00032', 'cpd12255', 'cpd15277', 'cpd01262', 'cpd01977', 'cpd12228', 'cpd03584', 'cpd01113', 'cpd00261', 'cpd00209', 'cpd00342', 'cpd15421', 'cpd15551', 'cpd15307', 'cpd08405', 'cpd03122', 'cpd00403', 'cpd03671', 'cpd00875', 'cpd01695', 'cpd00277', 'cpd08287', 'cpd11341', 'cpd15399', 'cpd12733', 'cpd00739', 'cpd02724', 'cpd15275', 'cpd02394', 'cpd00207', 'cpd12100', 'cpd15419', 'cpd11255', 'cpd00296', 'cpd15578', 'cpd02484', 'cpd02678', 'cpd15539', 'cpd00484', 'cpd03387', 'cpd00092', 'cpd00355', 'cpd15413', 'cpd00800', 'cpd01844', 'cpd00626', 'cpd01563', 'cpd03704', 'cpd15525', 'cpd00523', 'cpd15341', 'cpd15481', 'cpd01997', 'cpd02345', 'cpd15437', 'cpd00120', 'cpd08305', 'cpd15468', 'cpd03593', 'cpd00309', 'cpd15572', 'cpd15313', 'cpd15529', 'cpd01882', 'cpd00413', 'cpd00266', 'cpd00755', 'cpd03752', 'cpd15422', 'cpd01171', 'cpd00

In [ ]:
# --- 4️⃣ Initialize ThermoModel ---
mytfa = pytfa.ThermoModel(thermo_data, cobra_model)

# --- 5️⃣ Annotate metabolites from lexicon and apply compartment info ---

annotate_from_lexicon(mytfa, lexicon)
apply_compartment_data(mytfa, compartment_data)


2026-03-09 15:31:51,938 - thermomodel_None - INFO - # Model initialized with units kcal/mol and temperature 298.15 K
INFO:thermomodel_None:# Model initialized with units kcal/mol and temperature 298.15 K


# Lexicon autocreation V2

In [ ]:
from pathlib import Path
import pandas as pd

# --- Paths ---

compounds_path = MODELS_DIR / "ModelSEEDDatabase"/"Biochemistry"/"compounds.json"  # your ModelSEED compounds JSON
#lexicon_out_path = PHB_MODEL_TFA_DIR / "lexicon_auto.csv"



# --- Load ModelSEED compounds ---
import json
# --- Load ModelSEED compounds ---
with open(compounds_path) as f:
    compounds = json.load(f)

# --- Create mappings ---
name_to_seed = {}
bigg_to_seed = {}

for c in compounds:
    seed_id = c["id"]

    # Map names / synonyms
    names = [c.get("name", "")] + (c.get("aliases") or [])
    for n in names:
        if n:
            n_clean = str(n).strip().lower()
            name_to_seed[n_clean] = seed_id

    # Map BiGG IDs if available
    aliases = c.get("aliases") or []
    for entry in aliases:
        if isinstance(entry, str) and entry.startswith("BiGG:"):
            ids = entry.split(":", 1)[1]
            for bigg in ids.split(";"):
                bigg = bigg.strip().lower()
                if bigg:
                    bigg_to_seed[bigg] = seed_id

# --- Function to strip compartment suffix ---
def strip_compartment(met_id):
    return re.sub(r'_[cpep]$', '', met_id.lower())

# --- Map COBRA model metabolites ---
# Assume cobra_model is already loaded
lexicon_rows = []

for m in cobra_model.metabolites:
    met_id_clean = strip_compartment(m.id)

    # 1️⃣ Try BiGG ID mapping
    seed_id = bigg_to_seed.get(met_id_clean)

    # 2️⃣ Try metabolite name mapping (fallback)
    if seed_id is None and hasattr(m, "name"):
        seed_id = name_to_seed.get(m.name.lower())

    # 3️⃣ Try stripped metabolite id as a last resort
    if seed_id is None:
        seed_id = name_to_seed.get(met_id_clean)

    # 4️⃣ If still not found, leave empty
    if seed_id is None:
        seed_id = ""

    lexicon_rows.append({"met_id": m.id, "seed_id": seed_id})

# --- Save lexicon ---
lexicon_df = pd.DataFrame(lexicon_rows)
lexicon_df.to_csv(lexicon_out_path, index=False)

# --- Report coverage ---
mapped = (lexicon_df['seed_id'] != '').sum()
total = len(lexicon_df)
coverage = mapped / total * 100

print(f"Lexicon saved to {lexicon_out_path}")
print(f"Metabolites mapped: {mapped} / {total} ({coverage:.1f}%)")

# --- List unmapped metabolites for manual curation ---
unmapped = lexicon_df[lexicon_df['seed_id'] == '']
print(f"\nUnmapped metabolites ({len(unmapped)}):")
print(unmapped['met_id'].tolist())

Lexicon saved to /content/drive/MyDrive/metabolic_modelling/phb-optimization-rpalustris/models/phb/tfa/lexicon_auto.csv
Metabolites mapped: 1859 / 2124 (87.5%)

Unmapped metabolites (265):
['14dhncoa_c', 'h2o_cx_c', 'bm_pro_c', 'bm_cw_c', 'bm_pigm_c', 'o2_u', 'u3hga2_c', '3pg_cx_c', 'npdp_c', 'hco3_cx_c', 'bm_cofactors_c', 'co2_cx_c', 'bm_rna_c', 'o2_cx_c', 'bm_dna_c', 'bm_carbs_c', 'bm_memlip_c', 'lipidX2_c', 'phyto_c', 'h2o_u', 'dvpchlda_c', 'lipidAds2_c', 'cthzp_c', 'tczcaro_c', 'dczcaro_c', 'ttclyco_c', 'gthbpt_c', 'iscssh_c', 'iscsh_c', 'this_c', 'athis_c', 'thissh_c', 'ogmeACP_c', 'hgmeACP_c', 'egmeACP_c', 'gmeACP_c', 'opmeACP_c', 'hpmeACP_c', 'epmeACP_c', 'pmeACP_c', 'pimACP_c', 'octapb_c', 'lipopb_c', 'cdg_c', 'preq1_c', 'preqtrna_c', 'epxqtrna_c', 'quetrna_c', 'pa1619Z160_c', '12dgr1619Z160_c', 'sqdg160_c', 'sqdg1619Z160_c', 'mgdg1619Z160_c', 'dgdg1619Z160_c', 'dgdg161_c', 'mgdg161_c', 'glcdg1619Z160_c', 'glcdg161_c', 'glcdg1819Z1619Z_c', 'glcdg1819Z160_c', 'dgdg1819Z160Z_c', 

In [ ]:
# List metabolites with thermo data
annotated = [m.id for m in mytfa.metabolites if hasattr(m, "thermo")]
print(f"Metabolites annotated with thermodynamic data: {len(annotated)} / {len(mytfa.metabolites)}")

Metabolites annotated with thermodynamic data: 2124 / 2124


In [ ]:
for m in mytfa.metabolites[:10]:  # first 10 metabolites
    print(m.id, "->", getattr(m, "thermo", None))

gam6p_c -> {'id': None, 'pKa': [], 'error': None, 'deltaGf_std': 1000.0, 'deltaGf_err': 1000.0, 'mass': 1000.0, 'nH_std': None, 'charge_std': 1000.0, 'struct_cues': None, 'deltaGf_tr': 1000.0, 'pH': 7.5, 'ionicStr': 0.25}
cgly_c -> {'id': None, 'pKa': [], 'error': None, 'deltaGf_std': 1000.0, 'deltaGf_err': 1000.0, 'mass': 1000.0, 'nH_std': None, 'charge_std': 1000.0, 'struct_cues': None, 'deltaGf_tr': 1000.0, 'pH': 7.5, 'ionicStr': 0.25}
achms_c -> {'id': None, 'pKa': [], 'error': None, 'deltaGf_std': 1000.0, 'deltaGf_err': 1000.0, 'mass': 1000.0, 'nH_std': None, 'charge_std': 1000.0, 'struct_cues': None, 'deltaGf_tr': 1000.0, 'pH': 7.5, 'ionicStr': 0.25}
pcox_u -> {'id': None, 'pKa': [], 'error': None, 'deltaGf_std': 1000.0, 'deltaGf_err': 1000.0, 'mass': 1000.0, 'nH_std': None, 'charge_std': 1000.0, 'struct_cues': None, 'deltaGf_tr': 1000.0, 'pH': 7.0, 'ionicStr': 0.25}
octe9ACP_c -> {'id': None, 'pKa': [], 'error': None, 'deltaGf_std': 1000.0, 'deltaGf_err': 1000.0, 'mass': 1000.0,

In [ ]:
unmapped = lexicon_df[lexicon_df['seed_id'] == '']
print(f"Unmapped metabolites ({len(unmapped)}):")
print(unmapped['met_id'].tolist())

Unmapped metabolites (265):
['14dhncoa_c', 'h2o_cx_c', 'bm_pro_c', 'bm_cw_c', 'bm_pigm_c', 'o2_u', 'u3hga2_c', '3pg_cx_c', 'npdp_c', 'hco3_cx_c', 'bm_cofactors_c', 'co2_cx_c', 'bm_rna_c', 'o2_cx_c', 'bm_dna_c', 'bm_carbs_c', 'bm_memlip_c', 'lipidX2_c', 'phyto_c', 'h2o_u', 'dvpchlda_c', 'lipidAds2_c', 'cthzp_c', 'tczcaro_c', 'dczcaro_c', 'ttclyco_c', 'gthbpt_c', 'iscssh_c', 'iscsh_c', 'this_c', 'athis_c', 'thissh_c', 'ogmeACP_c', 'hgmeACP_c', 'egmeACP_c', 'gmeACP_c', 'opmeACP_c', 'hpmeACP_c', 'epmeACP_c', 'pmeACP_c', 'pimACP_c', 'octapb_c', 'lipopb_c', 'cdg_c', 'preq1_c', 'preqtrna_c', 'epxqtrna_c', 'quetrna_c', 'pa1619Z160_c', '12dgr1619Z160_c', 'sqdg160_c', 'sqdg1619Z160_c', 'mgdg1619Z160_c', 'dgdg1619Z160_c', 'dgdg161_c', 'mgdg161_c', 'glcdg1619Z160_c', 'glcdg161_c', 'glcdg1819Z1619Z_c', 'glcdg1819Z160_c', 'dgdg1819Z160Z_c', 'photon410_e', 'photon430_e', 'photon470_e', 'photon510_e', 'photon530_e', 'photon550_e', 'photon570_e', 'photon590_e', 'photon610_e', 'photon630_e', 'photon650_

In [ ]:
mytfa.prepare()

In [ ]:
thermo_data = load_thermoDB(PHB_MODEL_TFA_DIR / "thermo_data.thermodb")
thermo_data.keys
seed_ids_in_thermo = [str(k) for k in thermo_data['metabolites'].keys()]
print(seed_ids)


['cpd00805', 'cpd11364', 'cpd18424', 'cpd17488', 'cpd03902', 'cpd14767', 'cpd05076', 'cpd00475', 'cpd17550', 'cpd08432', 'cpd07669', 'cpd02398', 'cpd19498', 'cpd01442', 'cpd11437', 'cpd00087', 'cpd11395', 'cpd04959', 'cpd15625', 'cpd18361', 'cpd06546', 'cpd00372', 'cpd18691', 'cpd19841', 'cpd00872', 'cpd17787', 'cpd19729', 'cpd00304', 'cpd14524', 'cpd10597', 'cpd14588', 'cpd01231', 'cpd04691', 'cpd00111', 'cpd04068', 'cpd09476', 'cpd11126', 'cpd16375', 'cpd15649', 'cpd09999', 'cpd03405', 'cpd09059', 'cpd18413', 'cpd01057', 'cpd19453', 'cpd19201', 'cpd08677', 'cpd12309', 'cpd16152', 'cpd10577', 'cpd03314', 'cpd06273', 'cpd12213', 'cpd07931', 'cpd02409', 'cpd00542', 'cpd01389', 'cpd02322', 'cpd03481', 'cpd12243', 'cpd09124', 'cpd12572', 'cpd04888', 'cpd01221', 'cpd10369', 'cpd15423', 'cpd02901', 'cpd03043', 'cpd07770', 'cpd01306', 'cpd02020', 'cpd19165', 'cpd09816', 'cpd14564', 'cpd18260', 'cpd07302', 'cpd01778', 'cpd03354', 'cpd16530', 'cpd08525', 'cpd09445', 'cpd04738', 'cpd05001', 'cp

In [ ]:
seed_ids_in_lexicon = set(lexicon_df['seed_id'])
len(seed_ids_in_lexicon)

1295

In [ ]:
seed_ids_in_thermo = {str(k) for k in thermo_data['metabolites'].keys()}
seed_ids_in_lexicon = set(lexicon_df['seed_id'])

missing_in_thermo = seed_ids_in_lexicon - seed_ids_in_thermo
present_in_thermo = seed_ids_in_lexicon & seed_ids_in_thermo
unused_thermo_ids = seed_ids_in_thermo - seed_ids_in_lexicon

In [ ]:
print("Seed IDs in lexicon:", len(seed_ids_in_lexicon))
print("Seed IDs in ThermoDB:", len(seed_ids_in_thermo))
print("Lexicon IDs found in ThermoDB:", len(present_in_thermo))
print("Lexicon IDs missing in ThermoDB:", len(missing_in_thermo))

Seed IDs in lexicon: 1295
Seed IDs in ThermoDB: 18307
Lexicon IDs found in ThermoDB: 1175
Lexicon IDs missing in ThermoDB: 120


In [ ]:
sum("seed_id" in m.annotation for m in cobra_model.metabolites)

0

In [ ]:
cobra_model.metabolites[0].annotation

{'SeedID': 'cpd00288'}

In [ ]:
cobra_model.metabolites[0].annotation["SeedID"]


'cpd00288'

In [ ]:
for m in cobra_model.metabolites:
    if 'SeedID' in m.annotation:
        m.annotation['seed.compound'] = m.annotation['SeedID']

In [ ]:
for m in cobra_model.metabolites[:10]:
    print(m.id, m.annotation)

gam6p_c {'SeedID': 'cpd00288', 'seed.compound': 'cpd00288'}
cgly_c {}
achms_c {}
pcox_u {}
octe9ACP_c {}
fdp_c {}
5caiz_c {}
pep_c {}
coa_c {}
hgbam_c {}


# Fixing

In [ ]:
mytfa = ThermoModel(thermo_data, cobra_model)

mytfa.prepare()

2026-03-09 19:47:42,133 - thermomodel_None - INFO - # Model initialized with units kcal/mol and temperature 298.15 K
INFO:thermomodel_None:# Model initialized with units kcal/mol and temperature 298.15 K
2026-03-09 19:47:42,137 - thermomodel_None - INFO - # Model preparation starting...
INFO:thermomodel_None:# Model preparation starting...
DEBUG:thermomodel_None:Metabolite gam6p_c (D-Glucosamine 6-phosphate) has no seed_id
DEBUG:thermomodel_None:Metabolite cgly_c (Cys Gly C5H10N2O3S) has no seed_id
DEBUG:thermomodel_None:Metabolite achms_c (O Acetyl L homoserine C6H11NO4) has no seed_id
DEBUG:thermomodel_None:Metabolite pcox_u (Plastocyanin(Cu2+)) has no seed_id
DEBUG:thermomodel_None:Metabolite octe9ACP_c (Cis-octadec-9-enoyl-[acyl-carrier protein] ((9Z)-n-C18:1)) has no seed_id
DEBUG:thermomodel_None:Metabolite fdp_c (D-Fructose 1,6-bisphosphate) has no seed_id
DEBUG:thermomodel_None:Metabolite 5caiz_c (5-phosphoribosyl-5-carboxyaminoimidazole) has no seed_id
DEBUG:thermomodel_None:M

DEBUG:thermomodel_None:NDPK1 : thermo constraint NOT created
DEBUG:thermomodel_None:PGL : thermo constraint NOT created
DEBUG:thermomodel_None:EAR160y : thermo constraint NOT created
DEBUG:thermomodel_None:MECDPDHf : thermo constraint NOT created
DEBUG:thermomodel_None:MTHFR3_1 : thermo constraint NOT created
DEBUG:thermomodel_None:3HAD140 : thermo constraint NOT created
DEBUG:thermomodel_None:FBA3 : thermo constraint NOT created
DEBUG:thermomodel_None:LDAPAT : thermo constraint NOT created
DEBUG:thermomodel_None:MG2uabcpp : thermo constraint NOT created
DEBUG:thermomodel_None:HCO3E_1_cx : thermo constraint NOT created
DEBUG:thermomodel_None:ACODA : thermo constraint NOT created
DEBUG:thermomodel_None:GLU5K : thermo constraint NOT created
DEBUG:thermomodel_None:EAR100y : thermo constraint NOT created
DEBUG:thermomodel_None:PGM : thermo constraint NOT created
DEBUG:thermomodel_None:ASPOb : thermo constraint NOT created
DEBUG:thermomodel_None:DM_h2_c : thermo constraint NOT created
DEBUG

DEBUG:thermomodel_None:EX_spmd_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_ca2_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_nh4_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_arg__L_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_gln__L_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_mn2_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_hco3_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_mg2_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_ptrc_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_fe2_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_cu2_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_k_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_no3_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_fe3_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_mobd_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_ni2_e 

DEBUG:thermomodel_None:GLYK : thermo constraint NOT created
DEBUG:thermomodel_None:GLYabcpp : thermo constraint NOT created
DEBUG:thermomodel_None:GMPS : thermo constraint NOT created
DEBUG:thermomodel_None:GUACYC : thermo constraint NOT created
DEBUG:thermomodel_None:H2ASE_syn : thermo constraint NOT created
DEBUG:thermomodel_None:HCO3E : thermo constraint NOT created
DEBUG:thermomodel_None:HIBDkt : thermo constraint NOT created
DEBUG:thermomodel_None:HISabcpp : thermo constraint NOT created
DEBUG:thermomodel_None:HOXG : thermo constraint NOT created
DEBUG:thermomodel_None:HPROa : thermo constraint NOT created
DEBUG:thermomodel_None:HPYRRx : thermo constraint NOT created
DEBUG:thermomodel_None:HSDxi : thermo constraint NOT created
DEBUG:thermomodel_None:IMACTD : thermo constraint NOT created
DEBUG:thermomodel_None:KARA2 : thermo constraint NOT created
DEBUG:thermomodel_None:LALDO : thermo constraint NOT created
DEBUG:thermomodel_None:LCARS : thermo constraint NOT created
DEBUG:thermom

DEBUG:thermomodel_None:LPLIPAL2A160 : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL2A161 : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL2A180 : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL2A181 : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL2ATE120 : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL2ATE140 : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL2ATE141 : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL2ATE160 : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL2ATE161 : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL2ATE180 : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL2ATE181 : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL2ATG120 : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL2ATG140 : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL2ATG141 : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIP

DEBUG:thermomodel_None:MOTH4 : thermo constraint NOT created
DEBUG:thermomodel_None:MOTS1 : thermo constraint NOT created
DEBUG:thermomodel_None:MOTS2 : thermo constraint NOT created
DEBUG:thermomodel_None:MOTS3 : thermo constraint NOT created
DEBUG:thermomodel_None:MOTS4 : thermo constraint NOT created
DEBUG:thermomodel_None:MTHFR2_1 : thermo constraint NOT created
DEBUG:thermomodel_None:MTI : thermo constraint NOT created
DEBUG:thermomodel_None:NADHDH : thermo constraint NOT created
DEBUG:thermomodel_None:NADPHQR2 : thermo constraint NOT created
DEBUG:thermomodel_None:NADPHQR3 : thermo constraint NOT created
DEBUG:thermomodel_None:NADS1 : thermo constraint NOT created
DEBUG:thermomodel_None:NAt3_1 : thermo constraint NOT created
DEBUG:thermomodel_None:NFORGLUAH : thermo constraint NOT created
DEBUG:thermomodel_None:NH4t : thermo constraint NOT created
DEBUG:thermomodel_None:NIT1b_1 : thermo constraint NOT created
DEBUG:thermomodel_None:NTD5pp : thermo constraint NOT created
DEBUG:the

DEBUG:thermomodel_None:GUI2 : thermo constraint NOT created
DEBUG:thermomodel_None:TAGURr : thermo constraint NOT created
DEBUG:thermomodel_None:ALTRH : thermo constraint NOT created
DEBUG:thermomodel_None:EX_urea_e : thermo constraint NOT created
DEBUG:thermomodel_None:UREAtex : thermo constraint NOT created
DEBUG:thermomodel_None:UREA : thermo constraint NOT created
DEBUG:thermomodel_None:UREASE : thermo constraint NOT created
DEBUG:thermomodel_None:ARGN : thermo constraint NOT created
DEBUG:thermomodel_None:ARGN_1 : thermo constraint NOT created
DEBUG:thermomodel_None:UREAabcpp : thermo constraint NOT created
DEBUG:thermomodel_None:ASNS1 : thermo constraint NOT created
DEBUG:thermomodel_None:ASNTRS : thermo constraint NOT created
DEBUG:thermomodel_None:ASNS2 : thermo constraint NOT created
DEBUG:thermomodel_None:EX_ala__D_e : thermo constraint NOT created
DEBUG:thermomodel_None:DALAtex : thermo constraint NOT created
DEBUG:thermomodel_None:DALAt2pp : thermo constraint NOT created
DE

# *Lexicon_auto creation

In [19]:
# --- Paths ---
compounds_path = MODELS_DIR / "ModelSEEDDatabase" / "Biochemistry" / "compounds.json"
lexicon_out_path = PHB_MODEL_TFA_DIR / "lexicon_auto.csv"
thermo_path = DATA_DIR / "thermo_data.thermodb"

# --- Load ThermoDB ---
thermo_data = load_thermoDB(thermo_path)

# --- Load ModelSEED compounds ---
with open(compounds_path) as f:
    compounds = json.load(f)

# --- Create mapping from names / BiGG IDs to seed IDs ---
name_to_seed = {}
bigg_to_seed = {}

for c in compounds:
    seed_id = c["id"]

    # Names / synonyms
    names = [c.get("name", "")] + (c.get("aliases") or [])
    for n in names:
        if n:
            n_clean = str(n).strip().lower()
            name_to_seed[n_clean] = seed_id

    # BiGG IDs in aliases
    aliases = c.get("aliases") or []
    for entry in aliases:
        if isinstance(entry, str) and entry.startswith("BiGG:"):
            ids = entry.split(":", 1)[1]
            for bigg in ids.split(";"):
                bigg = bigg.strip().lower()
                if bigg:
                    bigg_to_seed[bigg] = seed_id

# --- Function to strip compartment suffix ---
def strip_compartment(met_id):
    return re.sub(r'_[cepu]$', '', met_id.lower())

# --- Map COBRA model metabolites ---
lexicon_rows = []

for m in cobra_model.metabolites:
    met_id_clean = strip_compartment(m.id)

    # 1. Try BiGG ID mapping
    seed_id = bigg_to_seed.get(met_id_clean)

    # 2. Try metabolite name mapping
    if seed_id is None and hasattr(m, "name"):
        seed_id = name_to_seed.get(m.name.lower())

    # 3. Try stripped metabolite id as fallback
    if seed_id is None:
        seed_id = name_to_seed.get(met_id_clean)

    # 4. If still not found, leave empty
    if seed_id is None:
        seed_id = ""

    lexicon_rows.append({"met_id": m.id, "seed_id": seed_id})

# --- Save lexicon ---
lexicon_df = pd.DataFrame(lexicon_rows)
lexicon_df.to_csv(lexicon_out_path, index=False)

# --- Report coverage ---
mapped = (lexicon_df['seed_id'] != '').sum()
total = len(lexicon_df)
coverage = mapped / total * 100
print(f"Lexicon saved to {lexicon_out_path}")
print(f"Metabolites mapped: {mapped} / {total} ({coverage:.1f}%)")

# --- Filter out empty seed IDs for PyTFA ---
lexicon_df = lexicon_df[lexicon_df['seed_id'] != '']
lexicon_df['seed_id'] = lexicon_df['seed_id'].astype(str)

# --- Check which seed IDs exist in ThermoDB ---
seed_ids_in_thermo = set(map(str, thermo_data['metabolites'].keys()))
missing_in_thermo = set(lexicon_df['seed_id']) - seed_ids_in_thermo

print(f"Number of seed IDs in lexicon not in ThermoDB: {len(missing_in_thermo)}")
if missing_in_thermo:
    print("Some metabolites will not get thermodynamic constraints:")
    print(sorted(list(missing_in_thermo)))



Lexicon saved to /content/drive/MyDrive/metabolic_modelling/phb-optimization-rpalustris/models/phb/tfa/lexicon_auto.csv
Metabolites mapped: 1862 / 2124 (87.7%)
Number of seed IDs in lexicon not in ThermoDB: 118
Some metabolites will not get thermodynamic constraints:
['cpd15465', 'cpd20862', 'cpd20863', 'cpd20921', 'cpd21090', 'cpd21120', 'cpd21184', 'cpd21214', 'cpd21472', 'cpd21479', 'cpd21488', 'cpd22021', 'cpd22061', 'cpd22062', 'cpd22075', 'cpd22076', 'cpd22077', 'cpd22085', 'cpd22119', 'cpd22495', 'cpd22621', 'cpd23280', 'cpd23281', 'cpd23282', 'cpd23828', 'cpd23829', 'cpd24196', 'cpd24423', 'cpd24605', 'cpd24697', 'cpd25615', 'cpd25960', 'cpd26318', 'cpd26385', 'cpd26436', 'cpd26518', 'cpd26527', 'cpd26726', 'cpd26729', 'cpd26805', 'cpd26830', 'cpd26831', 'cpd26836', 'cpd27013', 'cpd27042', 'cpd27103', 'cpd27358', 'cpd27371', 'cpd27379', 'cpd27436', 'cpd27551', 'cpd27563', 'cpd27569', 'cpd27608', 'cpd27735', 'cpd27797', 'cpd27830', 'cpd28023', 'cpd28024', 'cpd28030', 'cpd28060',

In [21]:
# --- Initialize ThermoModel ---
mytfa = ThermoModel(thermo_data, cobra_model)

lexicon = read_lexicon(lexicon_out_path)

# --- Annotate metabolites with lexicon and compartments ---
annotate_from_lexicon(mytfa, lexicon)
apply_compartment_data(mytfa, cobra_model.compartments)

# --- Check annotated metabolites ---
annotated = [m for m in mytfa.metabolites if hasattr(m, "thermo")]
print(f"Metabolites with thermodynamic data: {len(annotated)} / {len(mytfa.metabolites)}")

# --- Prepare TFA model ---
mytfa.prepare()  # Should now fail less, because lexicon filtering removed empty mappings
print("ThermoModel prepared successfully.")

2026-03-16 15:28:07,252 - thermomodel_None - INFO - # Model initialized with units kcal/mol and temperature 298.15 K
INFO:thermomodel_None:# Model initialized with units kcal/mol and temperature 298.15 K
2026-03-16 15:28:07,282 - thermomodel_None - INFO - # Model preparation starting...
INFO:thermomodel_None:# Model preparation starting...


Metabolites with thermodynamic data: 0 / 2124


TypeError: string indices must be integers, not 'str'

In [ ]:
annotated = [m for m in mytfa.metabolites if hasattr(m, "thermo")]
print(f"Metabolites with thermodynamic data: {len(annotated)} / {len(mytfa.metabolites)}")
print([m.id for m in annotated[:20]])  # first 20 for inspection

Metabolites with thermodynamic data: 2124 / 2124
['gam6p_c', 'cgly_c', 'achms_c', 'pcox_u', 'octe9ACP_c', 'fdp_c', '5caiz_c', 'pep_c', 'coa_c', 'hgbam_c', 'nac_c', 'r3mmal_c', 'leu__L_c', 'ahcys_c', 'ru5p__D_c', '14dhncoa_c', 'h2o_cx_c', 'h2s_c', 'orot_c', 'g1p_c']


In [ ]:
cobra_model.metabolites[0].annotation = {
  'SeedID': 'cpd00288'
}

In [ ]:
cobra_model.metabolites[0]

Metabolite identifier,gam6p_c
Name,D-Glucosamine 6-phosphate
Memory address,0x7bfc5884ec30
Formula,C6H13NO8P
Compartment,c
In 4 reaction(s),"GAMptspp, AGDC, GF6PTA, PGAMT"


In [ ]:
# Example seed ID
seed_id = "cpd00001"

# Check if it exists in ThermoDB
if seed_id in thermo_data['metabolites']:
    print(f"{seed_id} exists in ThermoDB")
else:
    print(f"{seed_id} NOT found in ThermoDB")

cpd00001 exists in ThermoDB


In [ ]:
# Get metabolite
met = mytfa.metabolites.get_by_id("h2o_c")

# Assign seed ID manually
met.seed_id = "cpd00001"

# Now check ThermoDB
if met.seed_id in thermo_data['metabolites']:
    print(f"{met.id} with seed ID {met.seed_id} exists in ThermoDB")
else:
    print(f"{met.id} with seed ID {met.seed_id} NOT found in ThermoDB")

h2o_c with seed ID cpd00001 exists in ThermoDB


In [ ]:
cobra_model.convert()

2026-03-09 05:59:05,324 - thermomodel_None - INFO - # Model conversion starting...
INFO:thermomodel_None:# Model conversion starting...
DEBUG:thermomodel_None:generating thermo variables for gam6p_c
DEBUG:thermomodel_None:Added variable: LC_gam6p_c
DEBUG:thermomodel_None:generating thermo variables for cgly_c
DEBUG:thermomodel_None:Added variable: LC_cgly_c
DEBUG:thermomodel_None:generating thermo variables for achms_c
DEBUG:thermomodel_None:Added variable: LC_achms_c
DEBUG:thermomodel_None:generating thermo variables for pcox_u
DEBUG:thermomodel_None:Added variable: LC_pcox_u
DEBUG:thermomodel_None:generating thermo variables for octe9ACP_c
DEBUG:thermomodel_None:Added variable: LC_octe9ACP_c
DEBUG:thermomodel_None:generating thermo variables for fdp_c
DEBUG:thermomodel_None:Added variable: LC_fdp_c
DEBUG:thermomodel_None:generating thermo variables for 5caiz_c
DEBUG:thermomodel_None:Added variable: LC_5caiz_c
DEBUG:thermomodel_None:generating thermo variables for pep_c
DEBUG:thermomo

KeyboardInterrupt: 

In [ ]:
sol = tmodel.optimize()
print("Thermo-FBA objective:", sol.objective_value)


DEBUG:thermomodel_None:'slim_optimize' ((), {}) 71.43 sec


Thermo-FBA objective: 1.0


In [ ]:
import json
from pathlib import Path

# Load ModelSEED compounds
with open(db_path) as f:
    compounds = json.load(f)  # dict keyed by 'cpdXXXX'
    print(compounds)


import pandas as pd

# Load your GEM metabolites
gem_mets = [
    '13dpg_c', '2pg_c', '3pg_c', '6pgc_c', '6pgl_c', 'ac_c', 'ac_e',
    'acald_c', 'acald_e', 'accoa_c', 'acon_C_c', 'actp_c', 'adp_c',
    'akg_c', 'akg_e', 'amp_c', 'atp_c', 'cit_c', 'co2_c', 'co2_e'
]

# Load ModelSEED thermodynamic database (JSON or CSV)
thermobase = pd.read_json(db_path)  # replace with your thermodb path
# Example columns: 'id', 'name', 'formula', 'synonyms'

def map_to_modelseed(gem_mets, gem_met_info, thermobase):
    """
    gem_met_info: dict {met_id: {'name': ..., 'formula': ...}}
    thermobase: pandas DataFrame of ModelSEED compounds
    """
    mapping = {}
    unmatched = []
    for met in gem_mets:
        info = gem_met_info.get(met, {})
        formula = info.get('formula', None)
        name = info.get('name', None)

        # Match by formula first
        candidates = thermobase[thermobase['formula'] == formula]
        if len(candidates) == 1:
            mapping[met] = candidates['id'].values[0]
            continue

        # Match by name / synonym
        candidates = thermobase[
            thermobase['name'].str.lower() == name.lower() |
            thermobase['synonyms'].apply(lambda x: name.lower() in [s.lower() for s in x])
        ]
        if len(candidates) >= 1:
            mapping[met] = candidates['id'].values[0]
        else:
            unmatched.append(met)

    return mapping, unmatched

# Example usage:
# gem_met_info = {met.id: {'name': met.name, 'formula': met.formula} for met in cobra_model.metabolites}
# mapping, unmatched = map_to_modelseed(gem_mets, gem_met_info, thermobase)


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
print("Number of metabolites with ΔG° info:", len(tmodel.metabolites))
print("Number of reactions with ΔG° info:", len(tmodel.reactions))


Number of metabolites with ΔG° info: 2128
Number of reactions with ΔG° info: 2739


In [ ]:
tmodel.prepare()  # sets up thermodynamic variables and constraints
tmodel.convert()  # adds thermodynamic constraints to the COBRA model


2025-12-02 18:56:22,544 - thermomodel_None - INFO - # Model preparation starting...
INFO:thermomodel_None:# Model preparation starting...
DEBUG:thermomodel_None:Metabolite 13dpg_c (3-Phospho-D-glyceroyl phosphate) has no seed_id
DEBUG:thermomodel_None:Metabolite 2pg_c (D-Glycerate 2-phosphate) has no seed_id
DEBUG:thermomodel_None:Metabolite 3pg_c (3-Phospho-D-glycerate) has no seed_id
DEBUG:thermomodel_None:Metabolite 6pgc_c (6-Phospho-D-gluconate) has no seed_id
DEBUG:thermomodel_None:Metabolite 6pgl_c (6-phospho-D-glucono-1,5-lactone) has no seed_id
DEBUG:thermomodel_None:Metabolite ac_c (Acetate) has no seed_id
DEBUG:thermomodel_None:Metabolite ac_e (Acetate) has no seed_id
DEBUG:thermomodel_None:Metabolite acald_c (Acetaldehyde) has no seed_id
DEBUG:thermomodel_None:Metabolite acald_e (Acetaldehyde) has no seed_id
DEBUG:thermomodel_None:Metabolite accoa_c (Acetyl-CoA) has no seed_id
DEBUG:thermomodel_None:Metabolite acon_C_c (cis-Aconitate) has no seed_id
DEBUG:thermomodel_None:Me

## Run model with coreected IDs

In [ ]:
sol = tmodel.optimize()
print("Thermo-FBA objective:", sol.objective_value)


DEBUG:thermomodel_None:'slim_optimize' ((), {}) 0.05 sec


Thermo-FBA objective: 0.8739215069684306


## Check fluxes

In [ ]:
# Top 10 reactions with largest flux
top_fluxes = sol.fluxes.abs().sort_values(ascending=False).head(10)
print(top_fluxes)


ATPS4r      45.514010
CYTBD       43.598985
NADH16      38.534610
EX_h2o_e    29.175827
H2Ot        29.175827
EX_co2_e    22.809833
CO2t        22.809833
EX_o2_e     21.799493
O2t         21.799493
EX_h_e      17.530865
Name: fluxes, dtype: float64


In [ ]:
# Look at key pathways
pathway_rxns = ['PGK', 'PYK', 'PDH', 'CS']
for rxn_id in pathway_rxns:
    print(rxn_id, sol.fluxes.get(rxn_id, "not in solution"))


PGK -16.023526143167604
PYK 1.7581774441067848
PDH 9.282532599166615
CS 6.00724957535033
